# v30 - v17 anchor plus our v29 fold-0 leg (full view + native 100 mm view, report-only), rank blend at w = 0.15

Explicit ensemble version (D-017): the v17 anchor (public 0.950: CoAtNet-384 SWA + ConvNeXt reader, cells
unchanged) and our v29 arm-L fold-0 leg (B2: ConvNeXt-tiny with a full 140 mm view and a 100 mm view rendered
from DICOM at 320 px per slice triplet, report-only target), blended per finding by rank at w = 0.15. Design:
`docs/research/v25-local-views-design.md` revision 3, sections 4 and 8 (decisions D-021, D-022): judged against
the running best public submission (v27); >= 0.952 four total folds, 0.951 or a hidden-digit tie win three,
otherwise B2 stops.

## Provenance

- Anchor: https://www.kaggle.com/code/sujanmajhisuzan/rsna-knee-apex-grandmaster-stack (version 1,
  Apache 2.0), built on nartaa/rsna-knee-0949-anatomical-mirror (Apache 2.0) and goodpjw2008's
  ConvNeXt reader (Apache 2.0); weights `nartaa/rsna-knee-publication-swa-weights-20261007`
  (licence "other"; trained with OAI data, D-019) and `goodpjw2008/rsna-knee-2-5d-convnext-reader`.
  Its cells below are unchanged, as in our v17 (public 0.950). See the authors' credits below.
- Our leg: the v19 leg script (v06 slot selection, v07 preprocessing, reference check) rendering each study
  twice (v07 at 140 mm; v28 = the same code at 100 mm), with the v29 trainer's dataset and model code (two
  views per centre, token types 0-9), weights `v29_fold_0_L_best.pt` from the private dataset
  `lingxd/rsna-knee-v29-weights`.

## Our changes (against v19)

1. This cell and a v30 configuration cell (one model: v29 arm L fold 0).
2. Leg: each study is prepared twice from DICOM (`CROP_MM` 140 and 100, otherwise the verbatim v07 code); both
   renderings must store the same slices. The reference check covers both renderings (byte-equal to the v07
   and v28 caches on three training studies). The model is built from the checkpoint's configuration (two
   views, local view from the 100 mm rendering) and loaded strictly; the OOF check re-predicts the fold-0
   reference study against the **v29** fold-0 OOF (tolerance 1e-3). Everything else as v19: two shards,
   absolute deadline, coverage >= 98%, otherwise the anchor is submitted unchanged.

Requirements: GPU T4 x2, internet off; the anchor's two datasets plus `lingxd/rsna-knee-v29-weights`.

In [ ]:
# v30 configuration
import time as _v30_time

VERSION = 'v30'
V30_T0 = _v30_time.time()
BLEND_W = 0.15                 # pre-registered (v25-local-views-design.md section 1)
K_INFER = 16
WEIGHT_SPECS = [('v29', 'L', 0)]   # (version, arm, fold) of each model
LEG_DEADLINE_S = 8.0 * 3600 - 15 * 60   # absolute: the leg is finished or killed by this notebook-elapsed time
LEG_MIN_S = 20 * 60            # do not start the leg with less time than this
MIN_COVERAGE = 0.98            # below this share of studies, submit the base unchanged
OOF_TOLERANCE = 1e-3
print(f'{VERSION}: blend weight {BLEND_W}, K_infer {K_INFER}, leg {WEIGHT_SPECS}', flush=True)

# RSNA Knee: accuracy inference (0.949 public AUC)

**Runnable inference notebook.** The exact checkpoint is attached and a T4 GPU is enabled. This version runs the scored pipeline and writes `submission.csv`.

Original 384-pixel full-field training; native 320-pixel center crop at inference, K94, two views with equal plain/anatomical-mirror probability averaging. One seed-42 training run, one averaged checkpoint at inference; averaged epochs [14, 11, 12]. The accuracy model is the original 384-trained checkpoint: the later native-320-crop-trained alternative scored 0.946 and is not substituted here.

The score is the displayed ordinary full-12 public leaderboard macro-AUC for submission `56906959`. The observed 26.1 minutes is submit-to-scored turnaround including queueing, orchestration and polling, not official worker runtime or an efficiency rank. Gold58 was reused for selection and development; it is not independent CV. This release does not establish significance or private-test performance.

The scored source's opening docstring and some comments describe older Raptor versions (0.924, 64 slices, 42 windows and earlier ensembles). They are retained to preserve the complete scored source bytes. This introduction, the live constants and the manifest describe the selected recipe: 96 slices, 94 windows, and one checkpoint.


## Reproduction requirements

Accept the competition rules and attach `rsna-knee-abnormality-detection`. Attach the exact checkpoint `raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt` from the [selected SWA weights](https://www.kaggle.com/datasets/nartaa/rsna-knee-publication-swa-weights-20261007), version 1. This public dependency contains only learned checkpoints and aggregate metadata. OAI images, OAI caches, report text and training label tables are not inference inputs and are not distributed.

Use the original Kaggle GPU environment: two Tesla T4 GPUs were available in the scored machine; the unchanged source uses its original CUDA worker policy. Runtime dependencies are Python, PyTorch with CUDA, timm, NumPy, pandas, pydicom and OpenCV. No downloads or pip installs are added. The scored Kaggle container is pinned to `gcr.io/kaggle-private-byod/python@sha256:2757e0c7d1e0a9cb43da657b97e223c321a98f5014bdf64f44f2f6b083ad2b2f`. The associated Kaggle qualification recorded torch 2.11.0+cu128 and timm 1.0.29.

Source SHA-256: `bcd864cbf349882e876959b49cb348cbb89b4c62fc204d4e4261049c5e3a79c5`. Checkpoint SHA-256: `7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef`. The original private scored source is `nartaa/rsna-knee-original384-s42-mirror320-infer`, version 1. Its private dependency is `nartaa/rsna-knee-oai-original384-s42-pair`, version 1; it remains private. This publication uses a separate notebook slug and leaves the private scored notebook unchanged.

The metadata enables T4 GPUs with internet disabled. `RUN_INFERENCE = True` runs the two code cells in order after verifying the attached checkpoint. The final source writes `submission.csv`; competition submission is managed separately.


## Credits and required acknowledgement

Built on dreaddevelopment's [Raptor CoAtNet/MIL inference recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). These are our retrained checkpoints, rather than those baseline weights. Weak-label sources include [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), combined with our report-derived labels. Source reports and study-level label tables are not included here. The CoAtNet backbone comes from timm. Competition data is provided under the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).


In [ ]:
# Authorized scoring run: verify the selected checkpoint before inference.
RUN_INFERENCE = True
if not RUN_INFERENCE:
    raise SystemExit("Static notebook only. Attach the exact checkpoint and competition, enable a T4 GPU, and set RUN_INFERENCE = True to run inference.")

import hashlib
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("This scored pipeline requires CUDA; CPU inference is not enabled.")
_weight_name = 'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt'
_candidates = [
    Path('/kaggle/input/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/rsna-knee-oai-original384-s42-pair') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-oai-original384-s42-pair') / _weight_name,
]
_weights = next((p for p in _candidates if p.is_file()), None)
if _weights is None:
    raise FileNotFoundError("Attach the exact selected weights; no alternative model is substituted.")

with _weights.open('rb') as _f:
    if hasattr(hashlib, "file_digest"):
        digest = hashlib.file_digest(_f, 'sha256').hexdigest()
    else:
        h = hashlib.sha256()
        while chunk := _f.read(65536):
            h.update(chunk)
        digest = h.hexdigest()
    assert digest == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', f"Checkpoint hash differs: got {digest}"
print(f"Verified verified SOTA weights: {_weights} (SHA-256 {digest})")


In [ ]:
# %% [code]
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import threading, queue
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
INPUT_RES = 320
INPUT_MODE = "crop"
IMG = 384   # native 384 corpus: the model saw 384 px crops directly, no 336->384 upsample
CROP_MM = 140.0
# 96 slices per study at native 384 px. Must match the corpus the weights were trained on
# (cache/build_corpus96.py).
SLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18)]
MAXS = sum(s[2] for s in SLOTS)                     # 96
K_EVAL = 94   # every window position of the 96-slice test volume (full coverage)
HDR_THREADS = 16   # per prep process: header reads are fuse-latency-bound, not CPU-bound
PX_THREADS = 8     # per prep process: pixel reads of the ~96 picked slices
QUEUE_MAX = 48     # studies in flight between the prep pool and the GPU consumers (RAM bound: ~14 MB each)
NORM = "imagenet"
FASTREAD_VARIANT = 'study_lut'   # kernel v3: research/fastread/readers.py, byte-identical to build_study
FR_HDR, FR_PX = 32, 16
_FASTREAD_B64 = "..."
def _load_fastread():
    import base64 as _b, importlib.util as _iu
    p = '/kaggle/working/_fastread.py' if os.path.isdir('/kaggle/working') else os.path.join(os.path.dirname(os.path.abspath(__file__)), '_fastread.py')
    if not os.path.exists(p):
        open(p, 'wb').write(_b.b64decode(_FASTREAD_B64))
    spec = _iu.spec_from_file_location('_fastread', p); m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
    return m
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [   # our 5-fold CoAtNet (public recipe reproduced on our folds/labels), equal-weight rank-mean
    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "w": 1.0},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3, img_size=INPUT_RES)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        # Bound backbone memory; the attention head still sees every window together.
        flat = x.flatten(0, 1)
        f = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    import hashlib
    with open(pt_path, "rb") as stream:
        observed_sha = hashlib.file_digest(stream, "sha256").hexdigest()
    assert observed_sha == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', "Bound checkpoint bytes differ"
    print(f"[weights] sha256 {observed_sha}", flush=True)
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    assert int(ck.get("res", res_default)) == 384, "Expected original 384 checkpoint"
    ck_res = INPUT_RES
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


SLOT_SIZES = [26, 22, 18, 12, 18]
SAGITTAL_SLOTS = (0, 1)

def _slot_index(D):
    b = np.cumsum([0] + SLOT_SIZES)
    if b[-1] != D: raise RuntimeError(f'slot sizes sum to {b[-1]}, volume has {D} slices')
    s_ = np.searchsorted(b, np.arange(D), side='right') - 1
    rel = (np.arange(D) - b[s_]) / np.maximum(1, np.array(SLOT_SIZES)[s_] - 1)
    return s_, rel

def _mirror_tri(tri, cs, D):
    """final-g1-bundle MIRROR_TTA: true L/R mirror of (k,3,H,W) uint8 eval triplets (research/rnd4/mirror/mirror_f0.py). The slot of each
    window's CENTRE slice decides the axis: sagittal (slots 0,1) -> reverse the 3 slice-channels; coronal/axial (2,3,4) -> flip the width.
    Done on uint8 BEFORE the per-channel LUT, i.e. 'mirror the knee, then normalise'."""
    s_, _ = _slot_index(D)
    sag = torch.from_numpy(np.isin(s_[np.asarray(cs, dtype=np.int64)], SAGITTAL_SLOTS)).to(tri.device).view(-1, 1, 1, 1)
    return torch.where(sag, tri.flip(1), tri.flip(-1))

def infer_pair_probs(model, plain, mirrored, device):
    assert plain.shape==mirrored.shape==(94,3,320,320)
    assert all(not m.training for m in model.modules()), "TTA must not update BN or enable dropout"
    p = infer_probs(model, plain, device)
    q = infer_probs(model, mirrored, device)
    return 0.5 * (p + q)

def eval_windows(vol, mask, k, res, norm=NORM, mirror=False):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0)
        if mirror:
            tri = _mirror_tri(torch.from_numpy(tri)[None], [c], D)[0].numpy()
        tri = tri.astype(np.float32) / 255.0
        assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
        lo = (IMG - res) // 2
        tri = tri[:, lo:lo + res, lo:lo + res]
        t = torch.from_numpy(np.ascontiguousarray(tri))
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


_WLUT = {}
def gpu_windows(vol, mask, k, res, dev, norm=NORM, mirror=False):
    """Same windows as eval_windows, built on the GPU from the uint8 volume. Values come from a per-channel
    256-entry table computed on the CPU with eval_windows' own float32 ops, so they are bit-identical."""
    D = vol.shape[0]
    cs = np.array([max(1, min(c, D - 2)) for c in _eval_centers(mask, D, k)])
    tri = torch.from_numpy(np.ascontiguousarray(vol[np.stack([cs - 1, cs, cs + 1], 1)])).to(dev)   # (k,3,IMG,IMG) uint8
    if mirror:
        tri = _mirror_tri(tri, cs, D)
    assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
    lo = (IMG - res) // 2
    tri = tri[:, :, lo:lo + res, lo:lo + res]
    key = (str(dev), norm)
    if key not in _WLUT:
        v = torch.from_numpy((np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0))
        t = torch.stack([v, v, v], 0).view(3, 256, 1)
        if norm == 'imagenet':
            t = (t - _MEAN.view(3, 1, 1)) / _STD.view(3, 1, 1)
        _WLUT[key] = t.view(3, 256).contiguous().to(dev)
    L = _WLUT[key]
    x = torch.stack([L[c][tri[:, c].long()] for c in range(3)], 1)                   # exact float32 values
    if x.shape[-1] != res:
        if norm == 'imagenet':   # resize must happen BEFORE normalisation, as in eval_windows
            L0 = _WLUT.setdefault((str(dev), 'none'), torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0).to(dev))
            x = L0[tri.long()]
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
            x = (x - _MEAN.to(dev)) / _STD.to(dev)
        else:
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    assert str(device).startswith("cuda"), "Qualified inference requires CUDA/fp16"
    x = xwins.unsqueeze(0).to(device)
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(x).float()
        assert torch.isfinite(logits).all(), "Nonfinite fp16 logits"
        o = torch.sigmoid(logits)
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def _hdr(f):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            iop = getattr(h, 'ImageOrientationPatient', None)
            ipp = getattr(h, 'ImagePositionPatient', None)
            if iop is not None and ipp is not None and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(getattr(h, 'InstanceNumber', 0) or 0)
            ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
            return (pos, f, ps, True)
        except Exception:
            return (0.0, f, 0.5, False)

    def order_and_meta(sdir):
        # Header reads are latency-bound on Kaggle's mounted competition data (~240k opens on the hidden
        # set); a thread pool hides that latency. Records and sort are unchanged -> identical ordering.
        fs = glob.glob(sdir + "/*.dcm")
        with ThreadPoolExecutor(max_workers=HDR_THREADS) as ex:
            out = list(ex.map(_hdr, fs))
        recs = [(pos, f, ps) for pos, f, ps, _ in out]; ps_list = [ps for _, _, ps, ok in out if ok]
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (build_corpus80.py, SPAN).
        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        sel = [files[min(p, n - 1)] for p in picks]
        def _px(fp_ps):
            fp, ps = fp_ps
            try: return read_px(fp), ps
            except Exception: return None, med_ps
        with ThreadPoolExecutor(max_workers=PX_THREADS) as ex:
            got = list(ex.map(_px, sel))                    # order preserved
        arrs = [a for a, _ in got]; pss = [ps for _, ps in got]
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask



# ---- parallel preprocessing helpers (module level so they pickle under fork) ----
_SER = None; _TSDIR = None; _READER = None
def _pre_init():
    global _READER
    _READER = _load_fastread().make_builder(FASTREAD_VARIANT, IMG, SLOTS, hdr_threads=FR_HDR, px_threads=FR_PX)
def _pre_one(sid):
    try:
        vol, mask = _READER(sid, _SER, _TSDIR)
    except Exception as e:
        print(f"  [pre] {sid[:16]} FAILED ({type(e).__name__}: {e})", flush=True)
        vol = np.zeros((MAXS, IMG, IMG), np.uint8); mask = np.zeros(MAXS, np.uint8)
    return sid, vol, mask

# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        # skip the competition data wherever Kaggle mounts it (/kaggle/input/competitions/<slug> OR /kaggle/input/<slug>):
        # recursively globbing its DICOM tree hung a commit run for 20+ min (2026-09-26)
        if "competition" in d.lower() or "abnormality-detection" in d or any(
                os.path.exists(os.path.join(d, x)) for x in ("test_series", "train_images", "train.csv", "test.csv")):
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]
    pos = {sid: i for i, sid in enumerate(test_ids)}

    # ---- 2. GPU consumers: one thread per GPU, every arm loaded on every GPU (per-study batch of 1,
    #         so the math is identical to the sequential kernel; only the wall-clock changes) ----
    xla = None
    try:
        import torch_xla; import torch_xla.core.xla_model as xm
        dev = torch_xla.device(); xla = xm
        print("device: XLA/TPU", dev, flush=True)
    except Exception as e:
        print("XLA unavailable:", type(e).__name__, str(e)[:80], "| device", dev, flush=True)
    devs = [f"cuda:{g}" for g in range(ngpu)] if (xla is None and ngpu > 0) else [dev]
    models = {}
    for d in devs:
        models[d] = []
        for a, arm in enumerate(ARMS):
            model, res = load_model(find_weight_file(arm["file"]), arm["arch"], arm["res"], d)
            models[d].append((model, res))
        print(f"[gpu {d}] loaded {A} arm(s) | {time.time()-t0:.0f}s", flush=True)
    q = queue.Queue(maxsize=QUEUE_MAX); done_n = [0]; lock = threading.Lock()

    def consumer(d):
        while True:
            item = q.get()
            if item is None: q.task_done(); return
            sid, vol, mask = item; i = pos[sid]
            try:
                for a, (model, res) in enumerate(models[d]):
                    xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM) if xla is not None else gpu_windows(vol, mask, K_EVAL, res, d)
                    assert xla is None, "Anatomical L/R TTA requires CUDA/fp16"
                    xm = gpu_windows(vol, mask, K_EVAL, res, d, norm=NORM, mirror=True)
                    arm_probs[a][i] = infer_pair_probs(model, xw, xm, d)
                    del xm
            except Exception as e:
                print(f"  [gpu {d}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            with lock:
                done_n[0] += 1
                if done_n[0] % 200 == 0 or done_n[0] == N:
                    print(f"  [gpu] {done_n[0]}/{N} | {time.time()-t0:.0f}s", flush=True)
            q.task_done()

    threads = [threading.Thread(target=consumer, args=(d,), daemon=True) for d in devs]
    for th in threads: th.start()

    # ---- 3. prep every study ONCE in a process pool and stream it to the GPUs as it lands ----
    import multiprocessing as mp
    global _SER, _TSDIR, _READER
    _SER, _TSDIR = SER, tsdir
    nproc = max(4, min(96, (os.cpu_count() or 8) - 8))
    t_pre = time.time()
    with mp.get_context("fork").Pool(nproc, initializer=_pre_init) as pool:
        for i, (sid, vol, mask) in enumerate(pool.imap_unordered(_pre_one, test_ids, chunksize=2)):
            q.put((sid, vol, mask))
            if (i + 1) % 200 == 0 or i + 1 == N:
                print(f"  [pre] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
    print(f"[pre] done: {N} studies in {time.time()-t_pre:.0f}s (procs {nproc} x fastread {FASTREAD_VARIANT} {FR_HDR}/{FR_PX} threads)", flush=True)
    for _ in devs: q.put(None)
    q.join()
    for th in threads: th.join()
    print(f"[gpu] done: {done_n[0]}/{N} studies on {len(devs)} device(s) | {time.time()-t0:.0f}s", flush=True)
    for d in devs:
        for m, _ in models[d]: del m
    models.clear(); gc.collect()

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission.csv"
    sub.to_csv(out, index=False)
    sub.to_csv("/kaggle/working/_sota_0949.csv", index=False)
    print("Saved SOTA 0.949 backup to /kaggle/working/_sota_0949.csv", flush=True)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import os
os.makedirs('/kaggle/working/own_src', exist_ok=True)
print('Created /kaggle/working/own_src directory.')


In [ ]:
%%writefile /kaggle/working/own_src/preprocess.py
"""Turn each DICOM series into a canonical uint8 volume [n_slices, SIZE, SIZE].

Canonical means: fixed physical pixel size, centre crop to a fixed field of view,
standard in-plane orientation, standard slice order and a per-series intensity window.
The same `load_series` is used for the training cache and for inference.

Usage: python src/preprocess.py [train|test] [n_series_limit]
"""
import os
import sys
from multiprocessing import Pool

import cv2
import numpy as np
import pandas as pd
import pydicom

DATA = "/mnt/ssd/rsna_knee/data"
WORK = "/mnt/ssd/rsna_knee/work"

MM_PER_PX = 0.4
SIZE = 384  # 153.6 mm field of view
MAX_SLICES = 64

# plane -> (normal axis, slice-order sign, (col axis, sign), (row axis, sign)) in LPS coordinates
# Sagittal: slices right->left, image columns anterior->posterior, rows superior->inferior
# Coronal:  slices anterior->posterior, columns patient-right->left, rows superior->inferior
# Axial:    slices superior->inferior, columns patient-right->left, rows anterior->posterior
CANON = {
    0: (+1, (1, +1), (2, -1)),
    1: (+1, (0, +1), (2, -1)),
    2: (-1, (0, +1), (1, +1)),
}
PLANES = ["Sagittal", "Coronal", "Axial"]


def _read(path):
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")
    if slope is not None and inter is not None:
        arr = arr * float(slope) + float(inter)
    iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None
    ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None
    ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None
    inst = float(ds.get("InstanceNumber") or 0)
    frames = list(arr) if arr.ndim == 3 else [arr]
    return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]


def _canon_slice(img, iop, ps, plane):
    """Reorient one slice to the canonical in-plane orientation; returns (img, (row_mm, col_mm))."""
    ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2  # no spacing: assume the image spans the crop
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[plane]
    if abs(r[cax]) < abs(r[rax]):  # rows and columns are swapped relative to canonical
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]
    if c[rax] * rsgn < 0:
        img = img[::-1]
    return img, ps


def _resample(img, ps):
    """Centre crop to SIZE*MM_PER_PX mm, resample to MM_PER_PX, zero-pad to SIZE x SIZE."""
    out = np.zeros((SIZE, SIZE), np.float32)
    h, w = img.shape
    ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))
    cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))
    nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def load_series(sdir, plane_name=None):
    """Returns (uint8 volume [n, SIZE, SIZE], info dict). Unreadable slices are skipped."""
    slices, bad = [], 0
    for fn in os.listdir(sdir):
        try:
            slices.extend(_read(os.path.join(sdir, fn)))
        except Exception:
            bad += 1
    info = {"n_raw": len(slices), "n_bad": bad}
    if not slices:
        return np.zeros((1, SIZE, SIZE), np.uint8), info

    iop = next((s[1] for s in slices if s[1] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:])
        plane = int(np.abs(normal).argmax())
    else:
        normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0
    info["plane"] = PLANES[plane]

    if normal is not None and all(s[2] is not None for s in slices):
        sgn = CANON[plane][0] * np.sign(normal[plane])
        slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)
        pos = [float(np.dot(s[2], normal)) for s in slices]
        info["slice_gap"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 0.0
    else:
        slices.sort(key=lambda s: s[4])
    if len(slices) > MAX_SLICES:
        keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)
        slices = [slices[i] for i in keep]

    vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])
    lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    info["n"] = len(vol)
    return (vol * 255).round().astype(np.uint8), info


def _job(args):
    split, study, series, plane, out_dir = args
    out = f"{out_dir}/{series}.npy"
    info = {"study": study, "series": series}
    if os.path.exists(out):
        return None
    try:
        vol, extra = load_series(f"{DATA}/{split}_series/{study}/{series}", plane)
        np.save(out + ".tmp.npy", vol)
        os.replace(out + ".tmp.npy", out)
        info.update(extra)
    except Exception as e:
        info["error"] = f"{type(e).__name__}: {e}"[:200]
    return info


def main():
    split = sys.argv[1] if len(sys.argv) > 1 else "train"
    limit = int(sys.argv[2]) if len(sys.argv) > 2 else None
    out_dir = f"{WORK}/cache{SIZE}_{split}"
    os.makedirs(out_dir, exist_ok=True)
    se = pd.read_csv(f"{DATA}/{split}_series.csv")
    if limit:
        se = se.sample(limit, random_state=0)
    tasks = [(split, a, b, p, out_dir) for a, b, p in zip(se.StudyInstanceUID, se.SeriesInstanceUID, se.Anatomical_Plane)]
    rows = []
    with Pool(18) as p:
        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=4)):
            if r:
                rows.append(r)
            if i % 1000 == 0:
                print(f"{i}/{len(tasks)}", flush=True)
    df = pd.DataFrame(rows)
    meta = f"{WORK}/cache{SIZE}_{split}_meta.csv"
    if os.path.exists(meta) and len(df):
        df = pd.concat([pd.read_csv(meta), df]).drop_duplicates("series", keep="last")
    if len(df):
        df.to_csv(meta, index=False)
        print("done", len(df), "errors:", int(df["error"].notna().sum()) if "error" in df else 0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/own_src/knee.py
"""Shared dataset and model code for training and inference."""
import math

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# slot = plane index + 3 * (1 - fat_suppressed): fat-suppressed planes first
N_SLOTS = 6
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def slot_of(plane, fat_sup):
    return PLANES.index(plane) + 3 * (1 - int(fat_sup))


def build_study_table(series_df, n_slices):
    """series_df: StudyInstanceUID, SeriesInstanceUID, Fat_Suppression, Anatomical_Plane.
    n_slices: dict series -> cached slice count. Returns {study: [[series ids of slot 0], ...]}
    with each slot's candidates sorted by slice count (largest first)."""
    table = {}
    for st, sr, fs, pl in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID,
                              series_df.Fat_Suppression, series_df.Anatomical_Plane):
        if n_slices.get(sr, 0) < 3:
            continue
        table.setdefault(st, [[] for _ in range(N_SLOTS)])[slot_of(pl, fs)].append(sr)
    for slots in table.values():
        for cands in slots:
            cands.sort(key=lambda s: -n_slices[s])
        # Intelligent anatomical plane fallback: if fat-sat is missing, use non-fat-sat (and vice-versa)
        for p in range(3):
            fs_s, nonfs_s = p, p + 3
            if not slots[fs_s] and slots[nonfs_s]:
                slots[fs_s] = list(slots[nonfs_s])
            elif not slots[nonfs_s] and slots[fs_s]:
                slots[nonfs_s] = list(slots[fs_s])
    return table


def window_centres(n, k, train, lo=0.04, hi=0.96):
    """k slice indices (centres of 3-slice windows) spread over the series."""
    a, b = max(1.0, lo * (n - 1)), min(n - 2.0, hi * (n - 1))
    if b < a:
        a = b = (n - 1) / 2
    if train:
        edges = np.linspace(a, b, k + 1)
        c = edges[:-1] + np.random.rand(k) * (edges[1:] - edges[:-1])
    else:
        c = np.linspace(a, b, k)
    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))


class StudyDataset(torch.utils.data.Dataset):
    """One item = one study: uint8 windows [N_SLOTS, k, 3, H, W], slot mask, slice positions, target."""

    def __init__(self, studies, table, cache_dir, targets=None, k=8, train=False, loader=None):
        self.studies, self.table, self.cache_dir = list(studies), table, cache_dir
        self.targets, self.k, self.train = targets, k, train
        self.loader = loader or (lambda s: np.load(f"{self.cache_dir}/{s}.npy", mmap_mode="r"))

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        st = self.studies[i]
        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])
        x, mask, pos = None, np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)
        for s, cands in enumerate(slots):
            if not cands:
                continue
            sr = cands[np.random.randint(len(cands))] if self.train else cands[0]
            vol = self.loader(sr)
            n = len(vol)
            c = window_centres(n, self.k, self.train)
            win = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])  # [k, 3, H, W]
            if x is None:
                x = np.zeros((N_SLOTS, self.k) + win.shape[1:], np.uint8)
            x[s], mask[s], pos[s] = win, True, c / max(n - 1, 1)
        if x is None:
            x = np.zeros((N_SLOTS, self.k, 3, 384, 384), np.uint8)
        y = self.targets[i] if self.targets is not None else np.zeros(len(LABELS), np.float32)
        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32))


def make_views(x, res, train, group, fov=0.92):
    """x: uint8 [N, 3, H, W] with N = n_series * group. Crops a centred `fov` fraction of the cached
    field of view and resizes to res; in training adds rotation/scale/shift and gain/gamma jitter
    that is shared within a series."""
    n = x.shape[0]
    ns = n // group
    dev = x.device
    x = x.float().div_(255.0)
    if train:
        ang = (torch.rand(ns, device=dev) - 0.5) * 2 * math.radians(12)
        sc = fov * (1 + (torch.rand(ns, device=dev) - 0.5) * 0.24)
        tx = (torch.rand(ns, device=dev) - 0.5) * 0.12
        ty = (torch.rand(ns, device=dev) - 0.5) * 0.12
        gamma = torch.exp((torch.rand(ns, device=dev) - 0.5) * 0.5)
        gain = 1 + (torch.rand(ns, device=dev) - 0.5) * 0.2
    else:
        ang = torch.zeros(ns, device=dev)
        sc = torch.full((ns,), fov, device=dev)
        tx = ty = torch.zeros(ns, device=dev)
        gamma = gain = None
    cos, sin = torch.cos(ang) * sc, torch.sin(ang) * sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    theta = theta.repeat_interleave(group, 0)
    grid = F.affine_grid(theta, (n, 3, res, res), align_corners=False)
    if x.shape[-1] > 1.5 * res:  # avoid aliasing when shrinking a lot
        x = F.avg_pool2d(x, 2) if x.shape[-1] >= 2 * res else F.interpolate(x, scale_factor=0.75, mode="bilinear", antialias=True)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    if train:
        g = gamma.repeat_interleave(group).view(n, 1, 1, 1)
        x = x.clamp_min(1e-4).pow(g) * gain.repeat_interleave(group).view(n, 1, 1, 1)
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    return (x - mean) / std


class KneeNet(nn.Module):
    """2D backbone on 3-slice windows -> tokens tagged with series slot and slice position ->
    small transformer over all windows of the study -> one attention pooling per finding."""

    def __init__(self, backbone="convnext_tiny", pretrained=True, dim=384, depth=2, drop=0.2, drop_path=0.1,
                 img_size=None):
        super().__init__()
        kw = {"drop_path_rate": drop_path} if drop_path else {}
        if img_size and backbone.startswith(("vit_", "eva", "beit")):  # fixed-grid transformers need the input size
            kw["img_size"] = img_size
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kw)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(N_SLOTS, dim)
        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, len(LABELS)))
        self.cls_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(len(LABELS)))

    def head(self, feat, mask, pos):
        """feat [B, S, K, D], mask [B, S] bool, pos [B, S, K] in 0..1 -> logits [B, 12]."""
        b, s, k, _ = feat.shape
        slot = torch.arange(s, device=feat.device).view(1, s, 1).expand(b, s, k)
        h = self.proj(feat) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))
        h = h.view(b, s * k, -1)
        pad = ~mask.unsqueeze(-1).expand(b, s, k).reshape(b, s * k)
        pad = pad & ~pad.all(1, keepdim=True)  # a study with no series at all attends to everything
        if self.tf is not None:
            h = self.tf(h, src_key_padding_mask=pad)
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)  # [B, T, C]
        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

    def forward(self, x, mask, pos, res=256, train=False):
        """x uint8 [B, S, K, 3, H, W]."""
        b, s, k = x.shape[:3]
        keep = mask.view(-1)  # run the backbone only on series that exist
        xv = x.view(b * s, k, *x.shape[3:])[keep].flatten(0, 1)
        f = self.enc(make_views(xv, res, train, k).contiguous(memory_format=torch.channels_last))
        feat = f.new_zeros(b * s, k, f.shape[-1])
        feat[keep] = f.view(-1, k, f.shape[-1])
        return self.head(feat.view(b, s, k, -1), mask, pos)


In [ ]:
%%writefile /kaggle/working/own_src/infer.py
"""Predict the 12 findings for every study straight from DICOM folders.

Used by the Kaggle submission notebook and locally to check that predictions from raw DICOMs
match those from the training cache.

Usage: python src/infer.py <data_dir> <split> <out_csv> <ckpt> [<ckpt> ...]
  data_dir contains <split>.csv, <split>_series.csv and <split>_series/
"""
import os
import sys

import numpy as np
import pandas as pd
import torch

from knee import LABELS, N_SLOTS, KneeNet, build_study_table, window_centres
from preprocess import MAX_SLICES, SIZE, load_series


class DicomStudyDataset(torch.utils.data.Dataset):
    """Loads every selected series of a study once; windows are cut per model setting later."""

    def __init__(self, studies, table, root, k):
        self.studies, self.table, self.root, self.k = list(studies), table, root, k

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        st = self.studies[i]
        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])
        x = np.zeros((N_SLOTS, self.k, 3, SIZE, SIZE), np.uint8)
        mask, pos = np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)
        for s, cands in enumerate(slots):
            if not cands:
                continue
            try:
                vol, _ = load_series(f"{self.root}/{st}/{cands[0]}")
            except Exception:
                continue
            n = len(vol)
            if n < 3:
                continue
            c = window_centres(n, self.k, False)
            x[s] = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])
            mask[s], pos[s] = True, c / max(n - 1, 1)
        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos)


def load_models(ckpts, device="cuda"):
    models = []
    for path in ckpts:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        # the pretrained tag after the dot is irrelevant here and unknown to older timm versions
        m = KneeNet(ck["args"]["backbone"].split(".")[0], pretrained=False, img_size=ck["args"]["res"])
        m.load_state_dict(ck["model"])
        models.append((m.to(device).to(memory_format=torch.channels_last).eval(), ck["args"]["res"]))
    return models


@torch.no_grad()
def run(data_dir, split, ckpts, k=16, studies=None, workers=4, bs=4):
    se = pd.read_csv(f"{data_dir}/{split}_series.csv")
    root = f"{data_dir}/{split}_series"
    if studies is None:
        studies = pd.read_csv(f"{data_dir}/{split}.csv").StudyInstanceUID.tolist()
    se = se[se.StudyInstanceUID.isin(set(studies))]
    n_files = {}
    for st, sr in zip(se.StudyInstanceUID, se.SeriesInstanceUID):
        d = f"{root}/{st}/{sr}"
        n_files[sr] = min(len(os.listdir(d)), MAX_SLICES) if os.path.isdir(d) else 0
    table = build_study_table(se, n_files)
    dl = torch.utils.data.DataLoader(DicomStudyDataset(studies, table, root, k), batch_size=bs,
                                     num_workers=workers, pin_memory=True)
    models = load_models(ckpts)
    out = []
    for x, mask, pos in dl:
        x, mask, pos = x.cuda(non_blocking=True), mask.cuda(), pos.cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            p = torch.stack([m(x, mask, pos, res=res).float().sigmoid() for m, res in models]).mean(0)
        out.append(p.cpu())
    return pd.DataFrame(torch.cat(out).numpy(), index=pd.Index(studies, name="StudyInstanceUID"), columns=LABELS)


if __name__ == "__main__":
    data_dir, split, out_csv, ckpts = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4:]
    run(data_dir, split, ckpts).to_csv(out_csv)
    print("wrote", out_csv)


In [ ]:
# ==============================================================================
# Grandmaster Apex Fusion: SOTA CoAtNet 384px (0.949) + ConvNeXt 2.5D MIL Reader
# ==============================================================================
import os as _o_os
import sys as _o_sys
import glob as _o_glob
import time as _o_time
import shutil as _o_shutil
import subprocess as _o_sp
import pandas as _o_pd
import numpy as _o_np
import torch as _o_torch
import gc as _o_gc

_o_pub = "/kaggle/working/submission.csv"
_o_sota = "/kaggle/working/_sota_0949.csv"
_o_bak = "/kaggle/working/_sota_backup.csv"

# Ensure we have our verified 0.949 SOTA submission safely backed up
if not _o_os.path.exists(_o_sota) and _o_os.path.exists(_o_pub):
    _o_shutil.copy(_o_pub, _o_sota)
_o_shutil.copy(_o_pub, _o_bak)
print("[Apex Fusion] SOTA baseline secured at:", _o_sota, flush=True)

try:
    # 1. Clean CUDA memory from Step 1
    _o_gc.collect()
    if _o_torch.cuda.is_available():
        _o_torch.cuda.empty_cache()

    # 2. Locate ConvNeXt reader assets and competition test data
    def _o_find(name):
        for pat in ["/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"]:
            for d in sorted(_o_glob.glob(pat)):
                if _o_os.path.exists(_o_os.path.join(d, name)):
                    return d.rstrip("/")
        raise FileNotFoundError(f"{name} not found in /kaggle/input")

    _o_assets = _o_find("cnxt_v0_fold0.pt")
    _o_data = _o_find("test_series.csv")
    _o_ck = sorted(_o_glob.glob(_o_assets + "/cnxt_v0_fold*.pt"))
    print(f"[ConvNeXt Reader] Found {len(_o_ck)} checkpoints in {_o_assets}", flush=True)

    # 3. Setup environment and install decoders if wheels present
    _o_env = dict(_o_os.environ)
    _o_target = "/kaggle/working/_own_env"
    _o_sp.run([_o_sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", _o_target,
               "--find-links", _o_assets, "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)
    _o_timm = [w for pat in ["/kaggle/input/*/timm-*.whl", "/kaggle/input/*/*/timm-*.whl"]
               for w in _o_glob.glob(pat)]
    if _o_timm:
        _o_sp.run([_o_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", _o_target, _o_timm[0]], check=False)
    _o_env["PYTHONPATH"] = _o_target + _o_os.pathsep + _o_env.get("PYTHONPATH", "")

    # Pick the GPU with highest free memory
    if _o_torch.cuda.is_available():
        _o_free = [_o_torch.cuda.mem_get_info(i)[0] for i in range(_o_torch.cuda.device_count())]
        best_gpu = max(range(len(_o_free)), key=lambda i: _o_free[i])
        _o_env["CUDA_VISIBLE_DEVICES"] = str(best_gpu)
        print(f"[ConvNeXt Reader] Selected GPU {best_gpu} with {_o_free[best_gpu] / 1e9:.1f} GB free", flush=True)

    # 4. Run ConvNeXt Reader inference
    _o_t0 = _o_time.time()
    _o_own_csv = "/kaggle/working/_own.csv"
    print("[ConvNeXt Reader] Starting inference on test set...", flush=True)
    _o_sp.run([_o_sys.executable, "/kaggle/working/own_src/infer.py", _o_data, "test", _o_own_csv] + _o_ck,
              check=True, env=_o_env)
    print(f"[ConvNeXt Reader] Finished inference in {_o_time.time() - _o_t0:.1f}s", flush=True)

    # 5. Load predictions
    sota_df = _o_pd.read_csv(_o_sota, dtype={"StudyInstanceUID": str})
    own_df = _o_pd.read_csv(_o_own_csv, dtype={"StudyInstanceUID": str})
    own_df = own_df.set_index("StudyInstanceUID").loc[sota_df.StudyInstanceUID].reset_index()

    labels = [c for c in sota_df.columns if c != "StudyInstanceUID"]
    assert own_df[labels].notna().all().all(), "ConvNeXt predictions contain NaN"
    assert (own_df[labels].nunique() > 1).all(), "ConvNeXt predictions are constant"

    # 6. Target-Specific Calibrated Rank Fusion
    # Exploit ConvNeXt strengths on Baker's Cyst (+0.036), Contusion (+0.025), Medial OA, ACL
    # while strictly relying on SOTA CoAtNet for Lateral Meniscus (+0.071), Fracture (+0.057), Lateral OA (+0.048)
    TARGET_WEIGHTS = {
        "Baker's": 0.35,           # ConvNeXt is dominant (0.976 vs 0.940)
        "Contusion": 0.30,         # ConvNeXt captures 3D bone marrow edema (0.966)
        "Medial OA": 0.25,         # ConvNeXt joint space sensitivity (0.966)
        "ACL": 0.25,               # ConvNeXt 0.965 AUC
        "Medial Meniscus": 0.20,   # Strong 2.5D sagittal slice view
        "MCL": 0.20,               # Coronal 2.5D feature extraction
        "Lateral Meniscus": 0.08,  # SOTA CoAtNet gained +0.071 (supreme)
        "Fracture": 0.08,          # SOTA CoAtNet gained +0.057 (supreme)
        "Lateral OA": 0.08,        # SOTA CoAtNet gained +0.048 (supreme)
        "PF OA": 0.08,             # SOTA CoAtNet multi-slot axial coverage
        "Effusion": 0.04,          # SOTA CoAtNet near-perfect (0.986)
        "Synovitis": 0.03,         # SOTA CoAtNet report supervision
    }

    rank_sota = sota_df[labels].rank(method="average", pct=True)
    rank_own = own_df[labels].rank(method="average", pct=True)

    fused_df = sota_df.copy()
    for col in labels:
        w = TARGET_WEIGHTS.get(col, 0.15)
        fused_df[col] = (1.0 - w) * rank_sota[col] + w * rank_own[col]

    # Final percentile rank normalization to [0, 1]
    fused_df[labels] = fused_df[labels].rank(method="average", pct=True)
    assert fused_df[labels].notna().all().all(), "Fused predictions contain NaN"
    assert (fused_df[labels].nunique() > 1).all(), "Fused predictions are constant"

    # Save to submission.csv
    fused_df.to_csv(_o_pub, index=False)
    print(f"[Apex Fusion SUCCESS] Successfully fused SOTA 0.949 + ConvNeXt 2.5D Reader into {_o_pub}!")
    print(f"[Apex Fusion SUCCESS] Applied Target Weights: {TARGET_WEIGHTS}")
    print(fused_df.head())

except Exception as e:
    print(f"[Apex Fusion EXCEPTION] {type(e).__name__}: {e}", flush=True)
    print("[Apex Fusion FALLBACK] Restoring pure SOTA 0.949 submission.csv to ensure 100% safety!", flush=True)
    _o_shutil.copy(_o_bak, _o_pub)

print("Done. Submission file size:", _o_os.path.getsize(_o_pub), "bytes")


## Our leg (v29 arm L, fold 0) and the blend

The anchor has written `/kaggle/working/submission.csv`; it is kept as the base.

In [ ]:
import gc as _v30_gc
import shutil as _v30_shutil
from pathlib import Path as _V30Path

_V30_WORK = _V30Path('/kaggle/working')
_V30_ANCHOR = _V30_WORK / 'v30_base_submission.csv'   # the v17 anchor output
_v30_shutil.copy2(_V30_WORK / 'submission.csv', _V30_ANCHOR)
import hashlib as _v30_hashlib
_v30_sha = lambda p: _v30_hashlib.sha256(p.read_bytes()).hexdigest() if p.is_file() else None
_v30_base_status = {'base_sha256': _v30_sha(_V30_ANCHOR), 'coatnet_only_sha256': _v30_sha(_V30_WORK / '_sota_0949.csv'),
                    'convnext_output_present': (_V30_WORK / '_own.csv').is_file()}
_v30_base_status['base_fusion_applied'] = bool(_v30_base_status['coatnet_only_sha256']) and \
    _v30_base_status['coatnet_only_sha256'] != _v30_base_status['base_sha256']
try:
    import torch as _v30_torch
    _v30_gc.collect()
    for _d in range(_v30_torch.cuda.device_count()):
        with _v30_torch.cuda.device(_d):
            _v30_torch.cuda.empty_cache()
except Exception as _exc:
    print('cuda cleanup skipped:', _exc, flush=True)
print(f'base done at {(_v30_time.time() - V30_T0) / 3600:.2f} h; base saved; {_v30_base_status}', flush=True)

In [ ]:
%%writefile /kaggle/working/v30_leg.py
"""v30 leg: the v29 arm-L fold-0 model (full view + 100 mm view rendered from DICOM) on one shard of the test
studies (v19 leg code; data and model sections from the v29 trainer)."""
import json
import math
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
FOCAL = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus']
N_SLOTS = 5
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]


def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
import base64
import hashlib
import io
import os
from concurrent.futures import ProcessPoolExecutor

# ---------------------------------------------------------------- v07 constants (verbatim)
SLOTS = ['SAG_FS', 'SAG_NFS', 'COR_FS', 'COR_OTHER', 'AX_FS']
SLOT_PLANE = {'SAG_FS': 'Sagittal', 'SAG_NFS': 'Sagittal', 'COR_FS': 'Coronal',
              'COR_OTHER': 'Coronal', 'AX_FS': 'Axial'}
MAX_SLICES = 32
CROP_MM = 140.0
OUT_PX = 320
WINDOW_PCT = (0.5, 99.5)
JPEG_QUALITY = 92
SIDE_DEADZONE_MM = 20.0

# ---------------------------------------------------------------- v07 shared functions (verbatim)
import cv2
import numpy as np
import pydicom

HEADER_TAGS = ['ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber',
               'PixelSpacing', 'Rows', 'Columns', 'Laterality']

def _floats(value, n):
    try:
        out = [float(v) for v in value]
    except Exception:
        return None
    return out if len(out) == n and all(np.isfinite(out)) else None

def order_slices(records):
    # records: list of dicts with name, ipp, iop, inst. Returns (ordered names, method, info).
    info = {}
    with_geom = [r for r in records if r['ipp'] is not None and r['iop'] is not None]
    if records and len(with_geom) == len(records):
        iop = np.asarray(with_geom[0]['iop'], dtype=np.float64)
        normal = np.cross(iop[:3], iop[3:])
        norm = np.linalg.norm(normal)
        if norm > 1e-6:
            normal = normal / norm
            pos = np.asarray([np.dot(r['ipp'], normal) for r in with_geom])
            info['n_unique_positions'] = int(len(np.unique(np.round(pos, 3))))
            info['obliquity_deg'] = float(np.degrees(np.arccos(min(1.0, np.abs(normal).max()))))
            iop_all = np.asarray([r['iop'] for r in with_geom], dtype=np.float64)
            info['iop_consistent'] = bool(np.allclose(iop_all, iop_all[0], atol=1e-3))
            if info['n_unique_positions'] >= 0.9 * len(records):
                order = np.argsort(pos, kind='stable')
                gaps = np.diff(pos[order])
                info['slice_gap_mm'] = float(np.median(np.abs(gaps))) if len(gaps) else float('nan')
                return [with_geom[i]['name'] for i in order], 'geometry', info
    if records and all(r['inst'] is not None for r in records):
        ordered = sorted(records, key=lambda r: (r['inst'], r['name']))
        return [r['name'] for r in ordered], 'instance_number', info
    return sorted(r['name'] for r in records), 'filename', info

SLOT_SPEC = [
    ('SAG_FS', 'Sagittal', True, 8),
    ('SAG_NFS', 'Sagittal', False, 6),
    ('COR_FS', 'Coronal', True, 8),
    ('COR_OTHER', 'Coronal', None, 4),
    ('AX_FS', 'Axial', True, 6),
]

def _rank_key(c):
    return (0 if c['geometry_ok'] else 1, -int(c['n_readable'] or 0), c['SeriesInstanceUID'])

def select_slots(study_series):
    # study_series: list of dicts with SeriesInstanceUID, Anatomical_Plane, Fluid_Sensitive,
    # geometry_ok, n_readable. Returns {slot: (series uid or None, duplicate flag, fallback flag)}.
    usable = [s for s in study_series if (s.get('n_readable') or 0) > 0]
    chosen, used = {}, set()
    for slot, plane, fluid_pref, _k in SLOT_SPEC:
        same_plane = sorted((s for s in usable if s['Anatomical_Plane'] == plane), key=_rank_key)
        if not same_plane:
            chosen[slot] = (None, False, False)
            continue
        unused = [s for s in same_plane if s['SeriesInstanceUID'] not in used]
        if fluid_pref is None:
            preferred = sorted(unused, key=lambda s: (int(s['Fluid_Sensitive']),) + _rank_key(s))
        else:
            preferred = [s for s in unused if bool(s['Fluid_Sensitive']) == fluid_pref]
        if preferred:
            pick, duplicate, fallback = preferred[0], False, False
        elif unused:
            pick, duplicate, fallback = unused[0], False, True
        else:
            pick, duplicate, fallback = same_plane[0], True, True
        used.add(pick['SeriesInstanceUID'])
        chosen[slot] = (pick['SeriesInstanceUID'], duplicate, fallback)
    return chosen

# Canonical patient-space directions (DICOM LPS: +x left, +y posterior, +z superior).
# Per plane: (column direction, row direction, slice direction); 'medial' depends on side.
CANONICAL = {
    'Sagittal': ('posterior', 'inferior', 'medial'),
    'Coronal': ('medial', 'inferior', 'posterior'),
    'Axial': ('medial', 'posterior', 'inferior'),
}

def direction_vector(name, side):
    if name == 'medial':
        # The right knee's medial side faces the patient's left (+x), and vice versa.
        return np.array([1.0, 0.0, 0.0]) if side == 'R' else np.array([-1.0, 0.0, 0.0])
    return {'posterior': np.array([0.0, 1.0, 0.0]),
            'inferior': np.array([0.0, 0.0, -1.0])}[name]

def series_headers(series_dir):
    # Read headers of every file; return records plus first-file metadata.
    names = sorted(p.name for p in series_dir.iterdir() if p.suffix.lower() == '.dcm')
    records, meta = [], None
    for name in names:
        ds = pydicom.dcmread(str(series_dir / name), stop_before_pixels=True, force=True,
                             specific_tags=HEADER_TAGS)
        try:
            inst = int(float(str(getattr(ds, 'InstanceNumber', '')).strip()))
        except ValueError:
            inst = None
        rec = {'name': name,
               'ipp': _floats(getattr(ds, 'ImagePositionPatient', None) or [], 3),
               'iop': _floats(getattr(ds, 'ImageOrientationPatient', None) or [], 6),
               'inst': inst}
        records.append(rec)
        if meta is None:
            meta = {'spacing': _floats(getattr(ds, 'PixelSpacing', None) or [], 2),
                    'rows': getattr(ds, 'Rows', None), 'cols': getattr(ds, 'Columns', None),
                    'laterality': str(getattr(ds, 'Laterality', '') or '').strip().upper()}
    return records, meta

def image_centre_x(rec, meta):
    if rec['ipp'] is None or rec['iop'] is None or not meta['spacing'] or not meta['rows'] or not meta['cols']:
        return None
    ipp, iop = np.asarray(rec['ipp']), np.asarray(rec['iop'])
    centre = (ipp + iop[:3] * meta['spacing'][1] * float(meta['cols']) / 2
              + iop[3:] * meta['spacing'][0] * float(meta['rows']) / 2)
    return float(centre[0])

def decide_side(tags, centre_xs):
    # Returns (side, source). Tag first, then geometry with a dead zone.
    letters = [t[0] for t in tags if t and t[0] in ('L', 'R')]
    if letters:
        return max(set(letters), key=letters.count), 'tag'
    if centre_xs:
        m = float(np.median(centre_xs))
        if abs(m) >= SIDE_DEADZONE_MM:
            return ('R' if m < 0 else 'L'), 'geometry'
    return 'L', 'unresolved'

def geometry_side(centre_xs):
    if not centre_xs:
        return None
    m = float(np.median(centre_xs))
    return None if abs(m) < SIDE_DEADZONE_MM else ('R' if m < 0 else 'L')

def crop_resize(arr, spacing):
    # Centre crop of CROP_MM (zero padding if smaller), resize to OUT_PX.
    rows, cols = arr.shape
    want_r = int(round(CROP_MM / spacing[0])) if spacing else rows
    want_c = int(round(CROP_MM / spacing[1])) if spacing else cols
    out = np.zeros((want_r, want_c), dtype=np.float32)
    src_r0, src_c0 = max(0, (rows - want_r) // 2), max(0, (cols - want_c) // 2)
    dst_r0, dst_c0 = max(0, (want_r - rows) // 2), max(0, (want_c - cols) // 2)
    h, w = min(rows, want_r), min(cols, want_c)
    out[dst_r0:dst_r0 + h, dst_c0:dst_c0 + w] = arr[src_r0:src_r0 + h, src_c0:src_c0 + w]
    interp = cv2.INTER_AREA if min(want_r, want_c) >= OUT_PX else cv2.INTER_LINEAR
    return cv2.resize(out, (OUT_PX, OUT_PX), interpolation=interp), (h, w)

def encode_jpeg(img_u8):
    ok, buf = cv2.imencode('.jpg', img_u8, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
    if not ok:
        raise RuntimeError('JPEG encode failed')
    return buf.tobytes()

def decode_jpeg(data):
    return cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)

def prepare_study(study, slot_series, image_root):
    # slot_series: {slot: SeriesInstanceUID or None}. Returns (per-slot dict, study info).
    # Per slot: {'jpeg': [bytes], 'raw_index': [int], 'n_series': int, ...} or None.
    uids = sorted({u for u in slot_series.values() if u})
    plane_of = {u: SLOT_PLANE[s] for s, u in slot_series.items() if u}
    headers, tags, centre_xs = {}, [], []
    for uid in uids:
        records, meta = series_headers(image_root / study / uid)
        ordered, method, _info = order_slices(records)
        by_name = {r['name']: r for r in records}
        headers[uid] = (ordered, method, meta, by_name)
        tags.append(meta['laterality'])
        mid = by_name[ordered[len(ordered) // 2]]
        cx = image_centre_x(mid, meta)
        if cx is not None:
            centre_xs.append(cx)
    side, side_source = decide_side(tags, centre_xs)
    info = {'side': side, 'side_source': side_source,
            'tag_side': next((t[0] for t in tags if t and t[0] in ('L', 'R')), ''),
            'geometry_side': geometry_side(centre_xs) or ''}

    stored = {}
    for uid in uids:
        ordered, method, meta, by_name = headers[uid]
        n = len(ordered)
        keep = list(range(n)) if n <= MAX_SLICES else \
            sorted(set(np.linspace(0, n - 1, MAX_SLICES).round().astype(int).tolist()))
        spacing = meta['spacing']
        crops, fields, kept, n_bad = [], [], [], 0
        for i in keep:
            try:
                ds = pydicom.dcmread(str(image_root / study / uid / ordered[i]), force=True)
                arr = ds.pixel_array.astype(np.float32)
                if arr.ndim != 2:
                    raise ValueError(f'unexpected shape {arr.shape}')
            except Exception:
                n_bad += 1   # an unreadable slice is skipped, not fatal for the study
                continue
            kept.append(i)
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            arr = arr * slope + intercept
            img, (h, w) = crop_resize(arr, spacing)
            crops.append(img)
            rows, cols = arr.shape
            r0, c0 = max(0, (rows - h) // 2), max(0, (cols - w) // 2)
            fields.append(arr[r0:r0 + h, c0:c0 + w].ravel())
        if not kept:
            stored[uid] = None
            continue
        lo, hi = np.percentile(np.concatenate(fields), WINDOW_PCT)
        # Orientation from the first stored slice's geometry.
        rec = by_name[ordered[kept[0]]]
        flip_lr = flip_ud = reverse = False
        if method == 'geometry' and rec['iop'] is not None:
            iop = np.asarray(rec['iop'])
            row_dir, col_dir = iop[:3], iop[3:]   # along a row (columns increase), down a column (rows increase)
            normal = np.cross(row_dir, col_dir)
            plane = plane_of[uid]
            want_cols, want_rows, want_slices = (direction_vector(n_, side) for n_ in CANONICAL[plane])
            flip_lr = float(np.dot(row_dir, want_cols)) < 0
            flip_ud = float(np.dot(col_dir, want_rows)) < 0
            reverse = float(np.dot(normal, want_slices)) < 0
        imgs = []
        for img in crops:
            u8 = np.clip((img - lo) / max(hi - lo, 1e-6) * 255.0, 0, 255).astype(np.uint8)
            if flip_lr:
                u8 = u8[:, ::-1]
            if flip_ud:
                u8 = u8[::-1, :]
            imgs.append(np.ascontiguousarray(u8))
        raw_index = list(kept)
        if reverse:
            imgs, raw_index = imgs[::-1], raw_index[::-1]
        stored[uid] = {'jpeg': [encode_jpeg(x) for x in imgs], 'raw_index': raw_index,
                       'n_series': n, 'sort_method': method, 'flip_lr': flip_lr,
                       'flip_ud': flip_ud, 'reverse': reverse, 'fov_ok': bool(spacing),
                       'n_decode_errors': n_bad,
                       'window': (float(lo), float(hi))}
    per_slot = {slot: (stored.get(uid) if uid else None) for slot, uid in slot_series.items()}
    return per_slot, info

CFG = {}   # filled from the checkpoint's configuration before any dataset or model is built


def local_dir(d):
    # The 100 mm rendering of a cache directory lives next to it.
    return Path(str(d) + '_local')


def prepare_views(study, slot_series, image_root):
    # v30: the verbatim v07 rendering twice, at 140 mm (v07) and 100 mm (v28); CROP_MM is read by crop_resize.
    global CROP_MM
    out = {}
    try:
        for name, mm in (('full', 140.0), ('local', 100.0)):
            CROP_MM = mm
            out[name] = prepare_study(study, slot_series, image_root)
    finally:
        CROP_MM = 140.0
    return out


# ----------------------------------------------------------------------------- data
def load_study(cache_dir, uid):
    with np.load(Path(cache_dir) / f'{uid}.npz') as z:
        return z['jpeg'], z['offsets'], z['slot']


def decode(jpeg, offsets, k):
    return cv2.imdecode(np.frombuffer(jpeg[offsets[k]:offsets[k + 1]].tobytes(), np.uint8),
                        cv2.IMREAD_GRAYSCALE)


def slot_centres(n, k, train, rng):
    if train:
        edges = np.linspace(0, n, k + 1)
        c = np.floor(edges[:-1] + rng.random(k) * (edges[1:] - edges[:-1])).astype(int)
    else:
        c = np.linspace(0, n - 1, k).round().astype(int)
    return np.clip(c, 0, n - 1)


def augment_views(tris, rng):
    # tris: list of (3, H, W) uint8 triplets of one centre (the views). One set of parameters is drawn (in v13's
    # order) and applied to every view; translations are image fractions (design v25 rev 2, finding 4).
    h, w = tris[0].shape[1:]
    angle = rng.uniform(-10, 10)
    scale = rng.uniform(0.9, 1.1)
    tx, ty = rng.uniform(-0.05, 0.05) * w, rng.uniform(-0.05, 0.05) * h
    m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    m[:, 2] += (tx, ty)
    gamma = rng.uniform(0.85, 1.15)
    gain, bias = rng.uniform(0.9, 1.1), rng.uniform(-0.05, 0.05)
    out = []
    for tri in tris:
        img = cv2.warpAffine(np.ascontiguousarray(tri.transpose(1, 2, 0)), m, (w, h),
                             flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        img = np.clip(img.astype(np.float32) / 255.0, 0, 1) ** gamma
        out.append(np.clip(img * gain + bias, 0, 1).transpose(2, 0, 1))
    return out


def augment(tri, rng):
    return augment_views([tri], rng)[0]


def local_view(tri, img, crop_px):
    # Central crop of a full-field triplet (B1: 100 mm of the 140 mm v07 field), resized to img.
    c0 = (tri.shape[1] - crop_px) // 2
    return np.stack([cv2.resize(np.ascontiguousarray(x[c0:c0 + crop_px, c0:c0 + crop_px]), (img, img),
                                interpolation=cv2.INTER_LINEAR) for x in tri])


def normalise(x, mode):
    # x: (3, H, W) float in [0, 1]. 'imagenet': ImageNet mean/std (v11). 'minmax': each channel
    # scaled to [0, 1] per slice, as MRI-CORE documents for its encoder input.
    if mode == 'imagenet':
        return (x - MEAN) / STD
    flat = x.reshape(3, -1)
    lo, hi = flat.min(1)[:, None, None], flat.max(1)[:, None, None]
    return np.where(hi > lo, (x - lo) / np.maximum(hi - lo, 1e-6), 0.0).astype(np.float32)


class StudyDataset(Dataset):
    def __init__(self, uids, targets, weights, cache_dir, k, train, img, seed, norm='imagenet'):
        self.uids, self.targets, self.weights = list(uids), targets, weights
        self.cache_dir, self.k, self.train, self.img, self.seed = cache_dir, k, train, img, seed
        self.norm = norm
        self.epoch = 0

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        rng = np.random.default_rng((self.seed, self.epoch, i)) if self.train else None
        jpeg, offsets, slot = load_study(self.cache_dir, self.uids[i])
        views = int(CFG.get('views', 1))
        from_cache = views == 2 and CFG.get('local_source') == 'cache'
        if from_cache:
            ljpeg, loffsets, lslot = load_study(CFG['local_cache_dir'], self.uids[i])
            assert np.array_equal(lslot, slot), f'local cache slots differ: {self.uids[i]}'
        tokens, slots, cache, lcache = [], [], {}, {}
        for s in range(N_SLOTS):
            idx = np.flatnonzero(slot == s)
            if len(idx) == 0:
                continue
            n = len(idx)
            for c in slot_centres(n, self.k, self.train, rng):
                chans = []
                for d in (-1, 0, 1):
                    j = int(idx[min(max(c + d, 0), n - 1)])
                    if j not in cache:
                        cache[j] = decode(jpeg, offsets, j)
                    chans.append(cache[j])
                tri = np.stack(chans)
                if tri.shape[1] != self.img:
                    tri = np.stack([cv2.resize(x, (self.img, self.img), interpolation=cv2.INTER_AREA) for x in tri])
                if views == 1:
                    x = augment(tri, rng) if self.train else tri.astype(np.float32) / 255.0
                    tokens.append(normalise(x, self.norm))
                    slots.append(s)
                    continue
                if from_cache:          # B2: the same slices rendered with a 100 mm field (v28)
                    lch = []
                    for d in (-1, 0, 1):
                        j = int(idx[min(max(c + d, 0), n - 1)])
                        if j not in lcache:
                            lcache[j] = decode(ljpeg, loffsets, j)
                        lch.append(lcache[j])
                    loc = np.stack(lch)
                    if loc.shape[1] != self.img:
                        loc = np.stack([cv2.resize(x, (self.img, self.img), interpolation=cv2.INTER_AREA) for x in loc])
                else:                   # B1: central crop of the full-field slices
                    loc = local_view(tri, self.img, int(CFG['local_crop_px']))
                if self.train:
                    xf, xl = augment_views([tri, loc], rng)
                else:
                    xf, xl = tri.astype(np.float32) / 255.0, loc.astype(np.float32) / 255.0
                tokens.append(normalise(xf, self.norm))
                slots.append(s)
                tokens.append(normalise(xl, self.norm))
                slots.append(s + N_SLOTS)
        return {'x': np.stack(tokens).astype(np.float32), 'slot': np.asarray(slots, np.int64),
                'y': self.targets[i], 'w': self.weights[i], 'uid': self.uids[i]}


def collate(batch):
    return batch


# ---------------------------------------------------------------------------- model
class Net(nn.Module):
    def __init__(self, backbone, n_out=12, att_dim=256, drop=0.2):
        super().__init__()
        self.backbone = backbone
        d = self.backbone.num_features
        self.slot_emb = nn.Embedding(N_SLOTS * int(CFG.get('views', 1)), d)   # 10 token types for two views
        self.norm = nn.LayerNorm(d)
        self.att_v = nn.Linear(d, att_dim)
        self.att_u = nn.Linear(d, att_dim)
        self.att_w = nn.Linear(att_dim, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.normal_(self.head_w, std=0.01)

    def encode(self, x, chunk=None):
        if chunk is None or x.shape[0] <= chunk:
            return self.backbone(x)
        return torch.cat([self.backbone(x[i:i + chunk]) for i in range(0, x.shape[0], chunk)])

    def pool(self, h, slot, counts):
        # h: (N, D) tokens of several studies concatenated; counts: tokens per study.
        h = self.norm(h + self.slot_emb(slot))
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).float()  # (N, 12)
        out, start = [], 0
        for n in counts:
            hs, as_ = h[start:start + n].float(), a[start:start + n]
            wts = torch.softmax(as_, dim=0)                                  # (n, 12)
            pooled = torch.einsum('nk,nd->kd', wts, hs)                      # (12, D)
            out.append((self.drop(pooled) * self.head_w).sum(-1) + self.head_b)
            start += n
        return torch.stack(out)                                              # (B, 12)


def batch_tensors(batch, device):
    x = torch.from_numpy(np.concatenate([b['x'] for b in batch])).to(device, non_blocking=True)
    slot = torch.from_numpy(np.concatenate([b['slot'] for b in batch])).to(device)
    counts = [len(b['slot']) for b in batch]
    return x, slot, counts


# ---------------------------------------------------------------- leg (v09/v10 code)
ARGS = json.loads(sys.argv[1])
REFERENCE = json.loads(Path(ARGS['reference_json']).read_text())
REFERENCE_LOCAL = json.loads(Path(ARGS['reference_local_json']).read_text()) if ARGS.get('reference_local_json') else {}


def series_info(image_root, study, uid):
    # Header-only facts used by select_slots (v06 semantics: n_readable, geometry_ok).
    try:
        records, meta = series_headers(image_root / study / uid)
        _ordered, method, _info = order_slices(records)
        return {'n_readable': len(records), 'geometry_ok': method == 'geometry' and bool(meta['spacing'])}
    except Exception:
        return {'n_readable': 0, 'geometry_ok': False}


def pack(per_slot):
    # Same layout as v07 cache_one: slots in SLOTS order, a shared series stored per slot.
    blobs, offsets, slot_idx, raw_idx = [], [0], [], []
    for s, slot in enumerate(SLOTS):
        entry = per_slot[slot]
        if entry is None:
            continue
        for b, r in zip(entry['jpeg'], entry['raw_index']):
            blobs.append(b)
            offsets.append(offsets[-1] + len(b))
            slot_idx.append(s)
            raw_idx.append(r)
    return (np.frombuffer(b''.join(blobs), dtype=np.uint8), np.asarray(offsets, dtype=np.int64),
            np.asarray(slot_idx, dtype=np.int8), np.asarray(raw_idx, dtype=np.int16))


def prepare_one(job):
    study, series_rows, image_root, cache_dir = job
    try:
        rows = []
        for r in series_rows:
            rows.append({**r, **series_info(Path(image_root), study, r['SeriesInstanceUID'])})
        chosen = select_slots(rows)
        slot_series = {slot: chosen[slot][0] for slot in SLOTS}
        views = prepare_views(study, slot_series, Path(image_root))
        packed = {k: pack(v[0]) for k, v in views.items()}
        if len(packed['full'][2]) == 0:
            return study, 'no slices'
        if not (np.array_equal(packed['full'][2], packed['local'][2]) and np.array_equal(packed['full'][3], packed['local'][3])):
            return study, 'the 140 mm and 100 mm renderings store different slices'
        for k, d in (('full', Path(cache_dir)), ('local', local_dir(cache_dir))):
            jpeg, offsets, slot, raw = packed[k]
            d.mkdir(parents=True, exist_ok=True)
            np.savez(d / f'{study}.npz', jpeg=jpeg, offsets=offsets, slot=slot, raw_index=raw)
        return study, ''
    except Exception as exc:
        return study, f'{type(exc).__name__}: {str(exc)[:200]}'


def reference_check(comp, cache_dir):
    # Re-create three v07 training studies and compare with the stored fingerprints.
    train_series = pd.read_csv(comp / 'train_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    if not (comp / 'train_series').is_dir():
        return {'status': 'skipped', 'reason': 'train_series not mounted'}
    out = {'status': 'pass', 'studies': {}}
    if set(REFERENCE_LOCAL) != set(REFERENCE):
        return {'status': 'fail', 'reason': 'reference sets differ', 'studies': {}}
    for uid, ref in REFERENCE.items():
        rows = train_series[train_series.StudyInstanceUID == uid][
            ['SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive']].to_dict('records')
        study, err = prepare_one((uid, rows, str(comp / 'train_series'), cache_dir))
        if err:
            out['studies'][uid] = {'error': err}
            out['status'] = 'fail'
            continue
        for view, refv, d in (('full', ref, Path(cache_dir)), ('local', REFERENCE_LOCAL[uid], local_dir(cache_dir))):
            with np.load(d / f'{uid}.npz') as z:
                jpeg, offs, slot, raw = z['jpeg'], z['offsets'], z['slot'], z['raw_index']
            ref_arr = np.load(io.BytesIO(base64.b64decode(refv['arrays_b64'])))
            same_layout = np.array_equal(slot, ref_arr['slot']) and np.array_equal(raw, ref_arr['raw_index'])
            byte_equal = hashlib.sha256(jpeg.tobytes()).hexdigest() == refv['jpeg_sha256']
            max_diff = None
            if same_layout:
                thumbs = np.stack([cv2.resize(decode(jpeg, offs, k), (8, 8), interpolation=cv2.INTER_AREA)
                                   for k in range(len(offs) - 1)]).astype(int)
                max_diff = int(np.abs(thumbs - ref_arr['thumbs'].astype(int)).max())
            ok = same_layout and (byte_equal or (max_diff is not None and max_diff <= ARGS['thumb_tolerance']))
            out['studies'][f'{view}:{uid}'] = {'byte_equal': byte_equal, 'same_layout': same_layout, 'thumb_max_diff': max_diff}
            if not ok:
                out['status'] = 'fail'
            (d / f'{uid}.npz').unlink(missing_ok=True)
    return out


def oof_check(models, comp, cache_dir, device):
    # End-to-end check of the weights: re-create the reference training studies from DICOM,
    # predict each with the fold model that did not train on it, compare with that fold's OOF.
    expect = ARGS.get('oof_expect') or {}
    if not expect:
        return {'status': 'skipped', 'reason': 'no expectations'}
    if not (comp / 'train_series').is_dir():
        return {'status': 'skipped', 'reason': 'train_series not mounted'}
    train_series = pd.read_csv(comp / 'train_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    ref_dir = Path(cache_dir) / 'oof_check'
    ref_dir.mkdir(parents=True, exist_ok=True)
    out = {'status': 'pass', 'tolerance': ARGS['oof_tolerance'], 'studies': {}}
    for uid, e in expect.items():
        rows = train_series[train_series.StudyInstanceUID == uid][
            ['SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive']].to_dict('records')
        _study, err = prepare_one((uid, rows, str(comp / 'train_series'), str(ref_dir)))
        if err:
            out['studies'][uid] = {'fold': e['fold'], 'error': err}
            out['status'] = 'fail'
            continue
        CFG['local_cache_dir'] = str(local_dir(ref_dir))
        ds = StudyDataset([uid], np.zeros((1, 12), np.float32), np.ones((1, 12), np.float32), str(ref_dir),
                          ARGS['k_infer'], False, ARGS['img'], 0)
        x, slot, counts = batch_tensors([ds[0]], device)
        m = models[ARGS['weight_folds'].index(e['fold'])]
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16,
                                             enabled=device.type == 'cuda'):
            logits = m.pool(m.encode(x, chunk=ARGS['eval_batch_tokens']), slot, counts)
        p = torch.sigmoid(logits.float()).cpu().numpy()[0]
        diff = float(np.abs(p - np.asarray(e['probs'], np.float32)).max())
        out['studies'][uid] = {'fold': e['fold'], 'max_abs_diff': round(diff, 6)}
        if not diff <= ARGS['oof_tolerance']:
            out['status'] = 'fail'
        (ref_dir / f'{uid}.npz').unlink(missing_ok=True)
        (local_dir(ref_dir) / f'{uid}.npz').unlink(missing_ok=True)
    return out


def main():
    t0 = time.time()
    comp, out_dir = Path(ARGS['comp']), Path(ARGS['out_dir'])
    cache_dir = Path(ARGS['cache_dir'])
    cache_dir.mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)
    receipt = {'shard': ARGS['shard'], 'status': 'running', 'opencv': cv2.__version__, 'torch': torch.__version__}
    write = lambda: (out_dir / f'v30_leg_receipt_{ARGS["shard"]}.json').write_text(json.dumps(receipt, indent=2))
    if ARGS['shard'] == 0:
        receipt['reference_check'] = reference_check(comp, cache_dir)
        log(f'reference check: {json.dumps(receipt["reference_check"])}')
    test = pd.read_csv(comp / 'test.csv', dtype={'StudyInstanceUID': str})
    series = pd.read_csv(comp / 'test_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    studies = sorted(test.StudyInstanceUID.unique())[ARGS['shard']::ARGS['n_shards']]
    by_study = {k: g[['SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive']].to_dict('records')
                for k, g in series.groupby('StudyInstanceUID')}
    jobs = [(s, by_study.get(s, []), str(comp / 'test_series'), str(cache_dir)) for s in studies]
    errors = {}
    with ProcessPoolExecutor(ARGS['workers']) as pool:
        for i, (study, err) in enumerate(pool.map(prepare_one, jobs, chunksize=2)):
            if err:
                errors[study] = err
            if (i + 1) % 100 == 0:
                log(f'shard {ARGS["shard"]}: prepared {i + 1}/{len(jobs)}')
            if time.time() - t0 > ARGS['time_limit_s']:
                pool.shutdown(wait=False, cancel_futures=True)
                break
    ready = [s for s in studies if s not in errors and (cache_dir / f'{s}.npz').is_file()]
    receipt.update(n_studies=len(studies), n_prepared=len(ready), prep_errors=dict(list(errors.items())[:20]),
                   prep_seconds=round(time.time() - t0, 1))
    log(f'shard {ARGS["shard"]}: prepared {len(ready)}/{len(studies)} in {receipt["prep_seconds"]}s')
    write()

    device = torch.device(ARGS.get('device', 'cuda'))
    models = []
    for f in ARGS['weights']:
        ck = torch.load(f, map_location='cpu', weights_only=False)
        c = ck['cfg']
        assert c['kind'] == 'timm' and c['norm'] == 'imagenet' and c['img'] == ARGS['img'], c
        assert int(c.get('views', 1)) == 2 and c.get('local_source') == 'cache', c   # B2 contract (v29)
        CFG.update(views=2, local_source='cache', local_cache_dir=str(local_dir(cache_dir)))
        bb = timm.create_model(c['backbone'], pretrained=False, num_classes=0, **(c.get('timm_kwargs') or {}))
        m = Net(bb)
        m.load_state_dict(ck['model'], strict=True)
        models.append(m.to(device).eval())
    if ARGS['shard'] == 0:
        receipt['oof_check'] = oof_check(models, comp, cache_dir, device)
        log(f'OOF check: {json.dumps(receipt["oof_check"])}')
        write()
    CFG['local_cache_dir'] = str(local_dir(cache_dir))   # the OOF check pointed it at its own directory
    zeros, ones = np.zeros((len(ready), 12), np.float32), np.ones((len(ready), 12), np.float32)
    ds = StudyDataset(ready, zeros, ones, str(cache_dir), ARGS['k_infer'], False, ARGS['img'], 0)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=ARGS['workers'], collate_fn=collate)
    rows, done = [], 0
    with torch.no_grad():
        for batch in loader:
            x, slot, counts = batch_tensors(batch, device)
            probs = []
            for m in models:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                    h = m.encode(x, chunk=ARGS['eval_batch_tokens'])
                    logits = m.pool(h, slot, counts)
                probs.append(torch.sigmoid(logits.float()).cpu().numpy()[0])
            rows.append([batch[0]['uid']] + np.concatenate(probs).tolist())
            done += 1
            if time.time() - t0 > ARGS['time_limit_s']:
                receipt['status'] = 'timeout'
                break
    cols = ['StudyInstanceUID'] + [f'{c}__f{k}' for k in range(len(models)) for c in LABELS]
    pd.DataFrame(rows, columns=cols).to_csv(out_dir / f'v30_leg_shard{ARGS["shard"]}.csv', index=False)
    if receipt['status'] == 'running':
        receipt['status'] = 'done'
    receipt.update(n_predicted=done, total_seconds=round(time.time() - t0, 1))
    write()
    log(json.dumps({k: v for k, v in receipt.items() if k != 'prep_errors'}))


if __name__ == '__main__':
    main()

In [ ]:
# Launch our leg: two shards, one per GPU.
import json as _v30_json
import os as _v30_os
import subprocess as _v30_sp
import sys as _v30_sys

def _v30_find(name, root=_V30Path('/kaggle/input')):
    hits = []
    for current, dirs, files in _v30_os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series', 'competitions')
                   and 'rsna-knee-abnormality-detection' not in d]
        if name in files:
            hits.append(_V30Path(current) / name)
    return hits

_v30_comp = next(p for p in (_V30Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
                             _V30Path('/kaggle/input/rsna-knee-abnormality-detection'))
                 if (p / 'test.csv').is_file())
import torch as _v30_torch
_v30_weights, _v30_weights_meta = [], {}
for _ver, _arm, _k in WEIGHT_SPECS:
    _name = f'{_ver}_fold_{_k}_{_arm}_best.pt'
    _hits = _v30_find(_name)
    assert len(_hits) == 1, f'{_name}: {_hits}'
    _cfg = _v30_torch.load(_hits[0], map_location='cpu', weights_only=False)['cfg']
    assert (_cfg['version'], _cfg['arm'], _cfg['fold']) == (_ver, _arm, _k), f'{_name}: {_cfg["version"]}, {_cfg["arm"]}, {_cfg["fold"]}'
    _v30_weights.append(str(_hits[0]))
    _v30_weights_meta[_name] = {'epochs': _cfg['epochs'], 'lr_backbone': _cfg['lr_backbone'], 'img': _cfg['img'], 'backbone': _cfg['backbone'],
                           'views': _cfg.get('views'), 'local_source': _cfg.get('local_source')}
_v30_weight_folds = [_k for _ver, _arm, _k in WEIGHT_SPECS]
print(_v30_json.dumps(_v30_weights_meta), flush=True)
_V30_OOF_EXPECT = _v30_json.loads('{"1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339": {"fold": 0, "probs": [0.046531, 0.053126, 0.068197, 0.029392, 0.038111, 0.032749, 0.073481, 0.102802, 0.229236, 0.116068, 0.061547, 0.106499]}}')   # v29 arm-L fold-0 OOF of the reference study
assert all(e['fold'] in _v30_weight_folds for e in _V30_OOF_EXPECT.values())
_v30_ref = _V30_WORK / 'v30_reference.json'
_v30_ref.write_text('{"1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585": {"jpeg_sha256": "fb3ed5f17b44209ad7e667b6e9c61969fa7761bb806a9afe5318c54efe4fd6b4", "n": 156, "arrays_b64": "UEsDBC0AAAAIAAAAIQB9b6ZV//////////8KABQAdGh1bWJzLm5weQEAEACAJwAAAAAAADwiAAAAAAAAnZlXc2NJlt9v3gtDADQA6AkaECQMCZDwhPfee+8IECBIECBB0HuyyKIpX112urpr2s12j+mdmRi3LqQNaUO72pAi9AH0sA96VoS+gRLskbalCL0IwYhz3e9knrx58/xP8pU/5gumALKLHHHLla2lFlfN5B7viLkCJnd5o7XdKq4XNlrlSvu6vVjfqsDrWyvFZgWe88TyBQFTCf9mBMwT5v/PjwxGiv/w26+MKEL0nnz3mo8iqGTl/Q0XIOjg8YMDEUAAoXa6F0QRgM5fbajbFqdRzgCAYhiJQgaoKvzr737pxMCI5+U3n8swhBjef363ABBMX0+tedv+Lr22LWiJjUVVElp0yCVSYpDHdXdTULxx8cOT92E8Jks9fPZ+AQOc/PeNNxYUoWZalp0KihCazdjSFQ4Bs7tWwwq0qMJod+ABiuL76L14erz0+uZlltydLJ1dvPBhxPTL69LLEgbUu+e50hGGTJ1t5w+ekBBcolGpHuIA6LDYMwECimLdI2M0/IS3frt/WaRNWArvDi5zHSOJF/b4UR0PIulU0L/XAww71eD26TDoWd7dbl30AHTYGUyGewAO3z/GHiRJ3PW7/bOVIcnp8f7adoQmeJyxptdDvcRyJp+InjNAYqOS2j3mIszi8Vb1cBygfE80FhyF4dMmeMNkTTKhqByVxtTXUWEuKB2WbDks0Y3UxGArnogU3rLR2lEu37wQA9HyfrNyNgcweSQZdrFRHHFwik3rtCVMfF/BN6k+NCu8OvGYYtVsjKyFpmc3A2Ff4VMh8fAglt54pgLms2apeq5AiPpYPOKFfjoZHO4g3RM2enxBoVC3agmGTHKBuWZxB+pRqa7qCniKLw2j1VY4tvHGjzp2atmVGwfo9WWSJo8WxXdOTLNHGO640e+NO1WGJUcq4zLIbasWV7QWUZtrHp9v8bmds7gVjW28TmOWw3q+cu0CQ8FiSu4xAozK5oyMDHm9OqtjddliXnKk4367wV+32JMbCZNnzen0VZ4FRfn1bHL9RRVv3mksFs4igOEoJNVmG4qnc7ij9KFwTG3xNlec9oKlBN+YNbhmc8Wacbe74Q8EylcxeXlndXH9yTrBsbdRLRwkAMO+mDJZbQBHneUwaEOhmN6W3t8I2VPO9eZl3heueP3Zq0jIsRkPuc8vM8rKWauw8Wizw3W2tZrZhrxlMWExGQBGn5ud6O2NBnX6x2fNXCDhXjx72Yx7F0PBwieBqGUjEfE9382rlk6Oykvn2yTvg91151oaMIz5nFhlALheAX+0G/Ja9ZPLrXI45o0/enW86E1Fwo036YSlkUpufLpeUlaO3h49ud4n+04v9o62S2DYmFlKmrUoRpsRmsb7ox6V8vRga8Ufc/sieycr7mQkmLhZy5m3ctn1dw/XlI2TF5Xzq0Ny5PRqP7tZAqOWQsmsUMD2+TJ2FzXolguWDxsrnoxJ51jeKtsSLodzdzltrMdjjZ3DTUnj5DKRW93rCG+dNUKBHGAYYimDdAHg+2bmhmlUV0QlCJ5slD0xizK8vl22pV3O1INmQdeMOWr7h/uCrav9TL15RsisH+xECgUwZoznbHolwGhs7gC927TmFjvPtoqugFPub25XDBmbIXyxXZY3g87W7vkxZ3e/kVrfvsTnVnf24uUCGLUloi7zAorr4/GHqV2qVkKWPdsvmiNOWXhvv6RuOo3hs53qfCvsO7u5Oh3f38+nDndvscRa86BYWgZD1ljAblejaC+bM0ylCFtFq+7iqKAoB8SundO8KmnSBDcOa5MVr7NyeHvWv9OI1g9PnqCh5fL24lID9NrCSZdZAVefSc5oN5mzWV9Unp+EmPWC3Fg/TgqDVlVg9bDal7ebTbHLHcreWr6xef4CuMrp1ezKBuix+2JOgwKg/UzOeD957GBjR3N55uxuLZs1lVPfQNmmcWbPysSEz210XNSxlXKhnD9/AnSlVCGysoGQrcGYV68BaB+TPTnWNXDc2jPvHdmwlXpUlzvT44s2tdl3nUKDfofatLEEksVsMvrgCRCWkglPbR3B611Rj1oFeQZjcpCCPzret9QvJKj7Ys3puRKD8dSC3XisBsyCT2dPOoC4GnGFGweAtpdKuhurCKow29x6NQBdzNFJRjdaf7Cvdx7MosqrQ+f8vgCQyxq9taIDpIpH6wiqQP9W0BZeWgGE1lLOvZkDQOB1BnQKgJGYU+NDXaD49FgjMMyB2dsr7QyXD3BLVoVBpEHQjNfo8ugArhm1RpJJgKa2F02bSQSMRVxu7QIAuLHxsYFeEHh5Yk6J+GDw+qFUKp9BsFBErfEoENQftOXLcJzzGWtxLYgA223SWvMCQI3qIlo5zCP9Y+O9vUD/8iLSkvNA58GNoaDjAJylYYiEpQA1JYM7GQECzDuRypoJAVOP3PmCEebDQDSsXYD8AHMctj/34iZzop9G8OVbYys8haD8gqFWEcLnk6q1dTYAk4eRwqYcQTsvW6lVGeSMWY1aAm0ng9nbD8ZuHuWeuSYRXHDbdOpmIriRZWOjyEZQyra/XBtFAPG8WT3mIABbfNA65UGOF4kG5qAlj4wy6KDr6pPUS28fgmlLmiv3AEApe55mZQQm6Lu1lTUqfK7wydbBIIJiipvjEwa0A7F0gN3O42PMIRrw/GMwa9sCCO0PZ9WEB+b9VsXbUMK8L31j3lEjCEJ4uXGZhhbhr9VgXkcQlMnobxsMh2EI8vK9wS1rASTxq/WgBOb3zqdrnoU4goB1HV+xBJ/T7bMEV5DDJ1SMEPSLdHF7R6FFUSIOB0h/411wz7dQ7MuHzaA4gyKF6o1VkgHIaC2nmaohSMcDLUv8BvJs9xCvCPUHmGBOTrUtjkQkoOyn0wrv3DGJGUleLfJLgKzMPg1KkwhiDf50X3UEEH5T59H9AkNQq5ArXYYdxk0Mz8xCHiOSKETUVNjKxbirLBnzcmtjpowbXnj6actQA8B8+bNXmQ8ExBQp5NTPqEin1eKSLRERQBmdnp2BFuugUAi4eOh5bIMTNS0Ez4r76sbUROXzr85C74aIro9fvsksMxFruprXXPKRPrnfJwkPIaBncpY72d0Ov7OH1JGP7SRuOd6cdO/o0cHRtxZO9dOP14GXmp74q/fPM0E1cGUXM7K8ERmWBt1SEw9g/TN8DqMXYHhSZw+ZUMidvH2/2NpWHpx/ePDtt6X55PvPr51PUgOxu1e3MbUDF8+lkhJjEJmw+KxymQzBBufnuAMDCMCTuqgd5MLS1S++vvrVG33t+suH3//iuTb45sMjz4sH4+XDmwufsEQJpPweyXwSjHo9JjHbiODHpcLhPji/MEpnN7GzsPr0u292r7+0lK4/3L755jOL78WnD9JvX09V9u4uXcK17kDCZRfPRcC432cRcdQIcVIu7utnwHnd00PuoBbLz3/+XST4S/viow+3P/nsL5zeZ58drb3/fLa8eXdrkTRpobjdJBJE0Alf0DHHViIdkKf33vPdZGJ3qfTyq58ZNb/1Zp98+vLJxfde77MvNldffDlbWb87N9h2+uJRu0IiCGMsZ8A3z1bc8710yBOoPSQ8rVh+98UTEev7UOHZF9vO/O/84Sdf1UyvvptZqd1Fw/GzwVjcKbfMQ97miosm5QhpUirupbW/Szq1A9+zuPzmJwUB670/++zjqsr4c1/4yTcrcz/9S+7a6iO9OXw7HE1bNRZ+GMex25OiCRGMXyrt7YHfI7GHhsdTloufvF6ZnboJp59/bOjnX/sjz7/eSv7k++na2pXTk/iEEY1YjF5RGsd1OLKiaSkgsWSKQfogXD97aSRC12L4xettweSRP/nqqxcFwaUn9urbq/rVX3KWNy5jQeXT0YTHpAiK8zi+3Z2Ss8WANCleYPT3A5REo3d2dMQ0z189nWFt+WJvfvZ1kb3riL/59sxa+mZmcXNvSc0/Z6TsOmmIE8dmra6gdGgOIU/Mywa64XpDofUQcXjH/IOX3wmmWi7HJ998SE83LOFPvssqIh95me2dkIh3NJp0mhSWySAmtLj985MihDIhlA5TaQiORKURMLxhKnP5G/70tl35+Osnoem6yv7iZ1ax+z0nc3Di4fOOGXG7Wa5gBtB5ndM/xxIj5PF50Vh/L0A7qDQiRjBOa8t/mOc2bfL9L8597DpX++Irp8DxYiJy8jAp5J72BW3aWd2kD/CU9oByfA6QxgRzY4O9KEqm0TrweAWPa/sryUxdZst9OAryWkL1i68Wxc5HQ4Gz23W18LIz4LTPaVhBhK0wegyj8P2NCvhjQ3QASNQeIgHHl0wb/41kZnkw7D27jvF22dGbL6oa+yXZv391ZTfcYGGH3yaaiiBTEoXJOCpGCGOz3DE6DQGknp4OPG7UMCn5t3JeAZdSrr7NcYvE+u7HutV0gho3H955AjfAaQ35OOwsMiGQavWjCwiOMTPNoPYiCKkbfj/4Hh9r5u9UvD3gm019V+bm0MTKZ+sBXQGRV64feuvHQGGOBVjsGhjii5XqURWCDrGnR7vpcP52dJLxGCix2b8LS84QD0/2iyvhKTJU+my/oS8hPeWXd94rHTJqyoemIwcIkS9ZEI/bENA3zmLQ+uH6QaaQCURQ5nF+/ch6jFgl4q9/azpGiIXXOw9jGwiaev8yUrUgRFM5xiktIUAo0syxnQjSO8Zi9ML1B0fppHR0giU+7/u/v2gifMvcN//0sQW5y+0PX9Vhvn73vvrcjQBdJGXciiAIj2/kis0I0j3OZNBh+1hnd3dnJ1IUc3/a+ucC0p8Q/P6P/w4mnODywU+/zyOI6vTdxbdGBFEYYvmXAQRhinQikwGBH+AQizYAExQJ4lQkaOU8tPzJhRDLwj9mvvIjqN13dnjrhfnPYS2dwHw9Z5CvPIB++hbGAj4FgmAs1mzfYJvvIXf2INJV7p7wn8QI2lT/kv97eF9a/9bwFbRDZrf7ySSC0MKBwMUszH92tzLPh9z43Nhom8d3wRUcYTzlbfD/ZQDBNVtf8f7rKIIO/8UvtP/EhHlq78vY77phni0cl76D/UWV6UhjDHK9swJuH7RYF6WLgpB+439k++94BI3//T84/0cHgnb88j/u/TcSvN/66y//EQet8cXb37fPxxfzTRq05NkZDhX6w/eQuzoAqoxRUQCzaeMMimJo715poEFQ1+1AO9sjhHF828CEi/z4h6Lo/VV8Ks9pc31FkxtalOafDmHwADu71sIqHwGiGrOd9RGUhP4Ih/U/2nYI+vIRAwbb54nEXngJTAnUHhzkyWVtDF5HsPULC/QCH5wh/wD+wGFtHqocnksfo0Ir5akiJMjzbY5gJ3xmwmeNdcJz6qYr0dGOR+mcuAdxhHY/UNwPDjCVULI2D7u7YHak+2D/xRZzrA8GJ4lHY0OQ57jFoX7Ik/weGdYGu/oI4Ae+7YBgsyQOgnio6iqhNQ4M3OA3lEchr6sux7iQVzqs0bYdsIjMlDY4yetG73ncPe948vh8m4Ligv/lqGaFgWf/+fZwFg6c7dHxMh9ylqAnqIQDMKWY9Qy1t2N4U2OwHxhG6Gg7wIdrkeO3TIxQuSsXGmQUW3n3q7Aa9sd/8ybf3mZxh+IpNxxIkVkT4EM/nRNjsx1wWHFEIhGDci2Zju6+16Idtc2t6iUVEJZ9ZbePgKLBxXOdFwfQeDq3UsABoAklEzYUQQeVowIa5PEdBBIORei5XGbriwJGWs1XG285GFZ1O025TgwLGv3qDHwRufVaaaUbRY2pgDMFpylLNjA1giIYEX78ePiaSj7v9odzPHXNrs28tmCEqtttSo+hWNKg1caYKFZdXl9aHIPj4fer6hQEZQ8PM6YxBCPB2U/AwHxJNb/87k3/aMvi8D6tYl0bTo8tJsaIaaPB4JFDf4nmdlSIYg5/QNEYBJBnDPMICI7S00UlYUBT0ynjzz5IBC2nxX5+SRpq+ux2nwfrLpkMeocVpaykDosBDYpzOv3zxSmAzTDG+oUUgKP29VC7MOBoaYzhZ5+6Fesul/34dmRq3+12uKq4garVarLEMWrJ9yTjdKAdPm1InpECHIcxOSKlAxytj0qj4oHvyGALPvp0SX3mC9n3bmT8pstjtzbJ4xWn22LOYAMZ42lCH0C7vNKYNqEE+JlxFlPaD3D9/VQoIEDxSmcOXj+v6S9cCfP2kV1YsNh1xvLIZMhoU6vi2EjAuhOU+1CadcZldukAcXpoZJo/Anl6d08/EdQfm4zRh49WLMfeknFnP6lYMrn1pkUhz6f1KTWpbqbbvu1fiKADRp7RZjZC+TE4wuVNAHw/rZvaRwKrn+jtsQcPSs7j4Jpxp1nQLJt9OnXcLnGpAlp1kjnlN2wkFEniiJqvsplMWCdraEQwwwKEfnpXdx8ZXXluVwbPj4q+B0GvcrlUNMbNMZMuElOYFV63OiSZdeoWK+pUz6RKLHEYTR093KFRMX8aRxigd3cPdqLLr2TKzGolG9jV5eK+cNEWMAZssUhBZZB7S7qIRmzRZ+rmwNC0Vjpn1ui7qLxhhnSeRSTS+npoQxSssh1Yy/uTucTdQX3ZHy56Vsy1rVK4otYulI60EZPUaE+WrUEWTyOeVYsNdPrMGEskYZGJlCHW+DAFWzxbWs2mG9ncdfOsEA/nwlfLm/vx+IrGubB0q4u5JRZvasPtn+XLRXyD1DTQN8tViBWsLiJ5mDszSsHV3jXTm8WbVP1xrlxONXLJJzvbDwKZNVVoobmlSYakvmD1JBJbEKr1UofSxOidmbfqlSwakdw/NTNJwa++aS3dNE4Klbt8vBJfL6SfHQZavpV1eULROLbG/PJIZnF7MyaSGFccSYuJ2c+R+s3qaTqRMjDFm+jE117v3Hw4uk7kXhYSj6uHy85X57sfCzvHkkyw+b5x7BXHysmLnRxfZA8uxZUOZt+kJeJw8fqJFPoAj0XB118fv/z86iYVeV5de79/XDKcHy2/b+xv8/OPTp/tPQjOlc53njqaAv7a8VLDGGT28t2yeILfRyANj7MnO/Grr44e/+TmYdpzWT15f3JYVO7Vcu9ax3tTS4/OPoa2HTNrr1rvPfnZqd3b3GnNNkF1ZLzRFI9OGGCxWWNd+OXnW7dv7i4LppOly7cne5W5jXz1zc7u9uDu69an8Zp5ovSm8WYpPjVSudt6cOAcp1jS0WSUR8MPM6dZI1347NXm1ctHJyVRq3j28sF6erBVKD0+qS/jdt83XxRXNJTGh7vX9dhIR+rFo4sj5wAmyybikSkyrn+CyWTQCIHr7ccf3q4Uxzdztx+eL8dwzWLx9ZPtMFr95vh5oyJAW999/PIo3w2iD96+PI71AG44VyjwCBiNCfkemJ9D5733+TQVNNxX10aF5D5Dcm3j9xYbwiH/7x/AdndgtY6AUanG0+Z7+AHLfZ4Tzd9bZEA68ecn/y/yB8OoJJfahzNia6Jth3NH6XaCJItMgXs/PKHi/sGOXsKPQfCDHgBKtaLaViVzoXCinaC5+9vxtmXENgqd0HZxpxxtEPQP0/6c+f8P3sPhadvbAhJ/yDkNzwWGpG2wvdvQ2EwNQTs+bzFD9YPgR3oZ9wSGR//M3w9XYk4orMO8puBxNDBgMDPG0E1DqyzGre3AJZlFGxva7v7BUWKbIxBwf8bbDkDOKBWv9sF6P6lSRNp+DCKdrG1dPvUMtMbltEMEWxzgzE9S2w0S8R3gR3xz/3ABlukge3wUyuABcK+no1Z4yxY3hmeh9SxFDVL44KRGMt3eNsGRyeS2KoM6pi1kyK2tM4NiEaD1ai7qG2v7qayGYLuhgq9sgnH4Yl69CrbPkQmZDMgTe/q6ifcyCGvLo7Gd7V2XbA90nGZCAaUGgPWzvYNUDwD5WqIa7wQg5nTIlV1QPsvF/AnohzQ4QCXf6697/SFej/ldsocdjMu187LcCwg7Z5fnizzYn/XlepEFQMJmlClhuTy7IIfLHtQPw0P9Xe3tGxy+zTuqxSOz9GRauLvzsKksgIH1k6uTZS3A7zTrjTUx1A8ug9rARsC8Ti5mw7KHOjoy0IPB2Ygn4DCAFMsbz/yyVZd9+ezpha5JFpTObq93fSi9tr3V2lcCQtSg0NiECKoyKcS8ITiNxsdH6MS2jOsgQP2xlDx6mZYkjhyly6eX+g2+Knv96Oooh02XD/Y2D2yAGjNJ1C4pgtOZFRLOGAIGJyaGaTAOHLGDAPVPM3zzNit0fx0tPnx8bV6JGhqXd3dXO1TR2tFm/SgIGGELXx5UAYreqJHzWFCeT06N9XVB/UYktfVT1nz9Pjtv+UM+cXV3Z1o6ce1ePX10tjunXT+olfcyGNvvFKuDeoSu0hlkPA6CG2KyRnt77vtPhP3PGa7fheZsf9vIPri4MK98mVy/fXxzdhJwrB1Us7vrdKHXp1AHvWBoQW2Xz8Dycajdfh9U70RSBx5DKoaTS40o8Ke90vFRWLv526XVhw/Pty+bvvJWKrazzVG6gzplMEUclyhdKr4akCA/CssXFE/o6MDhkHXjivypMfPbi8rGgrL08vfra3eVcOr2aaywZQvv7mmNJr9O6Sv3seflbv28uYMyxJweG2SgKIFAJOLx4DSsf/RF7Sd/eNZQPas//e73J/Fjiyjw9utscMsa3TmKmfR+Mc+9MiWckdmNUmtP9+A4a3xoDI8SCVCC4sHTnW+e/vy7L3712ebW6w+/+OXvznWL+4H8x+/jroNU4fRk2anxCebta2IZS2I1KB399CEGgzU4QcKIBDyeSADf2D9mX//y5A9fep8vffXNz/56T1/YDJU/++uE/zgX37ve9WsdWpdyXaWbkKoWZF4mfXCgd2xwvBPyBOgC/dn808L5r4K//lT9Zfo3P3/4x5Yq5zfWPv4p7t1POvbvDrxKbSguXJSbJoRsnUTPow/S6BMDI90YlM8deMgvHBaM18pP3yg+W/zyV5F3u1p/IFjfeRV21HRCd6HgWrB4dNKEycIZ5aim3PNUBpVG75+kwvGHExCPfi3eMClX5VdPpY8jxkvX47ranUieLX60q4wW4XSpaVGI9IKFhsNo4HLHWXEhnTHQ29fLomF4PJ4E+c/lGYV60fq7S8V5Wl0O/H3SlErYN8p/I7fPCwSqxsGCVSjkSesWuYE9PjGbFNGovV1D/VM0DH5/JCIe/SCzSZRh9980dNWAOlb+DwZ/M2/Orv1xLCWSyrTVBntJOC+QN1VTIfbEzGxN0Mnq6xvuY1FR3A/9f67yS1S25N9G3GsWhaP5x/Gl87LR0/yWuL5gWygcZzpfSITzqecCSo3LYgtbUzjpQO9A/xQFLsMEAh4HNp1xidK89a0gv25Smh89J6482HU4316BdVXYuv9Bgz1XiefX3nWjB3NstmirB5EM9vX3ThF+4PEgvFiTaw1/9bhn/5lJbf7PJ1CQPrS7/lMaVGzVyON/PwluHMaFjc/xoGaZ5UtPCAhnYnBwcBoF8P0RCbCuIWtMarj8YwajvrZtxLh5szJ9+jTcujMuhHZP128vSyvN3FJy62rj4Py4lR+orTQPzwtGpMu5vBKYBoCiNsph/YkZzY6VTRXKzOhViZM7e+WxSenePmocnebL1XQhs3K6tntxtF6gLS5t7J1kNQjZVq762rxKJ+qCvN4bLDdE6FhKpw4d3WqSj81q2+ZhfesknouGSrnC8cre+fZ6titdgBkno0I6rMV7nixf4MH6FlX5vbkaA4xkdarAyfGU+YFOp20dVBOHmaV4JJp2Hi41zzdqiyRfZmN/P69C8Pp81Q3Xd7JEPkOA7SviwWSuFwzA9gMnWyPqY4tJsbOzGD6LbUV84aTlILt2slItEZ3Jxt5OWoHgNIWyvc2LF2ZhHYeTrhXi8W7Qmwpr46cr9IVzt0N+tJUKbCY3HW5HyrCTrB1XylmCOVA/2E0pEEyRL9snAUqR6+Zh2sTLDtJRHxnQ0x5b8ChHll3bw4rmRtC3WVhxuawp3W5kdXdxMY03+qqH2wnIL6RLdibkZQYp5HGSZWvYRAA9OYvDth3Ei09Ny6rcitNzmq86bYaUuuWrbGczeZzBWzpsxSEvjxYtEwDtUlsVMH6ctBnxwLqbnNK61GUJOres8i+Y8xLXXnwt6tHlNQ1/vh7KruL03th+M6FBULmvZGFBXuvUEGAdLt9NeDgYoOS1DnmMjc4XFUaJLM4113K5fFGfla1lEy1Pso4ZbK6D9bQeQaXughm+v06NSwV5nKyR8k1hgJTV2qT2cZRXUKhFcz6moRCI5WyatLDRKB15UnXMajLvrieVCCZ1xts8WeOSt3lJMeWbxFBKQmsRq0ZRdkquEvJdowr/gtcm00VmluvLR9FkBTW7TcvrsQUEE9v8Zvj+CEqbAIcCbD6dizAxlBjQKMWqEZQZlyqEQn+/zDTnsEiN3uml+PluIJxBrSV3ZiU6D3mL3cSEE29OC7sNUE4kG2BAP2a9Wq4cRkciWo1E4u0UaebNEYlNzfS59jdscTsWuNiAfqYQvMrq0Y0CBBUoJ1HIT2aLnjav1+kl6gl00K/ViaUR0rxS5FwWmdWTLk+jaY7pcaGbrSOPfwLByewR5SAUIryF8TY/VYq54AFmUjlFhmHQ51WrRGJPh1Aj9G9JzGK23b60Y4orcYmrzRO3dwQhKP0FRW9bMGqZbZ6djwS4kLcawyLLGOj1y9Q8gQc/axdmE3MWNt/iKWybwlJC+nbrJOAeQMi6+LqyzfNsE+2NpOlwwMNGAeoMRpWWYUCPy1R8vh2bsc6nInwbc9YRX9y1h6TE7E3rJOYaRDr16S3TAARZ1rauBayU3dfmPbGQxToCqGGpUcz3YTO6uWxkxj4yFyikGh6fiFy4aB7GHIMIRZfeCrR36LpnaW1+LGTxseBwRlLVVrQf9CR8/7LMs6Ezel14TeKmzyfT4WW7m9udO2puZ2xDCFmVb2UnoaAkTXS1+WGvwQ7lJ+pZLOxmekGXP/X4SmVEZwxJedplo8n8yUJd65zuTh+sbxbMwwhJkd1ITLf/r9d3r4f7nXoYNoJaApVEigY6FytF254J41pt4oWkky6z7LgzNtNkZ3hnrV7SjSAkWbIagh1GsM57PU63mQx02L4hVzGWOwHl0JWTxfRQP81IVDwjXWz2GayzGibFv72yWlYMIx3i+KJjrL2hdr+fh3QZDLpu2L4iFjEskgHZxIvo3GrcpJIjc87Ke+a0Wr2GLZ0gu5ullWXZCNIh8KeMQ+B/FyIEvngWD70JLGqrBo+QRBz9gp2DMWYmZv0yaRdLtWCSSwUMinEpnizyexHihNIio95XSPc8jsUdhRUWmDIKlXMYQuRP8acNY+ggb3TcLBNSxmQLNoWBN0jWpPyRJIcGiMNCtajnX3l0YroXBgImVGwx/KAIvFHmqHoI7eUOjUgFHNLA3ILL7mb3kRRhh9fP6YE8XzpP+9ctVSiMu9r8qGxCMIwieM4QY0ROB1R+/xBHyCT2Tovtbs9ET4fIZTSbmZ0IkcEXCXr/tSYEXX3E9mgMigc50C2ON8AYme8EPcL+AaZ0BNc7zDcaNcPkDoFaoZKPkEDHOH+GS//Rli6Gu4+jo5/aB+s9wOjvHxrEIcSxQXofsxN0Uke5fBYZIwxNMCeGKfDGCKOvG/+jPWWA/jCOFEpnexx7+6h9sE7AD/dTacNE0NE1MD45TABYF71/oAdyuO4u0o/x/1WXolCWtI866d2dMCCsv6+7kwLHg0LvG2jvBxPInV33geLw7Q7/T1BLAwQtAAAACAAAACEA/fz3Nf//////////CAAUAHNsb3QubnB5AQAQABwBAAAAAAAAVAAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBek2morqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RqGpmY6mjoKtQpkAy4GAoCRAGDCC5jxAhYCAABQSwMELQAAAAgAAAAhAE2Kcu3//////////w0AFAByYXdfaW5kZXgubnB5AQAQALgBAAAAAAAAzAAAAAAAAACljs1OwmAQRU+xVKgIWEr5s/ARNZ8k3UgCC8PancYNC1amgRJIDJiWuCE8BS/soAl0LbOYZO7MvXP2b+PX94nBN1s9i5JprJ+VHi37OlB6vo43cbj6WMez6KC/hJ9JJHqyCL8imR+fBsOgF6id+nfZXRQd2vjc0qJJnRoeVVwqONxQpkSRawpcYZPnEossJhdkMIBz/UjPyGSKask2Jze23BbEUxRvWTIcyXIl05PsOg350pJv/lneB+65S9Ef2Bu/9Gn2E3nuSP7H/QNQSwECLQAtAAAACAAAACEAfW+mVTwiAACAJwAACgAAAAAAAAAAAAAAgAEAAAAAdGh1bWJzLm5weVBLAQItAC0AAAAIAAAAIQD9/Pc1VAAAABwBAAAIAAAAAAAAAAAAAACAAXgiAABzbG90Lm5weVBLAQItAC0AAAAIAAAAIQBNinLtzAAAALgBAAANAAAAAAAAAAAAAACAAQYjAAByYXdfaW5kZXgubnB5UEsFBgAAAAADAAMAqQAAABEkAAAAAA=="}, "1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339": {"jpeg_sha256": "337fb18f5e3c40c97233b327c1f850c884ada1246d7fab0796562d090dec7ab7", "n": 158, "arrays_b64": "UEsDBC0AAAAIAAAAIQAyI7hK//////////8KABQAdGh1bWJzLm5weQEAEAAAKAAAAAAAAMggAAAAAAAAnXr3W1pZ1/baiMaYYnpPJt1kYqoxGo29K4qoCChIE1BBkGoBEQQsSO+I2DXG9BnTqzGTyczzXu/3L337kPlhnved7/nh4/K6Fuecfa+9djnrvtdGfwO9vpGFQAf6jE6RWqjKyDudYdDezMg8nSFWqDQqvpyrUHWKiPuVfJlahO+ru/h9Inx96WZ2buZp4u9y5unh0/8/nzRAGQxKTzIA6U41hY4AkvMKqmqxRVfzKwgLh4pKqCnYkrKr6tOxRSeraYexhe113Ufx5V2miI+vdzbWlXF2ApygVeTRd2A/daUFtWn4+c8FhfV7cPPUqoamIwQ+s1P8E4E/ILl/GiC9lf9EXgCQw9E7OBmAiqaGtc34/sVeq4Z/DiClaSBMP49hl/IUlFsYltzxTJ6X8LOyng9wtMVyv68OUAHrkYd3E1CpvpQlwX6ydfWcrgsAu4VSJvcqbp5zq6I9F+N26mKsagJftDpThfvhPl1WchCiiEei8lxAVdrrheZrgO71FJZpfgY4JeVVqO7g5k33StklGHdU565kEPPBWgviCcvu9i9pOxGiC1vM9mIg12izssdvYO9d2XeNOQDXOZwKHB8kcfOym2sxLkM+eo9PTLRqxcxHqEHf6p/SJqcwVNcLPCy0r1SXnztER0mVvTdvNLQBVJXTKuRiBDvZd260dCFAWdWaom4SwD57tL6HjJj9WWLf0M4dNH1JmUuBjhUPFhQMdqFt1P7sm1QR9l/Kqtdi/D5G9k1mP8bnlQwXjeIFPWFfrTVsJ7H6q+r9lr17q0aqy+160k8F+sJ8mQalUfpvX2vtQqilWtCgVyE40ZJ9k20g4wnO6Su34YU9o3xKsexO5kjpNK/r1BGqoZ5i8CRdytOVVEgG0YGagdwbQg1CLKa6ftSSAlcpd25KLTsgqTa/XTK1H+Dn4lmt/UgKR8xu8ngyTrSM0GjaUOqVQk15jVROOtSoLc6S28iojaVtcFh3wLWGu9mmid1AphZ2eB14w90oDq64TqZ0iIStk75rJzgDdHqvZeeVQmUVVdVFPtyqrssdsKchJlVV65pKh+vN93K8jr1AphWxIlPHAO5UmJbdp1OEkm6WcfzuSYWawZDJ92dW9tXSB8U7Dwp1reXmqb2I0dRT4vcegGu8woLo1EFIZlawglOnAFXUaWLeC0kdMjFnQFd90NzZwVaITmbI2pqYg+LD6Uq51Or2HEftbF5lePYcXO4tKwk6j0GysLxjPnAGUA21Ox7OJPUoO/nDA/Xp/e3tPIPkzOmxpma1SXB8h44rHppynEKNXEGLf+4cXOwupU6HTgCZV92x4j+N8bSuhfDPSGrqlIwamrYr+EzFpOjCEVcrY8TEOpusYPO1eus5VNvFL3VHzsD57krKbPg4kIUN/EU/7r+Kzp4PZSL+eJPCYmcikYzSHxy4TNY0NtjWhGcRm061zJkuQkEvt2Nm7gwcE1KoD+InIInXJFubxnhqS9vD+FVgOGtlPhcLsWTUwaXhy0jQUuf6teMsorU2jD8yX4RcJV+6iNsf5tc0rcYwvqNBcT/6E8Y38h7EMqEhUCz3TLUBRUfR+kYuo0YGZTLQfhZVsWq8a6MXIKuPM7gaOgP7BLXUxdBJIPEpipUwTiCNDZ33pzOhcKV1wOemw63JUkV88gLcYNT4/R0/QRanamBNfQ7OyRjKh9FTkMqroazEjgPiVRrXgycBGsqM67FLkLM4Mhz0tcDP7hLNnOsinGdV24KCU5AhqJCuKM/A0a4W2QPcbzK7krYaPQKovdi+FjoOUJs9/SByCW7NuQweFw3OB5i6uOsCHO2oHfR2HoejkirusuIU7Olq6V5xHYckdgVzOXwIEKvKuxjEibCGP7sSugCXYx6Dx10Hx8I96gXHGUgXUDQ+8RFI765suS87Cqlias/a+BE87jLhmn8/ICZjdiF4EOMdM/fDZ+BQxD/o8pVCmj+gWXDgfjqpumB3OiT11NMXRbuA1NnUtTaWDqiDorw/uRMQ3bgy59wNULk8vxA+Cju9bpXTUwBkS0w26z4EJB5VE+lNBZKome7uTAESt4W7asU80dbaf39qO6AW29rSBE7wFdNzURxHitunnbLdBpJuVh2z7wXEbtbFepIAMZhMCwfnG3qz+IER5w06z7o6iYmm2fFwaRznn6ro8vQkjkvu1I1M4IRNH+u3a1IA1VIMtmaMq6GIBES+LOvpX+rE+bJSObUwjG2tfi1uwLbYtjI3gduL/EajAe8nmsHcrybjeemx86kYV9M1KSnCttBg8rViWzrgWVFiXN3g41mcz6DUuj43iuOkWwcM5qOAKnUjwzqcV0sGxxWNuH3+gIt3G9vbGrWhHNs8uX5VAkQcDxbE2BYZHszgfAo1Q/1jozif5RusVuI6p98qJ+K+0d8vuoTtFV1bB8Fbt3X8hQ4iDpV7Fed1KBh5MNuD298Y1BvHt2MiGnF6B/H1T0a3pxQ/PzbsH8Zu4bB+TLQX21MDvYZmbG+qrXNEPHcHlhdo2J7TD48T83pk1BUg+j9gdrmL8f10k7VvN8F3QypWMuFnsDcxjstmo/settmWpdkKbKt+n3yehPmt+tvyIh4+qfjLrwsE/+d++jhH8P++J19fkwi+0379RvA4yt36eJ7g713vZXh+kPl732P8OqQsv2v+jLfFgbfLoq1tmO8i4bFveJlRvSP8fwjdQNbMrFwg8BI9l+gfrtrKynB4L4YaP3FwOI2q8KcivOwd2rFfME+fH/KyLdnYj2fC20bMxy1rdzkL26S+Arkk4edumRBPr+U6ZUOD01kp0/uxFd8NP3ZZGwAVBkKaWgpAmme9rxXPN6qZHad2Y9wORfWonMBLs+uwrVQ3Nz524NfKG4z65YDUK298DTyA8uVX0Uo8vwcWn/ar8Xoj7q/rqgFiXeStPj0xH7J8jh6ByDvS5ppDqO/xTLhHA0mjq18XK5mAmtc+veLV4GVbfjAqlOLmqo9vRh3EflB3rk0S41DkSW0IvCtTmk5jUvLYw6hSxYb9E4HPH5R1QDLEX3wfwetzeSNiZY5itaX7/tZsx3Fc8cuezuOFOd5WNhZBKBCVcES923cOrX906ZlwzOr//a2lEpJtoadb7nq8vV6Fx1kTAHtGfn9vmiL4/7nuoQ/jM2h8rycZ+XxL8l7BngPK+1+jlhZ0ftL7/aW3BFKngmufvd1Y3r2J2drGEBwd/fZxZBLjKe8Glj14Ya/VssO2PUkTgXfhvvozZ+Wxz545GroSs3x9MKdA6Z7IzIf+cYTKP6wONQWSINP66Q17dBsgwTe32ITz101daZvyQtJE+ENYWXHlknr6i/upgHxtyfJtyT+D9vgD0Y+SSazL3j/pp3q3wy375kadYTeQpN+maAMn8P6Pl7G7ryY7Zjcjo9V3MoenN8MfA9uvrtp+Wxr/Be2fdobeMMdSEH3z/nCdZR9kebcel/QfBFL/14lS1Rm8wO/q6cLsJMf0l/C6tuzy2OpW7N3Hg1fmnd9nLB9S9y1ag9OVxr2o/dO6o1J7HLLiXx8Uq48BaWTLWCG7CED93EFh5yNX9NvMo2/sI56NrQfvt66c7Xz6h+7Z69M7YsYpJX/gIlJuxMfK+ZfgxssvsdKei0DybunLhFcxD21qKuhlyO39FrC9Eu0O/br5dPVr1inK+/+yr3+6lOodtfnE+qug/WXRUtt2A65ubM3miy8BKfRVV9h5A0Cy2V/WUg720Y8RkVuWHH76YsP3Oe9Y/+a/Zu2/3yBHA+7nYkM29N5/qK9uy4IrL77EsrmXgRT+qsrjYnzPZ00BrQoMgvgyXaxF1sBCfGCpgjT55bPR+S0XWZ/5XF3xu8CPrpjKuHfg1JvfFgrbrgIp+kVR1XETYOCRqKCxGgZbJiO0Nh0yjkQf9AcqkPPlW0n0zzyk2Vg20hfygeEPjdeJs+HQm98XatrxuEOfe+rZGG9c4uQ3VoK2RhmqqdeBsm/6nXy8HBmePGmy/XkXcV897S9+kwcUx0S0WHobdr/8Pt/SkQko+FpA5VzH+GnK3aZK6C4rnb9XpgEmb+wlVVsKjU/XxarPt6Fo/ZPuzuptuOHQu+8orwL56Wd/QyfmmWCglSbCwn6gK7uAUQRtjaXxu+VKKBPLP1QriyDnlzWt4sN1uLX+wZO9fB1O+IdHstV43tffu8tFZwH5NZjRMgD684sKcSJrYBYFcyvluA7o+61MmQfnX64Y+l5fglMb7303ly/C7ohJnWXAuKU39rxOrDt8TDqt5yyAprm6iJsL+aIcf0GlBM7093wt678Ne14v2tSvT8POl28dWQ9OQXLQ0JNlwbpl7rE+W4V1r7eTWduHE65SQiuS3IRjI7fNdaUMSPEJ3pSazgNpbnm899VOIAW+Dta8wjxuGWbemsT87Ysos22pANbJ+rxRXKj1tlSV9p6B3RN3rE3lDbg993mZ8Rggz/xgywvM16PfFaLX+H0f1NfecmKedLrYt82YCIZNFTkE30gZxWU6rEe9OQ5+cznGsbcabXsAGcLq2icYp3gvHn6O7QCvMMuN843FWJczgu3AcMndcexHwim858Z6wZ23NKi7Asju+K6zYyJSsWd7LTg/yqS6pQnCdquKvdjKNyQ5JuLaUXyPeC5QFRa5cJyO4vm4Dc+vYenLPEFY8qGXbg1+LpVPLvdgy375RTaCrfC1NZvAiWNF+WZsO4YLi5y4vUFmGXiNC9met7H4A3zNf/nrupLAvfklQvBjw9YHTzu2jPfK7HFsuU+L7owSzz1383A+ht4FXcsLTNCtX6zyNYxv/GwxEzxT+8228DOhE353+O9iS/naUzqe8FOWRfTPnL17x47bV7xS1T7D+TTr6xA3gu9nfh1WMgle/6YPJ3j/++TMToL33z2YEWBb/oFXKyP0zfN7ecR8XH2jqX+G5/PUN3OHi+D5zUkzoV92f5l7TuB2fvwYJvj71NvfX+N6F+58t0gJ/5Xv7tWoCR7eQ7TC9Si3PGF3TCoI2gd07UTiGhCC//RBRdcT9liXONHwUilWf/iTIij74YdM+o/4XdzmRLubLV3bErKgTEjIFiykuQkg6cL2vzr6Z/w5VV+iv9ISGTFdKIfaeZq4zmXqCNkDuw3nfsCT/tlBvmyQ6A/XU70nEpbSeYWwtXRdIo4bgZoEMOmn5H+KA1XxCTbA+nppJIOwoybVTcK28vQnCUvp6U3Et4d95J/iSKqekBYSwxR8WcgmbOeqn7BJgn49MbEkWqVtFwG7ar/7I459P+bzr2XZSfVb2/HXNKHfRWzXbbTZKOFvu2zCQsis5NbWsTNE80aVJAHcR/mx3qQfcRxkmN0D+Ns+4aBNgO2u2l4VBdu9QoeROCbZzeToiXjIDMpUGuHnTte1BDD5UGJYP3WNx7xYdh8X8k0juIMDtXWdLNzgWPuwhojrSKvQgGUQ7FXS3EQcSZXcbgKITnUfIvxclnPjUaxzz/PbOh14ho/XN3JE2M8ZukjajRucpQq7CD9na1u0ZYQfqmCcWGdScW8NMZ4sGU+NyzC4KuR2OPGDM00tXA3eSOfaWT0DOK7L7E4eMfDrVS1dxLFLZmfHZC62+1nUMWJj3ZNrRUGsO7O6hEI71rU/tzBEGlwOXOO0CYZwesrVyqXqZEAlFLpgCKfBCiV3QILxWcpWdxZObxXa4d4QDVCBoltqy8P+mrg92puAytvUEsNlLKs7B+SGg7hsamgXTODyTaATK7w4HTONXcY+Ei4/dOY+nwiXL0qZ3EIFVNfE68b1A2pptvB0OK56RrcEywCSgMoWjF0D8uCARBTNhG19erUqeBjrXY25z9OPEFuqkJmFgJoahGJJOy7Xakfbu3FcTY1snqoSUqQUdqepFvZaTV1cDw2ODhl0El8VkLrkRplnjIR4HbKuYS1CrZUsNr8H+ysxt/OFQGppaOaI22CXpo7F1/Hg5Litq3NUChl6k1bplQFZLOUxAr4U1Mmi0ybGSaixnMnkmxDi3DMIO3UoiVFXzxHL4KCujsHRqCAjaJYInGZ0yzymUwVGUHI3hz7u9+0jSSjdrKAvmdRQ3Mbi2BGZeU8pE5tRKqOukicaRCdkVc0M3QS6ZjcLGN4xlDtq7+l2+0jJ3cKpoM/7E1nMdK35g7vJTfckw+0+ckprbl+f2E7azamk8IVWdF5YSmnR+kjZw9Z6tsdBKh71MWROf3Jy72DE5/ZfIqun4xOm8OHk9soxizSclsorHNT3eJP2s8srxdIw6SqL1tCkn0LlCgd71OsmlVt8FWZPcBu5zxJyDvluJykH3ZYJ/ylyN3VsSOVPT+2uUamlweRDvLKKbmMo6VaTQMEzTiGGeMIT8DhJzCF/34TPl0yW+jxT486qJIXNqV9yXyep6Y7hYf/+FEWTeki5lHaAw2cYvS7STSrb1DdsRwy13BGZdCDhwIjD6/aTUY9jxRf2MZDS4jB5g0UkWa9Z7Vy4ROrtMuv7Izt26QYsApmPlEERjSgdTlQhHXA6vHZSs8LgC4X9O5DI4fPHglzEt712hWapJMHE6sy05zpJ6pibG/Olp4wuLsSkdvKhAU8sOukmXTG5p71ed+odtdnnDwfPQrvd4/TYOahxcsnrcVNQk+Op2x+4jti+sFMX2oU0IXdkyI62T0XmQmMO0jFPOBoI+PefN036g6HgNVQ/4bR7bO1QMjHu87paoHxy2ev03QDqVCAw5EtDPF8wOGZCJKs/4hsfI+0MBIKegPfQwdEpjyfoPQ/FeBqcVi7ccjkiblcT3LR7XWOBq/h+wG9zpgDd4w6MqhFS+T0+9ygiOYJ+bzCwM9U66fD6/QfhzqTb5x1rg/OT9qjPVw8nTUaHJXIBMl2RgM1GggqHK2jGvCt04HXTAhrCKL+PRBqY9ATC3hS4ZJr0eCeokD5gCpu8hbBNIbcOhw/DIeeMS4WZ9Ibd6zFjZqW4fG4PF0Dk9AY8owi6xzw+jwXBHqvHO+IuBJJ4IjQSvQSovXvQML0dUgxOi5KB86cj7DFh2XdzPOAK4IRcO+n3ObGuabY6PY5enE+1ruDY9M84gWic43O7MW9JFMNY15C4Qyo1JjJyfyRKx/lvz0Q0GMd58qIvND2FE3Ku3Tc9RU0kcvIPPpAWJ2xytO8Hv5za/098978+6G5Gwh6YCyZani4TJngmqf7Ojwak/+wgdbonYa/Jw4lArmntCYI6IGz5oR8OpvzHOC5+nE/0V1pvJo4ZoPnPZwnCvz1sSfhLrfmhY9D/Iw624/kO4nFDY0IoIPriNMF3iL24lvB3XkFJAEkHkv4pDjRQ4swkrEhkI2Qe4puniOMKNPktcUyCmrqsifjSOg78UxzInCUifk5AYf0QnbCGHiMh30irn94WJa77HiTm48riDz1F2kH6exzbQyMdhLzdtrFiII41yOtyLZe4Xn3yjFjepPjq28tE62btYAK4q3D73+O4sPhJ48ffTnx4u27Eds9rp48QgEeX789z8PPd7/7rBcHfpN6mRUKfoVt/8T85PeGuKPZpLIYnOvvhfHyCOFb49f4bws+tpeiCHNvLL18tEbJ1r6V//izhh9ejJ4A4IxM0DnS31ebFyqgxFLf6MP+XP/rwfQY3aLjvjk9hfPXCy9UhbO9M/fGQ+H1qv9X9jlgXksLOJOKQ6vu1g7cBuj2fFt2H8HKu//HbS7xjxA+c4VW8YvW/bizZcEPK+63HxLFhpf3he2JeD412JcZj7n0zImnBZYBz64UF71jxr39svvsJlwMPw4vv9wG0vXoSm8bjU/y29XoZx6UzfVg0Yz/tDPNGDuZ/u3QrQMfvocm3+XkAL6Ds1e9bG/h971+ff/w2G5Dycdy7dAi329p89yEdb4vR7/43qXjZax/FJrC/yZ4vS/VGfD+09U2D6w715p+/PxcAsj2deLhBBzQR9gfDeYBinz99enkHl//mr64/bkHyWN2maxPzv6l3c77BgZAjsvWlbwrQ2Jfvvz/rB+R9qZ171IvLbd9yxMkGcvztp9+ecSDdZfji/iiCk0rKpvEzG5d3ivdOagChhfkPr7pn8Tb87bfPD7Gf8GtdZMmK+w0/jxi1sCvybvPb/SHI6FdvTa9aIJfd8Nl43wEkc+9YXW0sBT2bsw6KHyI09+XNL/djCIXfDEaWQ0Caj74MDzrhmOf9b5txP9yVyr7fj9uB0lj9ccgagqQIj6mtix9Az0Y0zf2bZBT96JpY3EDItTE0/XANUuKhjdDwEmQEfv32fm0aqgTCP6KxFUSrrvulS7OJyGviZ3PV8+fQS8b7Kfe3dBR74fa7vyQlOV/pVh6+RrtjwXm3/jHcDq9/ebscR7Q+9WL/5EfUUEMbqRF/I5OfPXm2QFm5TPrkff3r0p/HSauPMF982548825s9ckmOhryuFymh6jUGp9+vvQK8SYclHb1FmpktWWVtX9LTnrybukx+2E2evj5/b/e/Osiafp+QGX6np4ce//LxrOtpJ9CnqH4yDpqMARVc/4vSBe3t3BEn5FA2VVBa/pMJkVsQ1HfMwrybUz+9+YfucgzbRZavx8ixx7FJpefkU8F3Lo1y2tEHTMJHCPvkNZro3O475FcpapnNT1PQq7ervtvnnQge3zkz61vNWjSquW6X5wnOQ1mtvHltv1+aY+h/x3KdyiaBMYXSGjQa0dYb5C4p5vFbV5JA5vIEXy+KkPjHsPWb++qkEmj0i49voGG6e3N0j92kt2sukbNB3TKIhIL1W9QQa9YPtL6aVsNvbGlvWn2Ehg6eoXusQ7QjztsK/fbQNMvnlj4NRfUXAaT93EXcraz2cOvUFqsVSru+ojOjrSIZMwPB7OYda3ttOlcUEi4DTaDCKTDEZo/1g0ivWbY+SAXekR8Qd+bNHAxRHz7cyDFmnr7RO/QrmmaSMl8d+SUqobeTp3JhE4lu2ZwqA84w37aaEQEtJHgSP+DLGjVdPFdb1PA0irm2hYAItTert6ngBYonT30z7vSbOWNjNqlY9Cmbq/VmHugZsJBG5lhQ5brl/GejZ8hVy3hPHhBwvEJ2VYLfk8Z/E71LMBMPUdIe09C9lIaq/rxNigf49YZHWw4/srdrF2vgbSt31yTm8fgmKtT8jiO8854Z/tcG84ryg6+2QBg5XG4bcsAE5UtjOpVzO+PpRTuSi2QHk3T7F9uALr/x9zqpzRIXjAZNzS47pnhMjYwMRWGOGwfDaDD3M4hzkcUHGYr3Yvz+4qm1vbmOiD3U/bb33HB4sKJfAkXFsvrv7zAiY58n8Uj6rL9rxltaxdwObvR1ubB9X/NGIMxQJxTQNIPPrnQKzuNzRGxPrSA5US9Ry6ec+1F4nHleDwkgvpFZzwY8cJBsyseifQS/GzUMFMJpvnBzxcVYoI/D/KG/csYXxoQC2ad+4BhlBuXML5k2T4Xm/HDbo19KRaRYT5skIlaiQyctiOBvyQTEvi9bJ1jDhfUBSaRyOfYA43dXcPToU7IC4/NR+NhSBMZY4FpCdYp+UwOjWD4XQkegavKHoJv0zu0o3OYaPJGBZ0e+26oEUtMkUgH5E5PxGOzftjG0c76opjZkq5WN1cQQ99/MIG/rh8ghEw6r2cohAv3m0PcTuvYTqgQCQzuIAOyvGOxaMwDKS2qkCPaigno3L3afOKHkhPHE/gcoYH4uWdnu1DtPIZljKaNb7CmQYVKag5EaJDlmZqfnrEDuUEZdMQaCH2VXXQT40mXziX4tIAtIXTEjjaBdBDrvUvilg41xhd16/SuMAVuTprCgbAF1898ly1ShfHHrt8iDh6Srl9J4Mt6JMR5QSpNopBhHrzIqG8TjqXBvQ7D1FSwBq7bh7yRaT2QSvgmW4xQKEczM4jIST9fS2yAPF4PoXu2Vbf3ibBSOU9rwANIg/w6xbgxWAHXbZPT0Vg/kAo7+kxh4mThyMXjif8HOX8lgc9i9BL/lpFS1SLtxTN6obVFJBlMg4KSpmFuoBQyra7FcAwT0j2e0OgmTkiOZBwgdg46eeHH+Q9blkPgKzv4vTj+C5QGlli7He7lVA/yfUWQaYksT4X5gPK6OlWjhJI4ejGdAKIjJxP4a6w+wus2qrSbGP/5+gqmWLUdCnLrHO2+Qrg6bF82Rlpx/yqNrA9PMDqasSsB/Gv9M5sVxPptoysVMnznYlt1S6cqFe5VNU4wPfmQOWSNGfwNGG8Y6uLsJvDn0xLAHT/27wWmmjjfSKVK5XJi/liNbd2abZBfSu9vCeXBdVN4fjRSCqhkzCau30bgL6YmgEk/dOXZDjkhVLezFEMqrDvOt/uCksEUuFesNwtCuXBrwjNv9OQAqvFYBPkE5PCFbT+E3A8heKaVe4h4mxq5ch1elzPyR7MCHcazGV5L5A5kucMRq/saluU+p5hQcOjQhZS/448z2ogXKa2cJe3dBXAu1MUy6VJQKZ+i6gllwR13NGryXISkBr9bcZGAHMz4Mf6/hOjBpkZiIrZX43XH+Iu8Bmr7UAoqr8hni4PZkD0eDdsmLwC5xhMwEsuH9l/88eL+9dlTW0aMZ3sVVSLAgjfzVjmdN4LxhXeorOhtyPVGop7Ji5Bc7Q17iHNVtO/c4b/jd5TdIWZle0WZkofxN3Iq2aLRbagmp6iBFcyCHEfIPz12GVIorlC0mMDvPXv87/jUu5eIfZxGqdCwMT7nbjWjy5qKaFXU2l7vLVSon46vWDLx+jqD04nKZM/Zo3/HJ185QdxNLaszEPjCsmqWYmIHicHhdZhcN1H5yHR8efI2bKe7Q8Eeoqc9Z/4tftKZvQQ+5V7DAAvjS3hVLM3UrqQWSneP234L1fZHZxYcBbCzzeXz6sgJ/MG/49GBxHZKKaTraTsAlQ3WsfWOPSnMRsVAfCo7iWaYnZt318Eujjvo0eKm6H/iUxP7MDmvWVuDF7bMXc/VO/amdnb0DaxO5SYxTLGZ2WArpPM9YY+UwO/9H/gf54nJObTeEvy4KkzlWtz7d2jl6v4H4/eSO2xzC3E/E9IFvpBXQuD3/zv+r31Ivt3YXbCN+LmWzva6DqWbrfr+FXvJNuFEfHY60gbpnd6Qp5vAHzh7EP1vfFIWVZifjMvgeQbbN3Ek3WozD866KlLF4zNz4TAH9kiCIXcCf/jC4X/Ak66Xc26RMf/FuNzYxPG9dodpMO6mbO+2Tc9GwgLYK43E/L14gkgnfj767/gfy3i1kJlJAhIjJpQs2I8e9rqHB2dD9bs0jvjczBwPDg3MLcbUuARLOp+FE9f/BVBLAwQtAAAACAAAACEA3YDwjf//////////CAAUAHNsb3QubnB5AQAQAB4BAAAAAAAAVAAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBek2morqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RqGphY6mjoKtQpkAy4GAoCRAGAiAJgJABa8AABQSwMELQAAAAgAAAAhAOEGP27//////////w0AFAByYXdfaW5kZXgubnB5AQAQALwBAAAAAAAAywAAAAAAAACtzU1LAlEYxfH/ZGaOmpnp5FterLgKs1EIIly7M9q0aBVDTihIyky4kT5FX7iTDPMBzLu697nP+Z2fp5fp86vDhq2dhfF7ZB+NHS9G1jf2YxV9RcHn2yqahX/zSbCMQ83jebAO9e4P7x/8gW++zd7HBYcMx2Q54ZQ8LkVKnFGmwgVVatTxaNCkRZtruhh63HDLHZY+h8gfpUIuMQqpcp44l4l0lVqdnfbffEe3liYN/XjaqGmzqkRFybKEkqSCxLzknBqyasqo0VHzL1BLAQItAC0AAAAIAAAAIQAyI7hKyCAAAAAoAAAKAAAAAAAAAAAAAACAAQAAAAB0aHVtYnMubnB5UEsBAi0ALQAAAAgAAAAhAN2A8I1UAAAAHgEAAAgAAAAAAAAAAAAAAIABBCEAAHNsb3QubnB5UEsBAi0ALQAAAAgAAAAhAOEGP27LAAAAvAEAAA0AAAAAAAAAAAAAAIABkiEAAHJhd19pbmRleC5ucHlQSwUGAAAAAAMAAwCpAAAAnCIAAAAA"}, "1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269": {"jpeg_sha256": "b61edc64958042a11c9e569384c205610de215e9ea457867916acc87bb11164d", "n": 86, "arrays_b64": "UEsDBC0AAAAIAAAAIQD9koYX//////////8KABQAdGh1bWJzLm5weQEAEAAAFgAAAAAAAG8PAAAAAAAA7VdpbxtJet50NSnrsE7LkkiJl0g2zya72XezySbZvG+JokSJuihR1mlZly1Zkg/JHt+2xh6N5/LYM5h7490B5kiyO9gJJlkkO0CyWGwOIAiyQRAgfyNN7YdskHzaz1Mg8DQL9T5vVb/9PFX1JDOUzpX+5EcXf7SLzFY3ZtYRTofsbWGIQ4fMra5vrk+fn1xdn63W+iPTyxtVuX9jfvpCVf5vYSiHjpF/VoduX/dHtQbgmD+4MH2nC6BXPyrlL7eBzpXX4/69HgA82+lQ9RQAuuXsSBwAoBxZmPbKCKsWFsOw/KCYPLegBclyaPlcHgWZJ8mlkWEU+O89KadmUaAoHQ1mzvUBENwfLWRlHtVqLlhWyvHRcqrSK8ebJkbXhuDqnDidL/hOXbztG83m/MqxD4KpQF4AXden0pkBO6ibnbOP5s4C2H2bFNa0AHSMj4TuhGUeaSg5uNJ2JIbyXi6s/irPpzha7HnnIZXxMxFAXGcpKcED7ZLfTBdF0FCKGEOFEQBjU3gwNg/g+uEhwbeg20WZuJ+RXJ9RHollUuhvCu4oyxUbZvZcLpYeAGLJbjSxWWAsWxELkwJ1+aAVYQfl9YzgJDVMLmNkSqBSwg0CCzDMMPeZi/VSbEEX9RpNFB9oShFmrVogQT6A9BtxHnROO416Up4/EkJxIk5OuLAMg8XFNZoP0+5kfM6H4RQppZ2S3oUJLnKAMOq0tP30qmBDMS8NkGGLFaFDANCCm6QTYtrHFP2YLzzjc3kZdyCaZDUk4cG3TLRO5keqAmpAA6TNcsgRdJjCgW/AarN55IJGKRfB5tlCwLaWY8lwmNGJgpsIih4DxbrsqwZCJ6JOZI/G9D7SZaXX2aCcwAtHWLvFQRbkMqIYRoW5HG3J5NxGKWw3cmmrNhBw6yyCuWex19MnILTqGufSOoWJvv71QBzPpd1w0GNz+n0JoCBQN0Yn0FHM6c3YVMyIo98zbezGIlivrerqGFezKo8YaN/wIr3dpmRn64KH0PEeHWBxsyWEJgDscqBBR1w9S1tcAaTdVLVqrB5tq2UM6banzadDDnsXETOeTk+Y1F39llblLGZUETYVQILmfsYZBIC02cK2dNMNqtth1zW3XEfPWPCextaKs8PY39ugipMdFktfQ/c8qurS6hrgSMCgws0d4GwF6Xc5cABEG4o7JLDFEZQNhcGOn6bMRhheCERRCwHDA3GTwYQo4XMxzRldrxJYcmgPamwEyg213u7UAOAkbHYLDg0mSNyIABC/wrp0NgACOwG7lgbAvR6xGlhZhg9H+vrl4S07KY3eDAM4Tpgt1hYA2oN2xKYBRJVzazplmR5lrCq5u/MW3W9qA6D+GCOdssxOXZsSbRZZbqmLjFUlY/cq2Weu6XjEY6ZPgc4rXqO6puZnqb4uGUF1oUdXw8wlLVpD4j6N1uR+emOErqt1+F9lOmqoOaatADI+CF3b/LUDMh+m/2zxd0aodXvr3wb+1gRB3hf/Ov73bRDU++wfPvkegiB48hf/+DMZoZ6Pv/t1Xa1j/vl3IShXEd76zcEgVJ4Sv/7dd3mIW8n903/+RxYClcpv//2fBQgKXf3yZ3/XCUFdl7955zuZDyp+9vL7hIym4/dfvw42At7jrftrdTsp/uOvf76iWBrzfvtf/zILdZ0v3/7+t0Wo7vz58et/SUBQcEz68FMSglo21l6/sy/HJw/X6Debrnmlm0FpvmdfEj8ZDV/o2soKbzxObkGRqZlHV+fLkGHu3q9u7dqhU5PVG+ceZiDIsXKnkL4FQXVbszHxcu8mQc9KbNl9XmCuZ8MT6KrIX72WHDtVzt19dE4sQcLEN1/uzBch3cr2xWh6G1KI228NBy5B0Nn1RR9WtixT1L7IZr1jHPtOOhLxTUenj24GxvoGJz6/l0vsNqXvffXh7sZtyH8wNx/xLUKt0VfeXAstQpBz48YYn6THRc+RyEjRobn81xNcXipdPHz9yLvqHL7y81uxpSPt8OFPny0W361P3hrNl4JVqLf09PmBMAZBzKV7eTLMLK37divCUrz6QfXPr/Br+fGnP333q/gDf+nidweZ4zfw8s7LrVjsUXfow3K6IsxAyLmffPlQmoQg4ehFLhdjbtwYGTh4ek2c/PC9u+9N7WcSt/7m+V+8d887uPjL47s/PnJXNj++tDr5sNf7p5X4SGoNWGc+/eIgNQtB7mdvCnEffunp45npL3Z9uRffXp/76lJAuvqrF8cfX+VSi7948vnVm+Tc0gfbd0IHXb6Plodz0QXIuvzBF7PRDASQD962+zhbdf/51s2XFXvyyV89HHlt3CYdfPtY2p1xBi58fmMzv2Ivbj57eyez3OT9aHMsLcUhZPH5p7PRFATMz484utR95eqjlfM3o30L739y5723RVXkjR8frD2UEP+dj7Yr1wcsQ3tvP5kpFxSRx1vlhJuAdBdef3eOLsjzP34w6L7UsJ9PBQipt2H9/ujq9LqqKfbG3Paz3d523+6r+9Vte3v64ovjuYAbSu7tVWOoDWofuns8J+QhiFzfKDgq0I5IczgPQ5dD9AW3rPv52OJhQoRBOnPuXHGiES76Vl9NJ9sg2+WRpUCyB1JGK+8sezn58y9ld51paFbw0Da3/DWvcRKiluWyP7dq7JA/0+npMeQsBFkrKGlHIahpk8JwEch5s9MTsTMQdLoUusxgsiyIxOV+CNJcE4uj7ZCs33TW2wJB9cuUZJKxbrHIGltluXCbadQhY8vCHJmXeaAUNTNVD/S3X77yGiHb2cufLD6Vt2fj17fMh/LxAYhWXbBZRmfGsVPD1tdeDtR8BFjFRvA/LfN2UHhwQQEyf80YrxQA4H8Zymb8ANQNo8Yh2YAUFO4pOWS7sqxQCzIPXJ+Yd5wcP2oeBpTVS4QrcFmr3HjKegOPlGDoG3+enwXgzLVQvDgv8wwyVLxUm0+EK9pl7JpOTZ/42CmdTKfb5omgMJrXvJb10fgLY/36Yz/rPaoH+ONUjD6Cgb4SpckDeXQhFAlna364Yr9Q81MYG4gqQGjW6w9wqXn6oSRYHRcLXTfzjM55X6vI7qH9jistsHDO0u+8UQe6ChxPrMrHqIEUXZB54M4SkVaBsakwaTeJC4V1gtMaImX3Me7SWUb4joMootUPG2Q/M+t1ax3AOY0J2KwCtFQItz0n5/eNoU4B7Mx6cTviTq9WecysZxKhfQw36BneLNu6tieEtm5KiEG92A/SZU/Adr4N6CftDiSnBPWTAu4ZBA/XgpzbQbtvbgsei5F1jl8X0P5+t4W7HjJoOjmjZoc361WSUbHBuCLq4W6YSVrtxngLrJ72EHiy/sErfNppI9V718SkRk92b1Szrp5es3pw3K9Vn0HVeNXRb+529TZuk3ZSI2jg4agDQ2Iq4JugOGqg7/75mZFJDO+4PjEYitFIx3a+GBb5/vbFQd6UQ82dwbJBbezSdZ1dQJxWFdIHj/N2h5m2wsk8R2M09Wi0mDuQrB0PCgORWVbfcjMZYklc07IV9wezKVNnomCzx87az6hHbZRRZTirGKcQg8lqhYt+uT5m5m4xEZ32GpoPYpJ32qNu2PMHKSmubloOBjAM7W8OD5HuSJeppS/v8MTtzta6Ua/dgyK9YMbnQb1245PxsDcjGBoOcyG2wKoVNyN+JhPU1y2FIyQaQZRkQtSjiKGuM+ITgry+UTlNGl2MvgUej7KsZNY+mwqzAbsBfloMcrxbA+5meW/Sg4Dx0YggZwDm5YiFtxHg9GLQOegwK+CSZFsO9tfD01EmzDlU7y+LBGEyg1fXBY9Vi4Br+wJHyeNj63lv2I6DjvkQNepAAFzk8JhVD+BcnPG5ZZ65sJfDXC1Dy2GGtvSCyIpI471nALnlZ92mPtCzExZIcz+AU6MxCWkHQCr6Baus69AiRzlkHU0EONaCKxruJijKUgeU6xkGVcNAcTBEIyr5mDI+4OvvkY/71kmfpJHR8DiUwE4DgO0IPCefZ8olL0dg4Af/+ME/fvCPH/zjj/YPhTlCtsjjvV/cGr3SC5xvrSYCoRsNB7ejIkaeo+5vllJewaN7MjOYKvvsjUtX8pEQowEwtVjB2mUfgE0S1Swj/+VGckENNE/24sHI5VMrt0J+nCu7D6sjSZbx9D0cyybGBHt9dT8fC3l1AKYro462mg10C2ybHN/36oQ01A2aFm+FJemisnBT4DBi0HI4Uw7wPN53NJCJ53hnfelqMiAFzHL+SspRiwONrO+sjKdGY7aBTqDM3g9J4U2lcDsgX5BHjDdnxpkAjWmOBzKRmB9vGDhMSQHRDmD3TNrRUZu/sjCrkb1MEfbopW6gDB+GI9Keknkl4HE7RkxXpgq+FMvrbuUL/rDX3SheyYdE3gFgc1myn+RX5G9bajwetzmoArDvYiyc2VMwBwncaR7Xb1V4JonT2t2hHBPk6SbXpYGwhMn16836T/IDWLxhq6GdJ8flunAXRFdgV2G7kSXclsX+SsXjIkxU78x42OV1YA1950cSQZtRvr7GRGv7STy9ba6hOS0tyfdVfIl3Mxtw36UU6bKt9+VmCYoyebSpmQjut2FNbZViNOYwyK8tKqK/j0eGu2qoHU4tyxdYU5kX6Spo3ciSbvtCd3iepHwIpyEqcTpicTXVDxb9YYdV9nVJcJ3MH6hJRQ17Jgtp+f7cXRBT3hKoWy7gDtdcp2uedZAIpTGO+8msx3lakSmxMbt8f1YE/MSZk/ytRlDD1koldgqA5ngkHkoAeLLkmXBNtvausT7BSGlVE9FAmne3gdykL+qU16+IiYLqJF7ZcrKRNc09Cst1rE9HxVgcgOSQN4KtNDcvO1irhjW3TfhIP2VvA/GZoOjul/eBREg0Kk/2MMVJ/rrgjLtWzsQAlYsAQI+4CE/mdN2Cx0BafPb6csDN4q5uEJ4K+D1yweuS8RhZf5IZ/v1OqsU7a0BVvEWPXM4xjnNLjaASY3iScSomAyjNmzQgNMPxHlwJGov5iVjzH8afOiPTQcCQjRTl9enGOMYTbgCjMZYnORxUAk6Bs2pBfIoVSKoeNI+WFlNtfxgPlCcPnYFwpBuAsyNelgzXg4EI7yUoEpoIE5yA6kF+kOVdZBM4MzM+mev8X/G/x2bSi8q87aNhjow2gFSCZWmCh8oRhvS6rWA4xrAo0Qq058eLg2rwf1sDyZrlijTnMl5Knn9wmKRZFweGgzTOumxgQqApB94OHGsDiUH9/xNfh7EW+b00JMt+MtQI+JKHoG0sKAkEijksYIoiMZvzDOQ5nxDTRvDfUEsDBC0AAAAIAAAAIQCakun9//////////8IABQAc2xvdC5ucHkBABAA1gAAAAAAAABSAAAAAAAAAJvsF+obEMnIUMZQrZ6SWpxcpG6loF6Taaiuo6Cell9UUpSYF59flJIKEndLzClOBYoXZyQWpAL5GhZmOpo6CrUK5AMuBnTAiAaY0AEzOmDBBABQSwMELQAAAAgAAAAhAI6uE5D//////////w0AFAByYXdfaW5kZXgubnB5AQAQACwBAAAAAAAAigAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBuk2mkrqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RoWZjqaOgq1CuQDLgYGRgYmBmYGFgZWBjYGdgYOBk4GLgZuBh4GXgY+Bn4GAQZCKig3QYBBiEEQSPIDebxAUW6gLCdQFTtQNStQFzNQNyPQFABQSwECLQAtAAAACAAAACEA/ZKGF28PAAAAFgAACgAAAAAAAAAAAAAAgAEAAAAAdGh1bWJzLm5weVBLAQItAC0AAAAIAAAAIQCakun9UgAAANYAAAAIAAAAAAAAAAAAAACAAasPAABzbG90Lm5weVBLAQItAC0AAAAIAAAAIQCOrhOQigAAACwBAAANAAAAAAAAAAAAAACAATcQAAByYXdfaW5kZXgubnB5UEsFBgAAAAADAAMAqQAAAAARAAAAAA=="}}')
_v30_ref_local = _V30_WORK / 'v30_reference_local.json'   # the same three studies in the v28 cache (100 mm)
_v30_ref_local.write_text('{"1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585": {"jpeg_sha256": "280a8ff9f1199ee903148bebd5e773656414299d68c370a5252716e04dc1aca4", "n": 156, "arrays_b64": "UEsDBC0AAAAIAAAAIQDAiCL2//////////8KABQAdGh1bWJzLm5weQEAEACAJwAAAAAAAKIjAAAAAAAAnXhncBtpmh66QYoiFSnmABIgcm7ERkd0N4AGGmjknANBEMwUqUgqUpniSOIozGgkjUYzSiONNHk2zGy4293butu9qtvbvbN99tWdz2X7n8s/bFf5lw3Nlq9sl3/5KxTe/r7u5/3efvvr73nevhlKBiNZQHBEsK6sT61OrihRsfLEYZNSJ1Y2llYOrVQXy0sr9ak342x1fnWqNb7arC5Ptfoqk82uEztaP7VOfFL8/9O6hKrwzPq95DjQ7QvN30qOAO2RfH0j3QeAeKk0U54AgLHJem6KBlonltJ1GwAAo9VETA6AgFAN6SXdWHT+5FZDs8Psaixvp9TgePgYf8ZvAHujPDnJMsDeat5p8cwAQsdidNxTAICuTFaJugEA7CdVarvKlatPH20aJ9wrx5qnGnA3fX4ZKkx7d2LBDIpPl4TkZJJ3VNYA8cEkowstACBUxCVIuhWPzWGyezi6WMmvLCM89+p48dAxzhm7TMDEsbyiejma9WYbQ9PxuUSysdgerVZj0aklYG+IRSNctTU9ZvZRrAErbpfr015kvkzy8/NR/5KTsGPFBMsGWNrprMCVXMQbz08O1NjTnLc5Ber8oUgtVAXbDXbCS7lWDq4uZ8Nl55l5CKKWg+ZyjaCczmw6xvl5lnHOx9KZKBdt5HTF/FTC06h20GQmFwungT0OwkUy5N2EShZAUelJ0mRCl2lrgUBwBK83Srzb66JTZytTSb83OFVl0quxtL/UPBAg09l4IAcOwayXciJnabUesWoVi3Y9hB5sZQi1I7CjNF3xORkSL5ydqfMMzddXEtXjbCBQnhPnw8lEjCsI1XYvSxNUGYNptw1SThIwhiU8loQJwRBHfTrtQZ0YXljx1dwE5p9c9RRmGMZdP2KKx1PJeKDaZXNzPtSIp0nUz+GYKeNDMFfMY4mbHRhKVKtpHiFwPH+QLftILFg+RFSabhdbOmT2Z9LpSLC03xIJclY1nqSQgIvEzMkQhrgyATRmg+1ooJ5N8FYEI0rL3qzfzURyB9HCLEf7M0uQL5FOB32ZXj0X5CCzNYY6MhRHOOM8hrOxMBxuZQ9LliKpGOwkmcxMsJyOhSPJRbI6F2SS6RkTn0hlI750n8YX9Nvsdp7A6t7j6aCfR1jv7KTNz2EUkZrN5bMIhVdj9Uh9ppyIpeed9YMxdzzZNIXzmRTHRgeUHp6zW+wsBaeim+W8NwazsSMNguVRxpRZ9hVzCEPNTpeDtemDJX+iSS0spcKYr2oKZnIZl90/IGX4ICaC3KyVz15drlA8TvNbmQAdwlnP4glfKdO6nca5VKQyc/tg7uwkPb06nw2tlaFIplhIz3L9Mmcg7PEaPW6D2+WZq1JelMD4esoTJTyu0rlcteAMZjJXZ5O1xfMFz5WSb2qtmUebRTicnyxHOf+g0s2HSYMugGotCF7MMYTJCvniUa8Po52laiITw93ZVLGcLC2tJPFQii+vVhIavx/xJSsZP4b0KZ0uL6rTeHC9jsUms17UZnHny+kI48SZcq2QjRDmeK5SSlcWKylPJpstHszl7YEIycWLxTDj7TUQbAC3mUlWq2awyVzQARnwqUougWKwNVctp720OXy4XCsWCrmoP13MZqdjeTQWc/LxfCFG8wcsTo8LwawErVYz1HQxT5oM9ORUqY6hDktxqpRmGTTyeHqqnI9zsVyhnM1P+2LBeByNRnKlDBnYa0M8BILCsaBMiZDNSRLV6Dy5RiOFWqxGplTiKRShz5Rm8ikXVl7KTrZyQVORXMocDqWredTbZXO4WCdmT6UtGgiuVxnEoOUSU1MoajJoDakMh9tsZntyphRvLfD5VCOfKvBBf7SgDPojpQLm3WFzON0oYSoUUBVkrNayTp2e8ZSnvZjdrNXyWZYkMZuen8uEnQQeD00H4ykuSnvyolAgms4iPqERoVw4aUiWPbRdXZqM+DRapzk2F3HYTEq5O8+hFAzp8VrMx5B2j2uBisVYH0ZUuzl/IB6zcqDKirmcmA5vlrO4OtwcSMEKjwGe3c0hkFZFFnqjXsykRQITGY6yE85Uz5TfRaNITeiNePwBaxAYcFgRGDPpp+pThFo33REJGWyQvgq6WmNyNiQMeVGTFnd3FvxOGHeH2otx2okhNcBY9vlZLALsIlE7iRtHG7MzJtnomNBdwnVGzQhodJn1KlIF2HnMpuFQ0B3zIA5/GMAnOafNlgV66tEQhcUAod0Fkw7DrsJyQyNTa4T6SV6rgGzgUMRqar0V4Fi6dXY2AigrXoJKpoGxpaLDzieAjlzJo/ckAFDB6t2koc230rSjbrNwuFY1Gj0MuLOCWVmfD9hZZTCqyQO76+mQ158EOhoN0hlOAiBZg7GEEwAGPD6CMoLW2VmLs46CXYm4DZ70A2DQrPdUwwCYtEo9mTQAxmMYE8wDoGsmEKNKLT6dr4YJHgB2Uhyq1baJi0UL3zAJ2znKjDYYsA2JI/YpHyA0ZmhbPQKA8oWkt1gQAINLs+nylADYGa1ni0VBi36dbjcB7o+z2ljZCoJwyEzXOAAYX/Nj5SAA7LtSdeWjrXmax1PFHADuCByZrDYAsE0/Wa4WAUDY7/R5yZ1ffvjOhvNtAgx+dH4BfyUDDjxLeR0/GhaAV18n0D/RCgT4L05Ef5MWCHas365/wQoEgrFKrioXCABQNCHqL/9m63SFdM0c+Hrx3hKSTgILlTzhmIwLjO/gYrQ2L+h7f17eN/2uAPSf5/YkzwgEnQm+RxUEBIL9hoF+xU++bFyZtuLr116vn/PB+DH9ZvbWjM56qvttchIdg+62nTwOy2T0bwSms4xUkn0pAKDgxH51UyBoU6nG5VDUu33rrNlwxeK+fS7mgI7zphc/fssHb4QLRz88Mq/4Wv8262tU6X+zY7mAyGH3Z4K9GKSYgFZa0yuG5HpL78qNw1uMetV8/MqVGq5bdV782bevNgMf5fOfffrOQ+KLRLl44dKC7pnooNPj0FDnAa0+GnTpy0Db4IhUrdf5n09XzxxXrjQePH1+lJ56Gf/BD7548YA8E7z52YsP7kGNZPHYjYtTlneNqSDpVOOz7WZtJs2rPEDHkEwv145funjn1urr9Z8033328UVm63Xxk5cvnjwwLhe3PvzovdsOVz5/6NKpkr4EFUIOwmCL7NUimWRIAYH7RQajQjN4YePJ87VfvvomfevF80327NfRx0+efvC2Nls5c/fhzet223SzeXo9YyTgpYTZYjThfWZ3MhZWy4Eeqc2iUg8cPPv6k6O//sEP/TefP7u8tPkNd+fRowfntPGzi1vbVy7rodJ8dn7RC1u1y3GrVmc09KG+RCwikbb1Gx02eW/f9OlPnp771eWPPVefPDtz8ua33hsPHz04AtXOTF+5uXnBaKjNJifrDALpFqOoQaszDOB8IhmckO7og1GHdH9/bePLD69/d/ML98VPnp946/rngQsPHz+okGsnyxdvXj+CWBrNeKmIoWZjM8pY9BpdPxZMpAJj0s5+O+YY6+ktnP/i0d1Pbm96zn3x8szG6qvIicdPb0ZyV0/kLr99I+nH56ejmShEGG21KIs5lLoBvLV/8qMTnf1WzCHq7Slc+vHzO1QhHT775av1YOqb9JGnLw/m2LOnUpe2F/C55NJsNM1aYiZrPuaOBozqAdzHp7nhsa5+CHGMHehJXf7h06sazhE899VnZ5DYF7kjz1/ly7obG4nNLVsFiswtxsN2F2WEKzF8NqtSDRFcKMUNizpbeHSspzt15UdPt2ibPnj5q88vRd++kz3+8vX0EnvzbGzrpOE0lJ6rB8N42Asj9ahrMehQD5Isn4+Ix3cNmDBSOnQg9PZPXtz7wKEPXP/uq6ufbB5Pnnv16cq8c/1k/NpJ5hxua4kGtyXOomQ1FpgheK0I9/jTUc1E14ARxpSj/fzNn71+/J1F4bn9wx+c35xKx668fHUyqCsfS2428od06rlIADUXGTVVCkcqFrdY7PCwYf/YYFe/0Ywo9u/3bv7i5e1vjUp0+0c/zFdC3viFZ6+PMcbkcuVq1ZOQqxv+BImn1BJvNZCYNFlVchvrDQYUY11DBiui7jlgOfXTV+e+MChMV7/+inTRjuzWs08Oa1Xx5sy1Fc6j0Na5BEO65RNsiY7X9RqJQu9xcWGFsnPQYLZphvrh4qevDj8xKB2XPvvabUe0/LUPn5/SKrOZI1uXgkWlOudP+AxBncyVRKk0oRkXqxi3L6qX7xzUGCHN4DDl/PDx7HOj1n3o6y85wq4NbXz4dHtW1cge37zWnNLo06FsEeNhhTsMW1C/WTyqaeHDmrG2QTVk1AyKQraNq2tfQ9rF4ldfRxOozn/o0cMbG4q1+Mzm1c1LcmPVX2jmGUTFsRadhTVJ+lVOysWrh4UjCoNWMdzb1Oemr/2ZQX2b++zz9CKpm6o+fbh9TXqSz97YvHFrAi1w9YUZl0bjdBgUelI3cUCLUu6AYgRobR+qiaHuphorvv6dQbPsfPQqf4YzZHPP72/fUJ1h6FtbW9sTxQA+v7zgE6vNdrVKDSvlXRobQQWlw0D/mFolG9xXcaiJr/6eh/qmKneXLp4ytc8/+ujhbWh5p//muzeuS77t986vH/aJlzq9UkhtlSoAhRbH/KNjgt3DUuX4yAFvQoM8+Ve3sB0z/Fuzl59wgtqVT5/dCTYA79qju7dkW0J/+vx6QH4doBVWvV1mE4gMGEmPiQUdkjG5ZKh3bFFvfvwPv30IVsmT009+/xR0r3/z4vL1ewBce/LovfhHAhQ9txEtvyMw2NVmE5wT7LZCjF0uEgjF4xPjowc6TuuN3/3h278R8KHcqd998meCoYOv7y/e+bFgIHn/8UcnvhaMheLnwtufCnp4nUG7vCgQwkZCQmgEwIB0QCE5AK7Zzb9+O/EZKDkeefjj/F2wbY0Lzk9dAdqagWB67hqwo4bjwdU7grYQJuHm1wSAzjpmzkACQbdSrdD0AsWG7U9r6r8UdN49/d1F9c8EwCx2Ytv8uQAoaqg09b5AEPYHZ9zbAgES8lfwqwJBn9/HQDGBoF0NqUZH2s3fHPo2Af834Y6Lf/Kj4+b/DAjZR19uQ//Qmufcxy+sfxAAIycv3Qv9qsXnjeXj5UctHiZj2UBRIACHtWadBtz3m9//08Lx/wG21f723z1a++9A2+hPf/8Xd/6rAOi4+/M/f/0fBQJh8/1PPv/bFl2Tp09svmhZSTKdyrbsXg2kVYLkTJMCW9V47ORyXAgCoPX6UXmrD+wpVwdaJC0A+iQdb2wL8L1pHQj+t7Yzb7eHdoJAZ0Q0EtnbwoeM+lBHyyK3TpDtLUfd5YruDRLsGf2//HzfhrEejbsPBMaVcjM2BoI7KZnW0+qDZRgO9bbwzKlZbg8gAHrcrgmwBQA7d33vAAC/NwbpqMkjAQGdAUcQNQgOuOywVw6C+4ImG6MEW+oHMocmWv6IYhTb1fKza0K6F3gDFwrfOLDZtA6fSQhqYYvVaQFBJeNwBHUgKA16W6UdCIxEx3S0BQS6kjaEEQECUAybxW0t27Zvb8sIHHm7M8t0gAhrtqRhIWgOO91pmxC0xKNxH9UGGnwk7nJ3AMOuHhltAoF2h81ibAUA7psY39MKAP3HlUowJRL6/ulcc47oBNHt28szZAeIT2YzWddukOBoysP2A1oCUsNEF7AH3St2joEC4dioZLiVUO5+iaUXkY7ig29Xnf7RHT5m9TpBdws9x1YXMowIZL2cL8BrQAeOas3+EXDIKYNYaAfQIescUbSeV25yMu9djXZMeUtVH29vS2EW3OSUCf3h+WNmxCKMprlwS68KnbSHwuJmUOlzkn5nN7DfOtYvH24D0nF3Mb28MjDF2HFTMNKRdMI2C0K0BywYo0ND7bGpXDrBRdtdXCJDZ/1Cc1glVXqk4CjS0z063gWk3BBETJ+yTGMYacXKPREchm2mwO4grDcYoPL+TCaSDpOZ/WwunWKShU67RqWVOBChvK+nt3e8ByxQDtxSOpmZgwnCbq1powiMOBysOGSDTJCVUyVau33QFBX75xO5QrXcq5doIZmO71T09vZ1jw8KyxRmswaPLa7iFIyYc+4MhsAOO00mYchktBBknIhN0lqPJVROF8ORkkQjVluk4sQBZV9/3/4R2Y5p2Iki3sNLZx0kgVijuQaG4rgDz2URi91iwSMRS+KcW+OkC77YFMMXzXKxCtFPREUTfX3DfYPG3auwn3C4V+ZXHRThsPrKSyhBorC9lEThViVozYYh70WfBuVyVnMK5TKIQqpmYCkrlw4Mjoz0WvatWiK0jZ6vpy0IYm5pu4QdQc0mM09ZLHa91hBya+kTrALiwmar18yGcemoBMOUZu14T0/v6ICuZ9maCFrY2VqmtbzNFnecs1lQs9nqIUxGt1oHeckWnlEaAqzV4bB6vaR0RGwl1Rqd6MCBXtGYYnAJz8Rt7GxhysJ4zZDN6zKbMYvBwuJ6fcCoMbk9auxoQKEOkjBktYcoQjIiNlAahbElvXrH5ZKRg66wVcvFwjmLh7Ubjawdsnhgo4Wy6bRulx4iMSVcz6g0fhy3avA0zkiHx1Ruvdwo6+kdkCkmRAWPIel3oXTCYgtiXj9tNlpa68fO2tVa/6zRRFkVcDar1REUhcvoHE6qh0Ryp0GuU/b29CuUiuGZlWIm7STpJcRVm77EOc0GW5QzWViDRlPa1JsYkwINxc0GkvK4lbAPJfSDo1KrSqpTDQwOSXXq4dlYbi3IMfR6Yz6Rn/NRbgNdKNntvFGtLX1ghlw6JR05SFkxhGVkDoLBoSGR1Cyf0ClHNRqJ3jC84OITkQCXPXwxU5mLeoIJmM00CJhDNEjzkNHIIYZIZDnhZZzhOJ6GScIyLhrTyVQ65bgDg0y64fkI40twnsLSYR9WIZzpGh2MzHnpqEvtii1gNi/vyQTKlZqLqM8zlZyPQltVg06h0IrGCTcK6UUryRC2XOen1ybT+eOpQO1IZLHxTjpXChv9mbP5qWIo7w8fyxyKoKSDrfoJikDDBZtywt6npd0ECY8fKvKZE0fCs4eKfvbwMt9sNo40LiQTTU/paGHDf2HWV2wWz+JctTbbjB+ymuLhTJZNO1F41IYpfBwtXsmFFteOx2fnYm7u8GpkrtyciZ5cijcj9ZXaaT9bjSwfiZ9II4npo8vsUs3pzM6WidxkyIwFWJTmXKL5THj52LFEK9BA/thRfjrdqASOHnXXi2tLibW6OetfOBg+uuTgWn+p5cMc4Z6vZUqNlNUc8Li8PlyUy/ALhw5nitEwV1g5FC8UKlnf/EK4UDiznDs0RSVSqyenjyy7WH72xMGlo2En1phtlOthu8LnZV0+ZCSejKyun5qMpuKBqbXTtVK1GnFPHy6X8punZ09Vw7HmhcvrZ84EvOXy5ZPrZ3ieCa4uzh0KOmV2LuCPusaB2VT9cItugbDPV2nxG2CP+t8Ux4I9fNn0PV32Grv+SLv/B+/+r7Z/0ug9vhsA9tKew4tDLX7DaW+9s2XNGHO4rXXBXjzg+54v9w/t+H/4QQlPehYCAGXp9gdLOADs5q/emDa0+Mml8UyrW36MKF0YeBOPDlJ872fHzj/S+B/9sPlDc+kIIDC9s/12NgQA0pP3b88zADCS2njrOA8AneyYJe9s+VHrtfCeN4Khv38X8Ef8G8MXluvBVRDAqMAkUgQB+9KlyyfcIIBsvX3zaBQQaJnscjoBCHY5BsYIactPX19fv/ANf7e3v5EDCVhh9FzpA1zQ6BhzBASYlryLBUAgfnnzWD3ZBgTPXjpdr3YCMoc+aMNAQdv4/gMjb/RE+86dO1oBNJYScXclKsynKXdiqQ9kqhQRKe0WZotLywF+SFi7fGK1nJYL0Wx1iqf2AntFfRPDIy18V8fOrnZAcOb5g4drrvPgyvbBs2t5GszeO13b4A3gZLMQWqS0wMrZ47MxEgWSC5VqziYGhoxWxcj4DkHbns49e7sAwYWbi+sb9MX9Ww+uHDydWBCuPrp68dpMClw5VUsdrXrA9cmlMu/hgFoxxbm1JkCss2JyUTfQsX9oqHufUPDOqdXSnLvivvnowTsPmmudV55sb9/cWhKev35w6sT5tPBCNldiY/G2qWyUI6U4qDIQDlgiAnYPiUb793UKHp16//lmiD209f6TTz7erkO3Ht/74O7D4+Jr75xcO3+zObGRj8d85ehYlU8ECSO8T82QKKqTCbvHxOOi7n3A7bPPXt7P07eXPvjo5Uf3c1PnP7z30bv3T3su3t04f+nmKrvSjMZD9QySyiQ5NwxJzT4Ghc2KXX0SqVjU1wteuPHp11/OMQ+qH3385ZcvZ986/PTexx8+uVY+dH/z2lu3ziUm5yLe6Hw9EPVzPsaqM5G8l0Dsuv4RmUoxMdrXceX2i2++OERcO/b69Q9eP1m9tfbi0dPH9x6vzj26dfutt24Va4fzztDCUpSnvIwb0mJ0wEsiFo14RK5Wikd6dq/fffrNN+vUye2XL7/54tPL9899/OHHzx69PrP+6M7G6bc+mKqeKEDE8rGC2+GhOYPR5QrybqfdoB9V6/SK8YEDK7eef/ajderwrY9ff/XVyyvv3Xn24vOX7z07dO7BnYXZzYezxY0yzC6dKGF2P89rjBGSD7JOu9E6otJq5KN9fSe3n374xYJ3/cqrl6+/fHHt4Z0XL14/u/1i68yDd5rpq+8frZ5vwMzCuSbsCJSjKmMW4yMBymGlWnidanRQdO3a/fsn8PTl9U8ev3yxcfTVOx99+OjhzWe3Dr57I8Jv3btUuzBHEwvnlmyYczarN+ZtgWiERmGXSNWKXzQ4dmVjxUc0N368+vJJ1Ek1fnrz+dNj1956dGv6nW00fOPe5diZORqb3diAEevBEqSdNvujKc6J+0QKjVElGZG/7Zn/5aUvvvvd9FZ90Yws/nbx6UM7v7X9snj3umvyyntvJTZmDNrG+kWLAyp7rdoVKx+Ohzx0eEyhNGjEw4rLm9999pf/8Hd/608c/ouV6ua/jD29crD8fOvr9P2bM/PvfnAzc3rShjYWNxWQyW9D9eumYCiVitGxcemwzDQxosz+9uv67//wh9/yiTM3z8w8/Mv4s/Xt8oONr8sPb64Wb7/7Ue5Yunk0mrkh0xpNhM1wXhcJBwNxS0wk6u6TS0Qy+pe/j/6Hf/wXdzy5rczahW9/Gbm1esq5dvYnuXunjnA333scavBXTgfdm4hcQ2adSEPGkEp/QeeYGDwwIBVLR9zPf1f867/71zxXe5m8/vPfzwZOVwMk4b8cfC9TcCYqZ6hccI7yUWdYD1KNB1J15cDQGIorwpqh/oH+IamS2fq5J5T9Gxe9+qp248nf+L1YmWDJIxXz4bSbsuiWjNYUjVmbx2Nu0mXSZI7QDqlaMojHNaLhwZGBEQ2z/utswPely3vm09qM+y8Cfjrsczk/rxIO1ooYG/OE3U3AupkjXthhgBT5JVKnFCkldH7/wMA+iWhYz61cj3Ds+3Hvjc8PJdnHtVi6EU16b89VUQ/GGPJLh1EnjulK68sOm8FhmL8W06kmbGr4oHJ4ZFAyPKjnjlx1s76DxyZvXD/p4Q6urc1sHC8UZ0/d4mMBpry5/SuepDDyk48/wm0mm+34Z+cMWoVWAx/lRkcGx0aGjPP1BzTLrW5d3nx+Puo/cfe9tRvvrUxf+OS1N5F0n/ziZ78o0DTm/u1f/XURh23w2Z/fxQxqrV53NiAdHRoZ6YeebL9wc/71v//4/p8f9fGn/+2dW69/fnHh9H+6UUge5R798r98sxzKuYp/+u9/eplx0tihPxyesxhNEHHNZBWJRkUiMwKZ4HAxA1E+g4nLzSzEYa1SW6jOzTshFZzLV2pRKhxjKX6mnKg2piPZeiGcK09mssksl81PFtyoFaWChaSSZPQmX25mNubQalTZyswcbVJDLR1VDjDBgJvkGsVQcXI2kCqlg6liOZ0OJ9lMtpJhbHAswBUiEidlgJjMTCNm1WqUyfLMNGvTahOZYpFFeRzH+EbOXygV3cl8gk8XaomkP+7J5CoJyurwu1zZwF4Xpjc68810yqzRKKPlZgO1GNWhTDHjJ2I0huPNtDuXLbiiuUggW6oFY86AL52vZhgzE/A5U1gnhRiNWLYZiJg0Wk2iXG8gTkgdSOarPmeesaKOZpisxRKuWCbI5soVfwTzeVO5cozS56sRMmJtpwIo5MtMkyHOoFXHK5Uq25LgoUwuzbuyOIJaqxxW8/lwNsGTxWrVG7Y5mUyhGqV1SzkOC2qFzpSDYnKThohPr1V7yqViNkDqXLlUNO9PuDCrdYqxFQm6pfP9WL5WcYesBNLCh2n1DGnBvONCMoYwjnROmrGZIJWtmKtTgbRBXYxOZfggarKYqySZ93qMWIxFEtUSEzQR9lSxEqSUiwmShfaAlMtJQW5+T1JlgpTymKuhslBaJeOqoc4g4bHpC2k+7bKbqDBl9xZLJG9CMDZfCbiU9azTq9kFul0B0gDTO+eVEKSYYJGUCjIr5XpX2AK78GmXsXyMi8UJPcrTKFsoYLzFYXbkK36XYjJEctou0O1lGS1k2TGvNhrkEsQcV+mMCtkExWgtOOoy6EsL0XgloMB5v5svtlgAQSBDuuqnZRUf7tV3grTLzmm06o6aRq+VSwymgFKjk01IELvcAJsUCkMSTXNZwoIGIqFAPYmHYNJicFf8lCTK4UH9ThCjjJ5W5nekNWq1XKa1+ZRKnUwqx7QTCtgsV0F+OZFOW4z2aChXmeSdAYx3210Jr3PUFXT6tDtAm8VEqQ0T7S5T6/krDSbapDEo5Spco5BCuFxtMk0Y4vVWQr3hcDPv8XndU5Opspd0DvM+lta0A5DRwaq0yjbKqtOolBCKQFqzXKHyyJVyOC5TQRapMb6kluNMMJItuTw+99xUrsSjyFCQYAjDDsBidfEaNdTmhlRKhUJvcBjVkEyuJFs3w1RlGjMi1UZPaBU23BkLTFMunpuppssRKzrswd2YrROwuwJRm87a5tfJ5NIJ2IBqFeYRyYSHUMqTVqnMYpfqYufUEghh4v5J2usNzU6miikbMsjgbqu9E8CDfJzUm9/gZRIxonJr5CaJWOrEFbIYLZ1AoQlDrKGfMOKxLDvp4l3xej1Zytsc/TwMO97gU8GQz2xrY9Vy+bjYrqLUCq9+XMFZ5LJsTCwmDTLKHXDoDJ7Wmk1TOJ3M15LFogU+wKrkmHkngLtQyhOwCTn+xM2zaqeYla79oj4qp2xiDxEkxnGHm7IHwpiVD7k9cdqK8sl6Il1CLPvcGjWi3wEgDoIwu81CT6557HGTGUW04RrDyHAIcaigQxOkHp200yallU/46HzAingzre2ggJt2OfU6u7odsNhtNg1tBKlqcS4W9Q2EaJtLVcw6jL4Jjc4VsGv9Hokdgo2eUpiOoZDFmS5HoznKsLNVxSISIWBAHLjBqQSZtVxURzW6TxgUEZ1oysRKx3XmUY3Zph8XWcblWqYlvxiNUo9nCpFwmtG2W81mZBgUyDAMsXnGQCuDoAqsucchkgWQ8ZDRLBIbaYnYqBaNii2yUSVFM6hDO65CEmmeT3jU4KjOoGjp8AGXQsfa9gBGpUwj1hM7J4akNmiClowODEmd1tbLNDAkNsklEjNGoXqjRWYOBV10wCMGepSKoVYdsI8cV2LqnYBaPjHaLze0SwZHJyTDxNBIX9+gzqKWmfqHR5WQViS304TVZB+HfB6yRfQjwB6peJ+wVd+gIik8tgNQTAwNdItl7eK+3qGhHrJH1HOgd0IpHzUOScYndObhYQ1OMrRt1MA4EQcJDwMd/QOdrUKo0ySa0PS3CWSSnp6eiYF2SU93/0Cfba+4u1XvjKkGdKNSuUhnGugVWS1el2lAg6MWk83QLxDu2fPmO+YOpXhwfK9QIBk/0NMr3i+UdXf3t0R2l6Z7b/fAsK5neFgmG1crerv79XrUqupWQka9Vic7IBD88TsssHvX7j2tPOzp37O350A7MNC3r3tf/762kb49e/f39XXu6RkdGpT0du7aOzQ0Ptazo29gZGRwcN/Of65H23d0dLTuo2Nf65IuoWB/3649e7t3gv29b+ze9t37Bnr7hve1d+7q7unv39PWtWdfd/e+nW3/XMgK29pbaRS0dXV0drbi2HWga1dnp1BwoKer683A7r0H9u3duwPs6Nq9a9/uHW/Kv66uDiHwPwFQSwMELQAAAAgAAAAhAP389zX//////////wgAFABzbG90Lm5weQEAEAAcAQAAAAAAAFQAAAAAAAAAm+wX6hsQychQxlCtnpJanFykbqWgXpNpqK6joJ6WX1RSlJgXn1+UkgoSd0vMKU4FihdnJBakAvkahqZmOpo6CrUKZAMuBgKAkQBgwguY8QIWAgAAUEsDBC0AAAAIAAAAIQBNinLt//////////8NABQAcmF3X2luZGV4Lm5weQEAEAC4AQAAAAAAAMwAAAAAAAAApY7NTsJgEEVPsVSoCFhK+bPwETWfJN1IAgvD2p3GDQtWpoESSAyYlrghPAUv7KAJdC2zmGTuzL1z9m/j1/eJwTdbPYuSaayflR4t+zpQer6ON3G4+ljHs+igv4SfSSR6sgi/IpkfnwbDoBeonfp32V0UHdr43NKiSZ0aHlVcKjjcUKZEkWsKXGGT5xKLLCYXZDCAc/1Iz8hkimrJNic3ttwWxFMUb1kyHMlyJdOT7DoN+dKSb/5Z3gfuuUvRH9gbv/Rp9hN57kj+x/0DUEsBAi0ALQAAAAgAAAAhAMCIIvaiIwAAgCcAAAoAAAAAAAAAAAAAAIABAAAAAHRodW1icy5ucHlQSwECLQAtAAAACAAAACEA/fz3NVQAAAAcAQAACAAAAAAAAAAAAAAAgAHeIwAAc2xvdC5ucHlQSwECLQAtAAAACAAAACEATYpy7cwAAAC4AQAADQAAAAAAAAAAAAAAgAFsJAAAcmF3X2luZGV4Lm5weVBLBQYAAAAAAwADAKkAAAB3JQAAAAA="}, "1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339": {"jpeg_sha256": "31ff809b3db0f11d0724d3e6ef1933feffa3717217f0486068a3c40e38798f23", "n": 158, "arrays_b64": "UEsDBC0AAAAIAAAAIQAHW6gF//////////8KABQAdGh1bWJzLm5weQEAEAAAKAAAAAAAAAAkAAAAAAAAnXgHU1vZuuU+ApzadgeHtju43Xa3MxlEEKCAIijnnAPKQijnLILIGYONwRjTdts4YTBJBGPcfe+7U1M1/2cOfd+8uVP15k3VnKJqlfbZ69P+9tY5ay2G6TwaUwQBDwje1Oo6NM6b9VdvhtzlN+9dvam3O11OlU1hd2p1R+NElaVDB493tKkcOvjzrfLq2ntXj/5u37savvr/c52CrqEwWHodhCipq28kVUPQ9aoqNA4JQReR1ShsFQSOVdQg0QQIIM6j0Y14BICKKtHNLScAQFxvbG49W9RIxzWSmYXnmBjUPRy14BSJUluObS1AVJCQVQ1YBHSpoaKqkVoICsqasI3kUwD6GtPMoJ0HoABJ9LRcKlEw27MCxs9Egz+R5dC+qzXaXcQW6rcXBTJvt0D+QwGOn/xNJvwOukKrbjDX3gAFDaiwHFUJoIu6+zFJXSPJ+LhfTL9H7ObxbVL+dZyhuKwao79626RkqcjmqyflAqeWzL0JFbfW3DCjSsEJfDFdX9cIEDdSQ3IWnsFRhh+K1EjSSG0FQ+Qtxurv3i2v8d9GSSU15QrL7W9dzZaWFl01gsKu/ZUlaIAuSu+RDDUUgKiPjaDpDKGYSe+SZzDYgao7JYxMZVP7veLKynB5M5dxr1jpL7snJegprT5soYxafqOygwjd1NyrIaLZoJBs761uETF9dcW/ysZZ5X2VyApqL7XKV1xZXhlmUAS023duygT0RhyTwPAZv5GiS3695ZVANFxxMY6tBN/W4cKVJAU51YBGSseM9waQNTXsLi0yXoKsqnLoqHJu8Z27Mr24rlnKEvn0V7iYilu3/VoEs7rkTrPQDN2sQUZQ9A5UL6a5jj3iujtQi6xmhF3IcDmyvNzgwSv5d27dNbapGlqNHFXacIePrrx9O6QrEpSV3m6WOaDayhIlnuVp6ibhayWxbGWkoQGpVXfXe2prkWUOE0YlKr5dHG3TkY1hprrLjqahkLfLEh1npWWld3EeF4SurkCLHcHmbha5XuPK1frQTSi9yI3x1tXXlMV0FLEceReZ9WnJiiGmYagdR6tF3qvoD3ylqKos4fR4IQKypjHaFW7tpNOwak2wxoXBNVolMoyrrrG+Mqdhi2Wk0qZcQtXMHqBoByM4Vm3l3aah8NcaJLLUNByGCPW1NeHBiLCHzMApdJoGP55IsSrZ+I7axvqGIZ28jS/HtvamVSROAiMf7ayX1iHLWoeC5y1NdSXewQiCgm6sDQxHzH6KgKrQSqyWZqpYIeER2tAkpG5Eb/dyvQHPWI9CKNNj2yaGK+y1daWcXPbLAKapNDwUgVikhprMeKfeSGDT28zyHPxtuoiGJRNgyTR7VuOw0m1h0WCGI1fqeJrBkXumulocr2/grAPTVJsczUKtrZj63HhW0UZk8DstYoeCQFMPGGndQkKLweMSaBQtBp0gFqZorcImVar7rgOLrhb1931hIqBR2aFuqJWJx/QOZmUBMqbN1W9R8Im4tsmQwCggkjwPc2KupAnetmioUW1VSoPZ0at6NBarGx07psURGvomByESk9IyMjLAiGOJjuBkO1+Ew/kn0iyZtBnrnF2QUCToFuNoT7DGYNJYEt1D37U1YcjGvqEiGQ6H6Z0YgvBUMmNotB/fhcMZDCMuMr+52ZtwsBlSAs480sXDSTHY0NhYsMSqVaY7+3vP6ZuasG29uQIREY/uHO2FcCwSq38kd3OYiNI4xqK3RMSmoUBQdFNIQcsms4xbmuZG0diI7nuHUGAbGs0dFzc3oayjowg6BUPoHOyH6nkYY+9Y54/97Lo213j6ewkek/Il5Zcl5EbGYIp52UBoaB0d0Hxl5LJNPSOZAi4WhbaPDEEtxMbWzv4cVCWpCnaP5C70GElm32D8jIJA8Lg6VaelLU20bJhxWkdpwPX3qU5ouBx1V38YwSTUYZ0DOYhAaSAN9OagCk15sm+g+3QkKtaFRmNFMgrF5RrWFolbGykxN61QRa1vTA8pC8SCVk33kAtBaEWiY8MpCMurYUz1d4IyLal/sL8HIX+Q1gVHIxBV3NIRG9chWtgYlsXGQpDVaEJ0iI9AG0nG0JATQkobMLmhEKjWUPSTAz5w2+AYGeiLI+hjnUL/aAzCSOlt0TEDhBESWTYTF0JrCCRbHweq0rfoHYNWqFSNxuQGAlCpnmcfHwiBG8boxPCAHyL0JTTekTCE1Am1wXENVMMni7w2HlSlaKFLY2zojpbUYR01gJu6BtLgQBDctqhiE70hcN2Wvj86EAC3HR6TNaMCVwytZn2MB34moVVaOx98L6CLGDIKOKeX9Lb18cEFo0zQPegDl83a5HDOBS5bHOODXe3garvJ1aG2gQs2lcdOl4NLdqqzg8wC5x0Sa5xfD07bLR1OTys4Y7f5s8MO8I1NHRsfNIPzbvP0eKcFnHPorT6TC3xhM9hcKgk467DGrWICOGkyhJ0td0GRQWM1SKrAMaNGG+g1g9NtUsP4gAp82aGcGO1XgRM2iz2n8YMim74tK+SDInOb0SGvAQUqg0aluAgQMgObzvoeINRGvMqnA8dMOt7QoBCccqu8g4MCgJDaXEOuKICENveomwkgps070fYLgCh2Tx+vEEBEW0dHE6y7dItZ4cWDIrZKZndSQJFSb5iINQOI1u4d9EcAgtDhHvPQAYTucA4bfwIQymVx02Ee0q5XlUOgANWm0NiuAwTDrDK4GgBCZdf3Omug81N7E2tWCLqU3Z/fMcO6HzxY3miDoNPJvfV1C8zj7+1spCEAfflkc3sBgnV7YHd9vQDW/5a95387WWDZWSHOzBacvv+PRw3rs4hjsbcp8vYMhBDupjr25iHo8ntfcvcd7B8Yyp4P20UAOjf90LQI6z+iewTZe6OcLrL3PP1USiVJOx89+HgTZzSbVIrt62e748Pu1ukriJD/0Rg3cgf6Id01q5E3A4Sio0XKlwLoV2+dxuwQ3JCTI/k1mk79+8ho54NmbXxv8z3Hhy4Z6w5GalSNhb2dr2U4Iw2BdayvuTkaUGSZ1ca5XoAgSxpDuh6neHMkvZKyOlLb49N2pToS3d7bFZOlyPmRV/dpDP6Z7MqCkBxoR6h6t7f6/QnwVeJD36hjACAkbPRD+2x3128Dz+Zx2UDP4as+d6UzF1nf21ZXGBpWcztr/TTR99HdPi1bFiuwPt/Jv5Hdh37p258dGngMCgxycj78qnP6/pxivtJtCG9u7fc0K0ecL7Y2dnA87U54bX+Xwr43/qndICFPnbJubm3vY19CpPjh8x7PG/CVcoK7lVy3zTzdnJ2rs6kiO7vbIxzRQ/uL/OpHGcv5zvtuY18sqZjbbrdKMK8v+De2drYJzxC8vv03fbYl6Af3inI58Eg682L7+UOszuE72NnobuMsOZ7nV3cC1NCy+9n6vjSFer7uMYspS78E1rZ3N5unC2T386sj2inoZveW7o1rmDPy5EPqg4mdc73bmsu9Em86H+ytrIiXkivB4XebqAeElbVBORPzsCH+an3/GSpxQr+w/t5XH4OQ0/u5HC0oHny8Ht7olEx63+UHcp/Gdy0zBx9eNLzKvPf0vlqu7mWtrk1LW+pHsOnXm7sztb4vTG823zPqXFD9491BI8FuG3uwnl19mZ72v91KTb1f+6Sc+7g1W/Vy4I07vTSK9Mjyb8eMxLpAc+fKxl5nreO0ZXN7BVtvgZpfbI/Ra/SRB08/9H54tvYo8G438HR5/zN7cX81iFq8v+hK9KnrjLr9xfspXK2C3L+5sx+utX4RzG+9bUBpIebS9lArUp5bW9i8/35tgzG8vRc5PFzvcrzbxkwSZh4N2V3C57Xq5OZkxs+toxIf5bc3FY3sk/d38o+xdTzI/mRzGFvDGHzxYvXJ3KsP/Omdg+HfNvLc4fV1zorq8VLWZeicQ+tSm0N9fnU9nvTk485HFop8fCK/9bSqhge5n2z119cwH8y/fPqqf+lDfGknP5V+uON5svd+fHDg5e/dOceDIdhSLA3M6G2NzU1LO9sblGr0sekPm/MV1VTIv7DVVYNsfWp72/vGopyfH9v7vc//ent6eePVI3ZmN/N02TnKs6bMz9rmVA5y668rW7sfaLUNBdNv9+ax9QTI+yQfoddQxiTLqTcjzYMvBnfmPan83uCL9dfZxtE957OV3p4mdZ/0sT7hV/K4l5fze+tcLAEx9fbjPKMBB9kefXQK6whx7pP0q7fI2ODg9oTW/TYffrj60lTu2dPOrc711zNf0yYMljkaXvzlYn53n8/AIwZeHdyXoYmQZWpPJ0IRyPqE70VftRk9tz9hTURfN8xuPw3USHbKB/LbXfXV8z9P6TRxEtJUOLrz7oBbR4aSc+tp2LMDVedLtA3DuBfQatdSVborC3sjzm7L798t7SzMViu3vx36vLbQWDH/xYxJb8VU2RFd+eW/4asoUOL+2xibwAGi7s46D575XQ9b/TFWrju5uNYfHeUtHX+2tzBXbts61v+P989qi+cKh202bU2ZB5F5++xzTYkQ8k+8TJPqOIAxTWvWYQXHp4jWfKzKWvDoea57CrtYMJ6fn23I5Auyh6+fVJc8RfSYjOLq0gQiuTSXryi1QNaZWXdTpQy0PmEJhY1KaIbofe+qDkG5gVgqWfkM8r160Wef30N45t+PlNSsQu42Jam4OAdpZp8/qSwNAe5SpxtVlgQNq1l6Q0UQGmHYfwtUeKBwMuB2VbyGzIubEcfjfcjatZ+917oLqdo0DXeKJwBj/HlXdUkCkLbiwrqSGKhcNTY3lHdDQa1iZ6AyAum7O7zBii1Imtuyih8cQGL7YaTR9RGQLVzU7ZLfQOOUi1tT1gtQBz2sxsosKNkUYZoqegBxcWx3q3Ec1DwwrLxsHgP14oi8r3cO3Mt4g4ZPOXBprTnN4z4Blz9GtKjKF+Da7qqpoeIRuLpjaWqsnQOlOzNbG9FV8FN++HAXfi9ffYKyvPBkwIVdleftOzk4sb+4PtvnBV/svD7A1b0G53eX3DVlcL29LApV8wRc3H/7yv5bHnyx+2ZtZGEAnDlY+u33UT04ln/14Wl3KSh482Fl+XEzKHi/+YTLWgAnd59rkVUL4Mu9xab6skVQtJZ/jO7Og4JXG3PYySQoeLf1SLBMBohnO6Pp4bMAMbsb6R6F9f/pjl/VlQMFaxv2SsooOLb1DoWumwKI/u3Ru73PAJTZmr47ZQVQOj9e+uEmgPw7E/RZWL9du0MJWIeh2Prco9+aANS/umSe1QBoct9aaTEDyLP18F5uGkC21Sd3l9QA0q8vkz7+CCDx1pvREZgn3lzd5gF4/HE+/+B7AMU/7M2uEQE0vjuOGmEB8L2EiQDwfWwmehpG8L1VV3SEiKtfQ0cIIf6C/9sFURWmi0fTubfZ5X/VwSp/OrpxoQ2P+KtOEeLfZ/6n/OMygqIExjOYGgwGnlKEbhRVH83GiLUnj/CLqrN/MRH/+Tous+6KsPCdnxsoVAaM50l4MQXGk1K679IRDTdY89cCii4W/nMZiP+Df5NKUNDg+Tf1OIYYxmstBJ4Ixis8hrkYxlP+pP7YUZ1r4kvQv/YD/XN3SoZjAZgHlf4PmVYNH1fJCJNvhLGcz7GS4fGruubg9/DU44IcqeCI8FXJiX/pB6p2rD7UwDavITeba4OxacDrtRUCRK1IbVDBdSrITeFGuM53GqPjS5hQgDX9/Nd5FZ06goJmUzJm+wYgSIT5RPsZgKCqp3LuLwCCaLQZLCcAAkPjtCsKYBtIQiXvwnUutA9wj8634G79qaP2WPz29tBdqIjUoLWZfwJFFNS41vQ9OMlus9rMF8BxYgtbYz0JCqhMToKKAIgqND58Ga7zlUBdDfdzXoYTWTNE6AwRTZFrqqAzZKSoRVMKfcMVOdst98BXZIpEH7gMTouFTKPpGCjiUSjdpKM6SJ7jSwhcUWO5rIAMcZGEZXB4rdAFcn0DRYqHrpDYRrUFC13B0mUC713oSiOFyLNcAN9iyDyz/QQ4QUcLI/UIcLsNz+W2uYt+IuIZXI4KcYXWADPkiGusFoFGz0Pc4YikAg0eKqvAkjjqaugejsAR+25Cv+goHLf1JFRpaBFJxanzt/AEpoDvOHWDj6ZwZJYTNxV4gVhkKixO+5UaiQJRVYtvYUkFiGYmgc61sRGYIEGgSpRDNU66XMXvulfMoQvlYvu1EjWeKVB4foLrBpUS3zms3+wzSz0n0CginSX2HpdwSFyB3l8kSlGlgqipgOxjqTX8LiJaQRPLpKYmPJPI5sj1TQQ5b0Yo9hW3KhRatSz0Y2sdjsERRX8w8MgcoTz9U1sXVcrypH/kOOg6JbdTzmQRpWKhVsJuJLM5Qh6Pw+CP83gmIpcsYgklVhSjDkfncYP1Jj6ZJ5aksc4+pkyozhAEyladlB1zCfHNHC5P7WDVYHhkFrNNgKHnREyVnI2j0thcNVtcgaZyGA6eS0aAy4dlniGWTC4It/GlrWQiJ5NloHAsOledaK1CqxgsYVRaQwjCibKDS6gncVkaowDZRGXRrGaPmMCE1+HK5FgSmSoVEXBQbSRNb5peiRVzqdIktrZepGxVZGi1OKehxRhlk1BoMU3nE1fXUWg0a0jJRlOZ7HQmGWeLKC5flK9ihkcmvQlWLcnfbeAl8UikykRXpnANaIudYkyKCbUUGVkX15ZUERktzn4qtZpMknhjvjYFBZ/0JtqSMylvuyvJpkmHQ15DilVflbCK1GlGfbPH1WrKcDF1aBXV2ScrK8fTWO1pDENBwBs6knpbtNne400qLOakrT0SUwkl/S6DO65A13t1Al1C1kwz2mnmTim6ucHMzvRRm5obKPyOGJoS71B6bAmOOh2J9HQkZFFtzhTvDmmN0gGvORYzoGoDZoa5Uyfg6h1q+whXIpWnEtkUisTCkUntqWacPR5K2pMqdsjZGXYmDSFL0OKPxVQOq9vk6MpqaIwOhcQ/pNNYnAGxJ62xuP0aviuFh5/nBo51AI22u91d/qRC4gt5U564MOpY93SGJwTt4QexwWCn3JYcfdTW5TXbe3seiDsjrlQuN0prDzo6c4PZoDOpV2VTibQl47F3ZqIxf5yYsD6IxQIRot+1HEyE4rRIdjJq7ozL/clQqiWVtMTiwbS5PeFMJ7rSCXs8kE71xtPmhD+ZSifjkXh11ORPRMIhpM/yIByMhFGh6FDMFovSPOFY1OpOyL3RWCzsitoSkUwsrQuFU5FsJOGJRdLRZDgRihWHbNFMNBC64TZ1BEOR6M1Auj9idsXrPKFEJO5NUAOhaDRlcqn9kXA0Y4SrRqKRZCiRTEbj4WQ4dcnjjGaS4djXNrUyFHEnz9pT6YDDk/7FF0/Fos4QyhkLxuM6OdUTD8Yybq3VE4lGk8FYIBiLxWPhriKN1pMMG+MIvlTi91lTCG570Ga1JU539GZierP2Z0cyFgoamsps6Ug46W5ieTLRSDRkU8GseCyagvD2jkTAkYFqFRKn09cLYaVyuaQjCUkTqbDChj9hzabDHZqz37THYsFY6EJxMAw3khI0wZsVi/m10C2FKWEN94JrQpFBHpoExQK5RO1TQ1hdh0UlOodg+ZJd1h8QBdJAJhG2F5wIBNOJ4ETJ+WQwnQ1F6wE4l+L9JUQN+ScnjnTpXEhd8Jc+fXPqn4L7/9B/7trh10covdtWeYQtRM13RzfOcGr+qXMF/6X+FzpUj9AwnhSpHdQjXeJls8ij2XVi2dF6oGPXTv1XPuSi6q5MBeP1oZWlI93+evzvn9hHPkJlTv14RLujrfyrv4LTBf+i2/9xoah0kRceqlINL1hhLH9wsGM48gGdU/N4GI+16SJHegld4pz/T/qB5FxbNAfrYavSnR6E9ZrW9fxZCkbZ64/v22G86aU+vXbUZ9tL4V+6feraMfAv+h95qvM+gXXf4FQalmGUx3oyTwsAIvlxZ30S5utCsXk1XP/SZOrp2SMaOnvtL2Lh8b/KTazPRZ9/CxChFbXnNWw3HV0J9YtTADH5dnn91XGAGFw5WHoA1yFLsZtIuM7XiUeGIyOEuI488iEnXr/c+jSEhQqn/61/8WkxKOz9zds9dB0cWxp5v/TiEih69Pe95RXYR7hHJhctcJ0WPP7Fd3CdM0pXLbyAH149GXnV3g6dffHp2eIDGXRm8a1zaZoJffe27834DA5cePHn5pt3V8HJyPP+jvFCUJjka5eO/BKdb50/AwHsw4WcRzuGKP3tYGf3YRoqeboysPzaB2Hf5x70DNigxtm/Ly/MtkJlrvnd/MK34Mfkm635hWPg+FB7399gH6EbXvrwTL9QJJjd2d979gghntt++mZ7CGFZ6u+NTqURqpH84rjbBbGH9rc+PWuBhMP7+dn1WxByaGbh3cPjoDOynN/ueHwlPLK9s72+/EVkKL+xdvD7ifjYZK5nZbrANvT2UZ92BNG2+XH94HUn1Plwb//F73Yo0bb8oe9TPTTlWt7cyfaSMw+287t7zyrC9/MHW59eFOdmZ3qm/nh2IToxkY5Z5471ft7bOnj3rnDx/t72yuzzwt/4a/uxncGC2fDa9t54yNH9aGfv4NOcovf5wZ87B4/FT6YmlaN/zmMns95o0vXoxtKn/Obhcv6nl2M7mxtTn64/47/L9z7/ePVhPL+1NWmfmHt8sPvx02jXo43N9bWN6dir+9PU7Kdhy5I/M97jGuQ9z29vfH77mjWVy29+HPkk7ea82nyaPlS/GdpafZ0zvn/37NPe/s7g0tPdD4sLrwYfrTyaFWS3evueB/t/63VE/PMHe1ufXs57Y2kYew8ybYzftlY1231r/W/NVqt9d235YPXFyvCHh+srA9PPxt69mJ1XpFbHFx/Fsyt9psjU5OdP+b3F6fvm7P7Ox+61BZ7gwf625+3Qe5u7k2rNHCw/WHPpckMHY7+vuWefzx2OTz9W+1fmN4YSydfd+uT74Z2Pu7sv5961d3/Y35rYnG6R9X96Enz+Zlel/SPzcOLju8xkn8sZOxif3/Fvrs790TWzIEq+XfijJ5rojapi+5lP+d39tzNLmanR3ZWxlbeKtHzV6Zw5zE8Pb7xf/P3TdmRoLOKO7S882g8s/H3m88jjh6Ged08PO70xq0/VuWneebN/uD75e3zMkh4aXPkz+kKLJ5v6D9+8+PT2v6/sf57vnRlX6Yc+Tj3eHs3+beLzwOLY318v/54ftPtZ/YqBxy0fZl7+4+3on4OLcrYp8eTvY2tiNkccPnzc/Yfxvx2sfn6QnorqOxYPJ2emunVPpnZmxoYeB0aX5iZ09sS0cW6NOuvzbmcm/pz60MI1Wcb+7N7Ewb7VcThvOVD+Lb9xsBBckrvHNnYmwgF7qyy3MGp1JUXSByu9HnXIL3r6yWRq5TqFY/nkE6zEIhk8dM5wsjm59o+nkrxyf2drb873TNq1sbM2wbUwRejJrVGGjqKqGz0c4XMYoprJw25pC58rW/ioNYvTTqn3QGST2nsU0s9TzF2Pe83zZqhjWjW+Fd/o0QvYInfi06BAzpTXBg6fcBUiOWfo83OmVCMXPj4cpkkdDrl9302T6CxK2UGK8SFNG1E/DDsXfNlO23OPnauW9XdshS0ShUxt+7wklGmU6qXDVZrEppEMfnzZIrGE1Z6P0y1CnVMt2nfS113MiGpaZ377OmK3PON6hW36PvWOtktkFKUGD0bk0jaZcvnNM7bIrBb3vt+mCVxmRfjjOpGjNytEnxqsO1aGq+3x9c79z36d6N13OU3YPxM+RPYKTayuF9vmlEQtkT7uiXUItQrB4vCckafUS1Obq0KWXC2S/3HiwWEH26Z8i5g53G/X+vPQy4H7478/PzjzWKxmpVYf1q7KtCLLNps+JZDLRVvqjn6OXCUeGe+NMBRKvv0DFHjfxWwf2IOM2/th98uPUHojvzGzvYa4D3vxJ/uqk3mRnJtd/fa7fZlMrN7+AbXJk0n5L3TiFwyJjNfbC1p2nrK7Dz6C5q2tl137m5B0P3/weL8L8izODu3NX4Mm2yWcx1UQ4nGAz4sMIU5txQQi+WHxDwd+sZi7wQKniFpzUyE4Vq8ymQeERTdUGpk218OlDbG0KnU6Ku4WSNWd6aDA1h6Me2MRVCAcTMMG6pTEG05kfFFwkiFX4IpAIVIUaBsRHvtZJhKqkn1MzIBQppB2RRkhnkCZ7gtSHNZI2B+NlHpj/q5UNlFEcQeyOV8MfMGh04mFoKCS3dPRzy/6UURnSrw9VFSWJJXwshGyAcdXe7sDeEubN90RS9yyepyZRCqKQJnt8YwrDk7zGCQynJereeOpPmbhZRGZznP0EZC9JKGAMRDBabBcnSMVxNr1wXg0nvjF7rNnYesJVWr16U5nDHzJlAoYML9K2NMbIyO+FRNaGeq+xuIklssn9YbQsmam3hrwN5nlflcgGb9iclszyWQMuqNQxxK2KPia36jmw7pdJvQFfHjEOUUDmcHK1d9O1DLZpESwwUjjWKy+UJNfHY1FsokfTB3uns5UArqmUkWTtjj4RoIiCopgvtJtsTZBXwrrcDRGsPpXE6qVQXcG6+WtBrM2HqpzSP22jmjoUpvFkukMRcAPXLbHb4a/X0NulR4D0D1zd7u6ETqLq8FQaYaqXyU1hFa6IoakotvbI5ZQjVvqjZqTkW8NBms8G4yBS2y23meNg6/4RKrhOIDuttht2jroCywST6Uqy64xamFkhyubKqSasCBc45BlOmOx1AW9zhZJuiPgIpPBc5qi4Cy1iW45CaDiZq3W2AidxNXCG6dCXqc3U7hsgauyBonT8lr9FUaKoasjHjtvcEZTkY4YuEBj0Ey6CDhNaaLqTwCohBGWmeqhk8SyOiKrDXmdWEdoaRE4qspKa/WtLH+lutGa0oYj5wyBgSGvNQrOU1kMgSoAThMIHC3cf5k4Y9c2QKdaS5BophZ5nVTVSGvh2SpL72IyTL6/Rtogz0iMiW+06dRAQBcD5+FkJVBawWlyk0AE73+pOtqhQEFftJShMAx99S+KahyJwmsvo95pjTL43kopUh4SasPnVfFoV1wVBN8IjQaRRAROosnqFvj870hVGmUDdAaDpFDpttJrqhIBnyD1lLXc4hibucEaMcWWUOuy5xVw9MgoQ+Ci0uVTs0jgOJqjqy+Az4+rkBrroDOtDG1IZLz7s6Bk6zlB7ynH3uiN4DnROoVGE7PpQt+q4DyU0vvAD5aIy0iqAMfQPNXR/3WKcXSWGQmdZlANSZfx5lXmwHyCKQhUYuSNQUtbqsFs1Lt87aGLukgilrX7wa/mRLADfw0UolSKn2D+LXSzxH4XOsMS6dlSy68/32eSMD5uuIpNLOVSqQmMMmhPdLSFL7cl09GMwwtuGWNRH/EyQJTL+Rdh/jUSWu64Bv9+uXQc0XbjF1ppXXM9L1LdcqOE2KSPoXQipUctc/9gSCUTaWc79KsmlgwJzgHoCr8Z9rXQdRpKZfwe+qaiHk1oNf1Scqe8Dg7q0WrSzXISSZhssPIkco3cd1WfSiSzXjt0Q5VMRzouAeg8thTOCdAVJt6iuwh9VYqsY7Lab92pqUC1UiXRevytajyFF0NxxVKh3Oz6xZBOx7MBO3RHFUuHk2UAOlt+FQ4G0A8yYofoG+hceV19C8N9u6yxsoncoooTaLcrGvH6SBNTGtC1e9rvGOKpVGfMjiiBH99oigKg4z8f5QLoWx4hwPsKOleFxLSw/HfqWyubqC2WNFVfi2vCOoMErDEczySspeZ0OpFKexBIfQxOgHo48Hxz5MOhrwhMO/E0+ApdjaZqfMUYZyWaSrZnaAQClcQIeEjVklC8q9NcbUsm04msrwBliESDAS9s5I8fBQnoDIpjrD0OP85IHMfoKcF3NjbTqa40C8thcHQZD6VBHe/M9JjqnalENpHxFzaZI8mw1/G/chV0qhTPLysEX8rQeIHVXtycEeE5FFdK0dLG4rp7OzgoTTSR7HI0dsDbF8uEinCWSDri1Rzlqb+C0LHbTa2/IODXIRnPc5iKiak2Er81mLXIQhx+NGfVoiyJeDLtQnekk13RtK+o2RrNRLzK/whkhdebKJcR4AydS+b6jfeoSRtVwIol7NYxNi+adtjI9mw6kglgPOlYKpKMHCNa4PV7VP+RKwuuNDefg+DXiZDIiqpu0ZPtNAk3EHZ6RrjMYMrpIbk6O2PxENabjKfC8chxkiP27/x/XojL1dVwHjlNlDYLQpIbzHg7Uy6LRjz+QRY9mHAFqN4s/MsJ4b2JeCoWi54guxPZqF/3v3Pw+eJbxyFwligmaMP8a+K4myuzxZKRSDe/JZoOxNmhZCadStIjqWQ6mcqcZkezuUzc/D8BUEsDBC0AAAAIAAAAIQDdgPCN//////////8IABQAc2xvdC5ucHkBABAAHgEAAAAAAABUAAAAAAAAAJvsF+obEMnIUMZQrZ6SWpxcpG6loF6Taaiuo6Cell9UUpSYF59flJIKEndLzClOBYoXZyQWpAL5GoamFjqaOgq1CmQDLgYCgJEAYCIAmAkAFrwAAFBLAwQtAAAACAAAACEA4QY/bv//////////DQAUAHJhd19pbmRleC5ucHkBABAAvAEAAAAAAADLAAAAAAAAAK3NTUsCURjF8f9kZo6amenkW16suAqzUQgiXLsz2rRoFUNOKEjKTLiRPkVfuJMM8wHMu7r3uc/5nZ+nl+nzq8OGrZ2F8XtkH40dL0bWN/ZjFX1FwefbKpqFf/NJsIxDzeN5sA717g/vH/yBb77N3scFhwzHZDnhlDwuRUqcUabCBVVq1PFo0KRFm2u6GHrccMsdlj6HyB+lQi4xCqlynjiXiXSVWp2d9t98R7eWJg39eNqoabOqREXJsoSSpILEvOScGrJqyqjRUfMvUEsBAi0ALQAAAAgAAAAhAAdbqAUAJAAAACgAAAoAAAAAAAAAAAAAAIABAAAAAHRodW1icy5ucHlQSwECLQAtAAAACAAAACEA3YDwjVQAAAAeAQAACAAAAAAAAAAAAAAAgAE8JAAAc2xvdC5ucHlQSwECLQAtAAAACAAAACEA4QY/bssAAAC8AQAADQAAAAAAAAAAAAAAgAHKJAAAcmF3X2luZGV4Lm5weVBLBQYAAAAAAwADAKkAAADUJQAAAAA="}, "1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269": {"jpeg_sha256": "edb5e1982c2abf2e164d79eb2bbdcfb025b4ab5638879f973d72b67c2ace539a", "n": 86, "arrays_b64": "UEsDBC0AAAAIAAAAIQC9CdaX//////////8KABQAdGh1bWJzLm5weQEAEAAAFgAAAAAAAEwQAAAAAAAA7ZjpcxvHlcBjDHWYiiTLlMQDxA0MMABmMJj7wGAGM7hP4iQB8ABBEgRvkRQliqQokropyzpoXdZhS7Ys+UpclmzL3jjHOontSuS1s5uktlK7tdmqZJNP+2Frvy+9u39BPrurj+pX9bqr6733e/V6I5lPdBSf+d6h7y1aB4dmB2asnN66NOe2OvXW2tTMwZn+yb6pmcGhb+XB/vHZoU35bL3/wNDm3sZQTj2z2SGn/qj+b2qN0MKFJ4OhyrGWhXduZ4TksQbn4lLSE1kEdk3M5pTkKLClONMZHugEGuDFeng2AjRsL6+FpiSgocE403cg0b1+KDbXXSpU1t5brCezQ03ZK+8MBnMTALV6Ph8vjwKWpcN9vfk8sCt/tpjxF4EG4lBnIDYAALuLw/Hs0NRTr9ifycYmfpeQ88FwlF0+k8zJ4ejO0o2heD6f3pJYrBiyA/EG5+FFOKzUgT2dF3gfeRJoIAdrvtT1Hx9jpXiEYb58QnJpP0P1fJ1gigFZwCdqnJUthwzFOdpqjRd2ZQZ4F5gIAewao7cVkkBTqcqIQuiXXpr2KzT335OoO+ijPQ8/gum0nxYK9wJ2I+LlYjNeu0VnFYlO3gnp7Mq+es5o1bhYgBtjYNApvEiRuCwy7E9wmAnyvOcmz5I4z3C4hBk1oOJNsg7Q0KaLTwsu0Ki24cogA5raQH5XbxmCLBZ6muX4iJdEj1F+r4diKZ+CbtqVZvVls9HMEJLPY7MY2838tBcya0GMX8liKAozfv0k64AdEDsi+OyCl0IjIYeWoyjc7QMNOE7ROtFk0IkYaaMMZksPwqREXAdurj/w4bJTlhhygnQ4HTYyyFstXIhC+YDZQtEMZhBNRpGiURMN6XkvAkOk3oSmHYioeDAhxNL3UNIP6XyMGKGcDhAjBdxodVV5BydrjXYuatUqBj1qc1lMtEFDyaiMuHQ6jV8a2rSsw92Vzr6MeztdrqpfFjEHguQ9SVFngUfCZtnXrjWmDjgsiTa1kxgj2sPtbRY5H4Zsmva2vVIyJzjb2zFcueDAfaDN66IktxWCM3CN0OotBK4bINWtep9gYamWZjrWZW1R2prVctbE2bXqln2obZExqFu0uGNMpKytVtapTxEWK+RDLrtNTQYOaVnEzXuMDmNr0qt9HgTNWizd2tQqmzQDgq51/36Lvi4bNW06HJT6BW2zhQC/P+e1WK2UY7WPYTxOk/WkIsVoh4U8EEsTNtDiL8eD+1pJvXEhKbY0tYNtyW7e0ao1mvf2hyxtdrsR6M1pzWYCjl+Nc6hDi0dmF30uUOdlDlcZxKztcN71BS1OfZB4q7P/OQuo1SyPFEEbaAeiRzvVJi3Y0DqjN+Jmx76NVcXVZmjcdXqOMWmgxu2r4w5QL+1smDuNKqC0d0vvq6kwzBkAcX1IdiI6YM/KQcllVgMNmQmPtc0BVB+LYNNeAKAfJ1r3qQHAcLO/XaMFgObrdYvLAgC71l6kUBAAtmQuRikYAADtkfm4fRsAbM/drzj3aM8Oh38+tv6nptWp0X89ceAblXPt6r9dznyualx9+/cPcn9VAcNvff30/H+qVMilz/789CuVakvfG//+u98+o1Lpb/7653/sm4/43vmnvz4uHizEvvrjX/6wa7RU+M2f//JnFb5S//oP//wPqraF429/+dFXzzw7cej9T+98oVK5Tn384/UvnlFtn779s3c/O9XNei799sFH60OC583/+JdvfPMR6fGf/vL7HRPlA2OvfvO0obQyW63e/UzlWkiFRt76e9XOwap47LPXVCpidfLE6xeXEop040ZQmevmpXO/ScrjNdGz8MEJvzhdPPp+Zrmqn67c/eZJ+fCOnqEXfvao55KKHc5MJUdPqHYWJ873xyJTgsiNnBUCY5uxUfhE6pwssqmDj4eD3T3pD39YzvKBziOff/RIKaK19U/eu+Ebe65rMt8dxjwqsHj87gbmzzI8e/6sFOjzJAsrv4gGR+J9xxYuDYvpzIWf3h+UQ5GF+x+/cT95PLq8cffO5bCPrS3Eoz5BaRQ6Xr3/eiwYSi4E3r+c4PPhu/23vswIXWMvPZpbW/Ck+9789F43d7hj5rUP37kVLAZvzF5dH8kFYkNn4pI8IDXHDr3+9oM+b3XtYebRTxKe6LnL7135alFIn771+emHb9UKw7d/sRHmrnYvXfjRtQwfoF47eMQXH/KFV64NH4xXRGvHxqNPPjgYLJ87/5O1a3fS3PIrv7r8+dtL+Pztpxc/+eDvqpWNL67cK/xw+PixH11Y89f5V1dqbNeCXzx2b7qeKqYg/ub7Hz6aiIVeeumbteyLT9zLt76+eH/ph8T4jV+99OMr9/oTN356/errMxPD5x/d7L/ISHfnjpR9y4r31IPaYhcft/vvvPvRG76EfGnj6drR0m1u9drn56Y8R5n6y5+uXmanM8G77720VBa7Kxfv318/oFD31s6cGx5MyfNPevIxRTYrt958756UHLy/+umRxxfu+u6deHz05dNX0VfOvzu1nDwbHbnz4GKttJY9cOWVa2d8k8Fzi9fPnA3g8cMPCjExgLUmLr351vUIc2bilXD3Ss9s76E7ufqDc6vM/PpC7cHGEb527ua5bGkp2PnyK1evMNHBmbOvbZxVIPrIub5sWMIaU6lXb73kJ17I09RC0C4sdQb7z4xD3nqw92j1QjqWkbtGD3t6K2WCO3l9DilEjhUHL6xR3j2n8+VqnCgAnGvl7rkcEb4UZkXKrJVvx2uTtH4//nLy9FlZ87xxnq6kOE0bOIO6fQhMtSzkRAI3cSrvCMwIQkzVVIkP53PB5zdynkqsp3HHiXhliIht3zbdKRQgrLGhqw+Og+bvN8TmorzdsEsFnxyV3RaNakfvgb4QwW/G88JYki2bj/3jRu8Lx5ust/9r2T535blt8Xfde4yHnlW1hNpMuo4tQOMoZGeWtgINlbHMCx2ACjB+8PTMc5scabAw+wFV6eMlVO25k8s+uY+FuQF5b/mOhy12Ore6ZKsZ7QMbjCyMoZPmhqYK7RTGmoCtxbXYfGiTR/vTE9mdQO3XIs8xvednvy56Qkx01nrhVCgpZ+rbi2MKPDY+CUQiLtg9kwPwsCT7ellgbyXoFGpNQAM/FeuPA2vHCC+JEQ8Xn+ASgQc2Ar9MiqIrec+xNJnJm5UfqPv7Qpgxeqsx4fdl0eIoQPX7RKJXAfaODlvRq62XpaAAa/CpqwWXV9fmO1+6i+KGtuhMYiNgAtXsWf+yYNFoqWvGEufmTMypZ5MpksZL4w3ouMAwU9HDlFNs19jCD0kY1bUi2UsB0qRv9oQO19xGXRvpnTjkMhk0bFVeQGyozj1kmGcxL4Ef2VooO1CXZrCfx1CL3sQvEShmVDs9F2jKaGyxmUe7IKOuRSMeyUFmQ7umUpp1Guxa1yi7bpDCND7VXmftTquhY1kiUasJYi76CNjSTkLLOVpnaDXZZ0bMBm3zXmk8ZzUb1PtDS0XIYFEbwkMdFrwI7e3Bqg6HEzSK5wJ0SrfX7p7okJnW3ZCp3Me1qZ/XgeWgQatu3sP1EKDJ0LYXnZMgLa5ukxdEm5O3tDKVihNx2a10qRwSFQm1p1MJSXS7TLlUB5dOU6aUR9feYTNCGbuxxdTSpO2CzVq6rbl90m5BdK2tluGEHXZCFiiZiweHFcIYDHsYVLHrA5JCwZxJE1Kc8VJFsYQpnauwv9mcZQxIQL0PTOFarkXdou9TIJsRNNkCEZYvJiA16/cSnlmbmhMFCosbNFIAhlHYpkuGNERMrdMlRA2XoTXGmKedxFy4rRQFHQhlsyc2uR/O21sDPopJjRjUgo+lnUOgSYl5EDhqNvszJgNiMRtE2eJiUaQlrCABhUXUI1IrCApUayFGMrRo0kVlQsJTOltMwmmxS80UOgKkM6Mlh/tMoJszctMp1OWjjUxdQuAu2Woshwlr7wyimxYpl0OvhmclwQ0TBmY8wlAYrQ/V66THG1ez9QJBh9M6Z18IdGcli73gd5Net9mQL5CkSEPmA16ewEzq50YkgXSbwd1LMk8jKN8yWuY8Lsysz1cKokyr9ygliaM5sJVZ4n0CDLaJJ0WKcDnglW6OxNqbgcUawzhb9qsujnCso1kLlNMBwbR/PwB3y7zPtqeh7W7IU8GMW9vvRAISwjS6rno4DypqvuPHd/z4jh/f8eNv5Yf32tMl8cjynpVfZOL9oa7RNzvkKOrK3juflCUSCsRXq+koD/vJ/p5MJBXzQmOFaKwrTjgWC3FZYfxq/vrTMtM7/vzar3qTXbHU6MOC5HfZ+868EA7RpD0WHO/t9FOQj+rqTIQjQd5eScdj1QyGrGQ7/H4uotk38qnkVnp3Fz6ux7PJjoH1YdaLgZW1i4EASkC8MFLNsQzipVKdqWQ4RJg7U3Igk0PY1VRACggZI4AtkG1IZid8+VA0kCnWV+a9NGbrWL4QETEUZJWRWowSWTeeyiRkX5yAOnNKKFB0eY6nA4royUAAmDfpNbFdhvmZmC/XO3NwQSRpKHZyPUvjqBWTh3oYKuSCgsli1h/qp6F8Z1D25V3syR5JEdioE1DzZqsh8Vx7ZTgpFmvTUwshnLBJp44M0xhigbKDUdQhWPRKNm9PFLMhtFiNUHzCLs5XpZCkuN2qfV4cZYab9uf7JRdSmRg7EcJRZ+DFYxUGd0P2nn7RDcNWg1hSrHgStMN90zTGUhA9PRAIh2mbA9gdEvDwRNPuVMHDYNlq9bgftzvF87NllsQhZHR4070wq4npSSDBlNlurdRsiJeB2NpAVg7zoA1oDHewlaHdjaGwP+5J5rNLPsQN0atznZv+6HTVBzgS9UAg05NyJfpsZry7Crm9BCh09SalAAMhwFZfNVRnt21hI4G4P+dnZhUyhLILh+oeFLRT1ZqXBnEb6O0Ow3QBB209WYgMiAicrSmcwjttAIDN9sygAAB2+EOevMs84FcSgjgze5BGDqFIZUHG2SBsYrrCjs2Qo2z93QQXFXF7YVYKBiindbOeXv9wEQRU6t44yVfM+2oROipGFqujUFTiwbk5EpExnYbpktWUCbHAg2lcToluc+yoGJFxhxVQNaUP55sAoKnczXjH1DtKvQwXSMz4xx0wiTgi87LZxDptbDlgstmcTmowgXKSgkPhNUnZRLdjM4/qXPotgGp7YCLoq+7aIk2ICh8oEUMiy5Mu/3g3zfEsQQ6kMIoiUHQw5YAphdOHjwucB3Pj3ybiZ7cBm3lZn+iIRLYB7mxQZKOKpe5nWJpIDBd57tsvsdGEk2Y4FKkVEQcfxZyJozznpREGUKk2j9hUB3ZiAREGAH0yxGN+urUaZTiGinQVOY/XQ2IjCZimOMw9VKJJwe9GuiY4VnA7yP9V/XZSbdVQ7mYA2B/PCVoR3VMqcR6ODXV0ewXRyyDDcTcBYxxez7KbchTuK1Es53K4Vf93+bejyWzcBah2yaNxswfZkS4LgsAFlUqMkzy0vZTBccjFu2oxnOY3PWs0QTAEArlUqv9/gArYodZ8X6XaxsyXbD54W3DSJ3pJP91dILwsCRYzOGa2s86q7KIJJwyNyW7K7QRt/wNQSwMELQAAAAgAAAAhAJqS6f3//////////wgAFABzbG90Lm5weQEAEADWAAAAAAAAAFIAAAAAAAAAm+wX6hsQychQxlCtnpJanFykbqWgXpNpqK6joJ6WX1RSlJgXn1+UkgoSd0vMKU4FihdnJBakAvkaFmY6mjoKtQrkAy4GdMCIBpjQATM6YMEEAFBLAwQtAAAACAAAACEAjq4TkP//////////DQAUAHJhd19pbmRleC5ucHkBABAALAEAAAAAAACKAAAAAAAAAJvsF+obEMnIUMZQrZ6SWpxcpG6loG6TaaSuo6Cell9UUpSYF59flJIKEndLzClOBYoXZyQWpAL5GhZmOpo6CrUK5AMuBgZGBiYGZgYWBlYGNgZ2Bg4GTgYuBm4GHgZeBj4GfgYBBkIqKDdBgEGIQRBI8gN5vEBRbqAsJ1AVO1A1K1AXM1A3I9AUAFBLAQItAC0AAAAIAAAAIQC9CdaXTBAAAAAWAAAKAAAAAAAAAAAAAACAAQAAAAB0aHVtYnMubnB5UEsBAi0ALQAAAAgAAAAhAJqS6f1SAAAA1gAAAAgAAAAAAAAAAAAAAIABiBAAAHNsb3QubnB5UEsBAi0ALQAAAAgAAAAhAI6uE5CKAAAALAEAAA0AAAAAAAAAAAAAAIABFBEAAHJhd19pbmRleC5ucHlQSwUGAAAAAAMAAwCpAAAA3REAAAAA"}}')
_v30_tmp = _V30Path('/kaggle/tmp') if _V30Path('/kaggle/tmp').is_dir() else _V30Path('/tmp')
_v30_left = LEG_DEADLINE_S - (_v30_time.time() - V30_T0)
_v30_leg = {'launched': _v30_left >= LEG_MIN_S, 'seconds_left': round(_v30_left), 'killed_shards': []}
_v30_diag = {}
for _mod in ('pydicom', 'pylibjpeg', 'openjpeg', 'libjpeg', 'gdcm', 'PIL', 'cv2'):
    try:
        _v30_diag[_mod] = getattr(__import__(_mod), '__version__', 'present')
    except Exception:
        _v30_diag[_mod] = None
try:
    from PIL import features as _v30_features
    _v30_diag['pil_jpg_2000'] = bool(_v30_features.check('jpg_2000'))
except Exception:
    _v30_diag['pil_jpg_2000'] = None
_v30_leg['decoders'] = _v30_diag
_v30_leg_t0 = _v30_time.time()
if _v30_leg['launched']:
    _procs = []
    for _shard in range(2):
        _args = {'shard': _shard, 'n_shards': 2, 'comp': str(_v30_comp), 'weights': _v30_weights,
                 'k_infer': K_INFER, 'img': 320, 'eval_batch_tokens': 160, 'workers': 2,
                 'cache_dir': str(_v30_tmp / f'v30cache{_shard}'), 'out_dir': str(_V30_WORK / 'v30_leg'),
                 'reference_json': str(_v30_ref), 'reference_local_json': str(_v30_ref_local), 'thumb_tolerance': 3, 'time_limit_s': _v30_left,
                 'oof_expect': _V30_OOF_EXPECT, 'oof_tolerance': OOF_TOLERANCE, 'weight_folds': _v30_weight_folds}
        _env = {**_v30_os.environ, 'CUDA_VISIBLE_DEVICES': str(_shard)}
        _logf = open(_V30_WORK / f'v30_leg_log_{_shard}.txt', 'w')
        _procs.append((_v30_sp.Popen([_v30_sys.executable, '/kaggle/working/v30_leg.py', _v30_json.dumps(_args)],
                                     env=_env, stdout=_logf, stderr=_v30_sp.STDOUT, start_new_session=True), _logf))
    _v30_deadline = V30_T0 + LEG_DEADLINE_S
    for _shard, (_p, _logf) in enumerate(_procs):
        try:
            _p.wait(timeout=max(1.0, _v30_deadline - _v30_time.time()))
        except _v30_sp.TimeoutExpired:
            try:
                import signal as _v30_signal
                _v30_os.killpg(_v30_os.getpgid(_p.pid), _v30_signal.SIGKILL)   # the shard and its pool workers
            except Exception:
                _p.kill()
            _p.wait()
            _v30_leg['killed_shards'].append(_shard)
        _logf.close()
        print(f'leg shard {_shard} exit code {_p.returncode}', flush=True)
        print(''.join(open(_V30_WORK / f'v30_leg_log_{_shard}.txt').readlines()[-8:]), flush=True)
_v30_leg['leg_seconds'] = round(_v30_time.time() - _v30_leg_t0, 1)
print(_v30_json.dumps(_v30_leg), flush=True)

In [ ]:
# Blend with fallbacks, validate, write submission.csv.
import numpy as _np
import pandas as _pd

_labels = _pd.read_csv(_v30_comp / 'sample_submission.csv', nrows=0).columns.tolist()[1:]
_test_ids = _pd.read_csv(_v30_comp / 'test.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID.tolist()
_anchor = _pd.read_csv(_V30_ANCHOR, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID').loc[_test_ids, _labels]
_receipts = {}
for _shard in range(2):
    _f = _V30_WORK / 'v30_leg' / f'v30_leg_receipt_{_shard}.json'
    _receipts[_shard] = _v30_json.loads(_f.read_text()) if _f.is_file() else {'status': 'missing'}
_parts = [_pd.read_csv(f, dtype={'StudyInstanceUID': str}) for f in sorted((_V30_WORK / 'v30_leg').glob('v30_leg_shard*.csv'))] \
    if (_V30_WORK / 'v30_leg').is_dir() else []
_ours = _pd.concat(_parts).set_index('StudyInstanceUID') if _parts else _pd.DataFrame()
_ref = _receipts[0].get('reference_check', {}).get('status')
_coverage = float(_ours.index.isin(_test_ids).sum()) / max(len(_test_ids), 1) if len(_ours) else 0.0
_ref_studies = _receipts[0].get('reference_check', {}).get('studies', {})
_byte_equal = bool(_ref_studies) and all(v.get('byte_equal') for v in _ref_studies.values())
_oof = _receipts[0].get('oof_check', {}).get('status')
_shards_done = all(v.get('status') == 'done' for v in _receipts.values())
_use_leg = bool(_v30_leg['launched'] and not _v30_leg['killed_shards'] and _shards_done and _ref == 'pass'
                and _byte_equal and _oof == 'pass' and _coverage >= MIN_COVERAGE)
_decision = {'leg_launched': _v30_leg['launched'], 'reference_check': _ref, 'coverage': _coverage,
             'shard_status': {k: v.get('status') for k, v in _receipts.items()}, 'blend_weight': BLEND_W,
             'used_leg': _use_leg, 'oof_check': _receipts[0].get('oof_check', {}).get('status'),
             'weights': _v30_weights_meta, 'reference_byte_equal': _byte_equal, 'shards_done': _shards_done,
             'killed_shards': _v30_leg['killed_shards'], 'leg_seconds': _v30_leg.get('leg_seconds'),
             'decoders': _v30_leg.get('decoders'), 'base': _v30_base_status}

_anchor_rank = _anchor.rank(pct=True)
if _use_leg:
    _fold_ranks = []
    for _k in range(len(_v30_weights)):
        _cols = [f'{c}__f{_k}' for c in _labels]
        _r = _ours[_cols].copy()
        _r.columns = _labels
        _fold_ranks.append(_r.rank(pct=True))
    _ours_rank = (sum(_fold_ranks) / len(_fold_ranks)).rank(pct=True).reindex(_test_ids)
    _missing = _ours_rank.isna().any(axis=1)
    _ours_rank[_missing] = _anchor_rank[_missing]
    _decision['studies_filled_from_anchor'] = int(_missing.sum())
    _final = ((1 - BLEND_W) * _anchor_rank + BLEND_W * _ours_rank).rank(pct=True)
    _decision['mean_spearman_anchor_vs_ours'] = float(_np.mean([
        _anchor_rank[c].corr(_ours_rank[c], method='spearman') for c in _labels])) if len(_test_ids) > 2 else None
else:
    _final = _anchor.copy()

_sub = _final.reset_index()
_sub = _sub[['StudyInstanceUID'] + _labels]
assert _sub.columns.tolist() == _pd.read_csv(_v30_comp / 'sample_submission.csv', nrows=0).columns.tolist()
assert _sub.StudyInstanceUID.tolist() == _test_ids
assert _np.isfinite(_sub[_labels].to_numpy(float)).all()
_sub.to_csv(_V30_WORK / 'submission.csv', index=False)
_sub.to_csv(_V30_WORK / 'v30_submission.csv', index=False)
_decision['elapsed_hours'] = round((_v30_time.time() - V30_T0) / 3600, 2)
(_V30_WORK / 'v30_receipt.json').write_text(_v30_json.dumps({'decision': _decision, 'leg': _receipts}, indent=2, default=str))
print(_v30_json.dumps(_decision, indent=2, default=str), flush=True)